# MMDS Phase 6 historical replay

Mode: full.

In [ ]:
import importlib.util
from importlib.metadata import PackageNotFoundError, version
import os
import platform
import subprocess
import sys
from pathlib import Path

print(f"Python {platform.python_version()}", flush=True)
try:
    installed = version("pyspark")
except PackageNotFoundError:
    installed = None
if installed != "4.3.0":
    subprocess.run([sys.executable, "-m", "pip", "install", "pyspark==4.3.0"], check=True)
if importlib.util.find_spec("pyarrow") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "pyarrow"], check=True)
subprocess.run(["java", "-version"], check=True)
os.environ["PYSPARK_PYTHON"] = sys.executable
if importlib.util.find_spec('matplotlib') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'matplotlib'], check=True)


In [ ]:
project = Path('/kaggle/temp/mmds_phase6/code')
sources = {'src/ingest/dataset_audit.py': '"""Measure the MMDS source without changing or exporting traffic records."""\n\nimport argparse\nfrom collections import Counter\nimport csv\nfrom datetime import datetime\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport time\nimport unicodedata\n\n\nLABEL_NAMES = {"label", "traffic_label", "class", "attack_type"}\nTIME_NAMES = {"timestamp", "time_stamp", "flow_timestamp"}\nIDENTIFIER_NAMES = {"flow_id", "src_ip", "dst_ip", "source_ip", "destination_ip"}\nNONNEGATIVE = re.compile(r"duration|length|(?:^|_)len(?:_|$)|pkts?|packets?|bytes?|byts?|count|(?:^|_)iat(?:_|$)|flags?|active|idle|(?:^|_)rate(?:_|$)|(?:^|_)ps$")\nSENTINELS = {"infinity", "+infinity", "-infinity", "inf", "+inf", "-inf"}\nBENIGN = "BENIGN"\n# The quantized comparison is limited to these non-identity flow measurements.\nNEAR_FIELDS = {\n    "flow_duration", "tot_fwd_pkts", "tot_bwd_pkts",\n    "totlen_fwd_pkts", "totlen_bwd_pkts", "flow_iat_mean",\n}\n\n\ndef normalize_column(value):\n    value = unicodedata.normalize("NFKC", value).lstrip("\\ufeff").strip().casefold()\n    return re.sub(r"[^\\w]+", "_", value, flags=re.UNICODE).strip("_")\n\n\ndef normalize_label(value):\n    return " ".join(unicodedata.normalize("NFKC", value).split()).casefold() if value is not None else None\n\n\ndef proposed_binary(value):\n    """Only BENIGN has a default mapping; observed attack families need review."""\n    return "benign" if normalize_label(value) == BENIGN.casefold() else "unmapped"\n\n\ndef capture_day(name):\n    match = re.search(r"-(\\d{2}-\\d{2}-\\d{4})_", name)\n    return datetime.strptime(match.group(1), "%d-%m-%Y").date().isoformat() if match else None\n\n\ndef inspect_schema(header):\n    names = [normalize_column(col) for col in header]\n    original_counts = Counter(header)\n    normalized_counts = Counter(names)\n    return {\n        "original": header,\n        "normalized": names,\n        "exact_duplicates": sorted(k for k, n in original_counts.items() if n > 1),\n        "normalized_duplicates": sorted(k for k, n in normalized_counts.items() if n > 1),\n        "whitespace_columns": [i for i, raw in enumerate(header) if raw != raw.strip()],\n        "encoding_columns": [i for i, raw in enumerate(header) if any(ord(c) > 127 for c in raw)],\n        "unsupported_indices": [i for i, name in enumerate(names) if not name or normalized_counts[name] > 1],\n        "label_indices": [i for i, name in enumerate(names) if name in LABEL_NAMES],\n        "timestamp_indices": [i for i, name in enumerate(names) if name in TIME_NAMES],\n    }\n\n\ndef compare_schema(reference, candidate):\n    left, right = reference["normalized"], candidate["normalized"]\n    return {\n        "same_order": left == right,\n        "same_original_header": reference["original"] == candidate["original"],\n        "missing": list((Counter(left) - Counter(right)).elements()),\n        "extra": list((Counter(right) - Counter(left)).elements()),\n        "compatible": Counter(left) == Counter(right)\n        and not reference["unsupported_indices"] and not candidate["unsupported_indices"],\n    }\n\n\ndef read_inventory(root, config):\n    root = Path(root)\n    expected = {f\'{config["directory"]}/{name}\': size for name, size in config["files"].items()}\n    available = {p.relative_to(root).as_posix(): p for p in root.rglob("*") if p.is_file()}\n    names = Counter(path.rsplit("/", 1)[-1] for path in available)\n    items = []\n    for rel in sorted(expected.keys() | available.keys()):\n        path = available.get(rel)\n        name = rel.rsplit("/", 1)[-1]\n        size = None\n        access = "missing"\n        if path is not None:\n            try:\n                size = path.stat().st_size\n                with path.open("rb") as source:\n                    source.read(1)\n                access = "readable"\n            except OSError:\n                access = "unreadable"\n        items.append({\n            "path": rel, "name": name, "capture_day": capture_day(name) if rel in expected else None,\n            "bytes": size, "expected_bytes": expected.get(rel), "access": access,\n            "unexpected": rel not in expected, "duplicate_basename": names[name] > 1,\n            "size_matches": size == expected[rel] if rel in expected else None,\n        })\n    return items\n\n\ndef counts_reconcile(counts, labels=None):\n    return (\n        counts["physical_lines"] == sum(counts[key] for key in ("blank", "headers", "malformed", "parsed"))\n        and counts["malformed"] == sum(counts[key] for key in ("bad_width_or_quotes", "spark_corrupt"))\n        and counts["data_records"] == counts["parsed"] + counts["malformed"]\n        and (labels is None or sum(row["count"] for row in labels) == counts["parsed"])\n    )\n\n\ndef stable_sample(raw, seed, modulus):\n    from pyspark.sql import functions as F\n\n    return F.pmod(F.xxhash64(F.lit(seed), raw), F.lit(modulus)) == 0\n\n\ndef timestamp_columns(raw, formats):\n    from pyspark.sql import functions as F\n\n    value = F.trim(raw)\n    primary_options = [F.try_to_timestamp(value, F.lit(fmt)) for fmt in formats]\n    shadow_options = [\n        F.try_to_timestamp(value, F.lit(fmt.replace("d/M/yyyy", "M/d/yyyy")))\n        for fmt in formats if fmt.startswith("d/M/yyyy")\n    ]\n    primary = F.coalesce(*primary_options)\n    shadow = F.coalesce(*shadow_options)\n    matched = F.coalesce(*[F.when(expr.isNotNull(), F.lit(fmt)) for fmt, expr in zip(formats, primary_options)])\n    return primary, shadow, matched\n\n\ndef numeric_fields(value):\n    from pyspark.sql import functions as F\n\n    stripped = F.trim(value)\n    present = value.isNotNull() & (stripped != "")\n    number = stripped.try_cast("double")\n    nan = present & F.isnan(number)\n    inf = present & (\n        F.lower(stripped).isin(*sorted(SENTINELS)) | F.coalesce(F.abs(number) == float("inf"), F.lit(False))\n    )\n    finite = present & number.isNotNull() & ~nan & ~inf\n    failed = present & number.isNull() & ~inf\n    return present, number, nan, inf, finite, failed\n\n\ndef atomic_json(path, value):\n    stage = path.with_suffix(".tmp")\n    stage.write_text(json.dumps(value, ensure_ascii=False, indent=2, default=str) + "\\n", encoding="utf-8")\n    stage.replace(path)\n\n\ndef spark_session(config, scratch):\n    from pyspark.sql import SparkSession\n\n    spark_config = config["spark"]\n    os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")\n    return (\n        SparkSession.builder.appName("mmds-phase-1-dataset-audit")\n        .master(spark_config["master"])\n        .config("spark.driver.memory", spark_config["driver_memory"])\n        .config("spark.local.dir", str(scratch))\n        .config("spark.sql.shuffle.partitions", spark_config["shuffle_partitions"])\n        .config("spark.sql.files.maxPartitionBytes", spark_config["input_partition_bytes"])\n        .config("spark.sql.files.openCostInBytes", 4194304)\n        .config("spark.sql.adaptive.enabled", "true")\n        .config("spark.sql.session.timeZone", "UTC")\n        .config("spark.sql.timestampType", "TIMESTAMP_NTZ")\n        .config("spark.sql.legacy.timeParserPolicy", "CORRECTED")\n        .config("spark.sql.csv.parser.columnPruning.enabled", "false")\n        .config("spark.ui.enabled", "false")\n        .getOrCreate()\n    )\n\n\ndef record_frame(spark, path, header_line, width):\n    from pyspark.sql import functions as F\n\n    raw = F.col("raw")\n    blank = raw.rlike(r"^\\s*$")\n    header = raw == header_line\n    comma_width = F.length(raw) - F.length(F.replace(raw, F.lit(","), F.lit(""))) + 1\n    quoted = F.instr(raw, \'"\') > 0\n    field = r\'(?:"(?:[^"]|"")*"|[^",]*)\'\n    quote_grammar = "^" + field + "(?:," + field + ")*$"\n    masked = F.regexp_replace(raw, r\'"(?:[^"]|"")*"\', \'""\')\n    quoted_width = F.length(masked) - F.length(F.replace(masked, F.lit(","), F.lit(""))) + 1\n    valid = F.when(quoted, raw.rlike(quote_grammar) & (quoted_width == width)).otherwise(comma_width == width)\n    status = F.when(blank, "blank").when(header, "headers").when(~valid, "bad_width_or_quotes")\n    ddl = ", ".join([f"c{i} STRING" for i in range(width)] + ["_corrupt STRING"])\n    options = {\n        "mode": "PERMISSIVE", "columnNameOfCorruptRecord": "_corrupt",\n        "quote": \'"\', "escape": \'"\', "nullValue": "\\u0000", "ignoreLeadingWhiteSpace": "false",\n        "ignoreTrailingWhiteSpace": "false", "unescapedQuoteHandling": "RAISE_ERROR",\n    }\n    physical = spark.read.text(str(path)).toDF("raw")\n    parsed = physical.withColumn("csv", F.when(status.isNull(), F.from_csv(raw, ddl, options)))\n    kind = F.coalesce(status, F.when(F.col("csv._corrupt").isNotNull(), "spark_corrupt"), F.lit("parsed"))\n    return parsed.select("raw", kind.alias("kind"), *[F.col(f"csv.c{i}").alias(f"c{i}") for i in range(width)])\n\n\ndef count_records(frame):\n    counts = {row["kind"]: row["count"] for row in frame.groupBy("kind").count().collect()}\n    summary = {k: counts.get(k, 0) for k in ("blank", "headers", "bad_width_or_quotes", "spark_corrupt", "parsed")}\n    summary["malformed"] = summary["bad_width_or_quotes"] + summary["spark_corrupt"]\n    summary["data_records"] = summary["parsed"] + summary["malformed"]\n    summary["physical_lines"] = summary["data_records"] + summary["headers"] + summary["blank"]\n    summary["repeated_header_lines"] = max(0, summary["headers"] - 1)\n    return summary\n\n\ndef count_if(condition):\n    from pyspark.sql import functions as F\n\n    return F.count(F.when(condition, 1))\n\n\ndef profile_columns(frame, schema, config):\n    from pyspark.sql import functions as F\n\n    sampled = frame.where(stable_sample(F.col("raw"), config["sample_seed"], config["sample_modulus"]))\n    sample_count = sampled.count()\n    # Exact value counts within the small sample; collect only a row per column.\n    elements = [\n        F.struct(F.lit(i).alias("index"), F.col(f"c{i}").alias("value"))\n        for i in range(len(schema["original"]))\n    ]\n    sample_summary = (\n        sampled.select(F.explode(F.array(*elements)).alias("cell")).select("cell.*")\n        .where(F.col("value").isNotNull() & (F.trim("value") != ""))\n        .groupBy("index", "value").count()\n        .groupBy("index").agg(F.sum("count").alias("present"), F.max("count").alias("dominant"))\n    )\n    sample_counts = {row["index"]: (row["present"], row["dominant"]) for row in sample_summary.collect()}\n    profiles = []\n    # Limit the size of each aggregate projection and reuse the on-disk parse.\n    for start in range(0, len(schema["original"]), 24):\n        full = []\n        stop = min(start + 24, len(schema["original"]))\n        for i in range(start, stop):\n            col = F.col(f"c{i}")\n            present, number, nan, inf, finite, failed = numeric_fields(col)\n            measures = {\n                "null": count_if(col.isNull()), "empty_or_whitespace": count_if(col.isNotNull() & (F.trim(col) == "")),\n                "present": count_if(present), "finite_numeric": count_if(finite),\n                "nan": count_if(nan), "infinite": count_if(inf), "numeric_conversion_failures": count_if(failed),\n                "negative": count_if(finite & (number < 0)),\n                "lexical_min": F.min(F.when(present, col)), "lexical_max": F.max(F.when(present, col)),\n            }\n            full.extend(expr.alias(f"c{i}_{key}") for key, expr in measures.items())\n        full_result = frame.agg(*full).first().asDict()\n        for i in range(start, stop):\n            prefix = f"c{i}_"\n            row = {key[len(prefix):]: value for key, value in full_result.items() if key.startswith(prefix)}\n            low, high = row.pop("lexical_min"), row.pop("lexical_max")\n            row["constant_nonempty"] = low == high if row["present"] else None\n            row["sample_present"], top_count = sample_counts.get(i, (0, 0))\n            row["sample_dominant_count"] = top_count\n            row["sample_dominant_fraction"] = top_count / row["sample_present"] if row["sample_present"] else None\n            row["near_constant_sample"] = row["sample_dominant_fraction"] >= 0.99 if row["sample_present"] else None\n            name = schema["normalized"][i]\n            support = row["finite_numeric"] + row["nan"] + row["infinite"]\n            identifier = name in IDENTIFIER_NAMES\n            numeric = support > 0 and support / row["present"] >= 0.95\n            row["type_candidate"] = (\n                "label" if i in schema["label_indices"] else "timestamp" if i in schema["timestamp_indices"]\n                else "unsupported" if i in schema["unsupported_indices"]\n                else "identifier" if identifier else "categorical" if name in {"protocol", "protocol_number"}\n                else "numeric" if numeric else "categorical" if row["present"] else "unobserved"\n            )\n            row["invalid_negative"] = (\n                row["negative"] if numeric and NONNEGATIVE.search(name) and not name.startswith("init_") else None\n            )\n            if row["type_candidate"] != "numeric":\n                row["numeric_conversion_failures"] = None\n            profiles.append(dict(row, index=i, original=schema["original"][i], normalized=name, sample_rows=sample_count))\n    return profiles\n\n\ndef summarize_time(frame, schema, formats, day):\n    from pyspark.sql import functions as F\n\n    results = []\n    for i in schema["timestamp_indices"]:\n        raw = F.col(f"c{i}")\n        primary, shadow, matched = timestamp_columns(raw, formats)\n        stamped = frame.select(raw.alias("source"), primary.alias("parsed"), shadow.alias("shadow"), matched.alias("format"))\n        parsed, shadow, source = F.col("parsed"), F.col("shadow"), F.col("source")\n        present = source.isNotNull() & (F.trim(source) != "")\n        date = F.to_date(parsed)\n        aggregates = stamped.agg(\n            F.count("*").alias("rows"), count_if(~present).alias("missing"), count_if(present & parsed.isNull()).alias("invalid"),\n            count_if(parsed.isNotNull()).alias("parsed"), F.min(parsed).alias("earliest"), F.max(parsed).alias("latest"),\n            count_if(shadow.isNotNull() & parsed.isNull()).alias("mdy_only"),\n            count_if(shadow.isNotNull() & parsed.isNotNull() & (F.to_date(shadow) != date)).alias("ambiguous"),\n            count_if(date == F.lit(day)).alias("date_matches"),\n            count_if(date.isNotNull() & (date != F.lit(day))).alias("date_mismatches"),\n            count_if(F.to_date(shadow) == F.lit(day)).alias("mdy_date_matches"),\n        ).first().asDict()\n        dates = [row.asDict() for row in stamped.where(parsed.isNotNull()).groupBy(date.alias("date"), "format").count().limit(501).collect()]\n        if len(dates) <= 500:\n            aggregates["date_formats"] = sorted(dates, key=lambda row: (row["date"], row["format"]))\n        results.append(dict(aggregates, index=i, original=schema["original"][i]))\n    return results\n\n\ndef fingerprint_frame(frame, schema, path, config):\n    from pyspark.sql import functions as F\n\n    label = schema["label_indices"]\n    if len(label) != 1:\n        return None\n    columns = sorted(\n        (name, i) for i, name in enumerate(schema["normalized"])\n        if name not in LABEL_NAMES | TIME_NAMES | IDENTIFIER_NAMES and i not in schema["unsupported_indices"]\n    )\n    schema_key = hashlib.sha256(json.dumps([name for name, _ in columns]).encode()).hexdigest()[:16]\n    selected = [(name, i) for name, i in columns if name in NEAR_FIELDS]\n    raw = F.col("raw")\n    near = F.lit(None).cast("string")\n    if len(selected) >= 3:\n        buckets = [F.floor(F.col(f"c{i}").try_cast("double") / (1000 if "duration" in name or "iat" in name else 10))\n                   for name, i in selected]\n        complete = F.lit(True)\n        for bucket in buckets:\n            complete = complete & bucket.isNotNull()\n        near = F.when(stable_sample(raw, config["sample_seed"], config["sample_modulus"]) & complete,\n                      F.sha2(F.to_json(F.array(*buckets)), 256))\n    return frame.select(\n        F.sha2(raw, 256).alias("row_hash"),\n        F.sha2(F.to_json(F.array(*[F.col(f"c{i}") for _, i in columns])), 256).alias("feature_hash"),\n        F.col(f"c{label[0]}").alias("label"), near.alias("near_hash"),\n        F.lit(schema_key).alias("schema_key"), F.lit(path).alias("source"),\n    )\n\n\ndef summarize_fingerprints(spark, paths):\n    from pyspark.sql import functions as F\n\n    rows = spark.read.parquet(*[str(path) for path in paths])\n    duplicate_groups = rows.groupBy("row_hash").count().where(F.col("count") > 1)\n    duplicates = duplicate_groups.agg(F.count("*").alias("groups"), F.sum(F.col("count") - 1).alias("excess")).first().asDict()\n    known = rows.where(F.col("label").isNotNull() & (F.trim(F.col("label")) != ""))\n    conflicts = (known.groupBy("schema_key", "feature_hash")\n                 .agg(F.min("label").alias("first"), F.max("label").alias("last"), F.count("*").alias("rows"))\n                 .where(F.col("first") != F.col("last"))\n                 .agg(F.count("*").alias("groups"), F.sum("rows").alias("rows")).first().asDict())\n    near_sample = rows.where(F.col("near_hash").isNotNull())\n    near = (near_sample.groupBy("schema_key", "near_hash")\n            .agg(F.min("feature_hash").alias("first"), F.max("feature_hash").alias("last"), F.count("*").alias("rows"))\n            .where(F.col("first") != F.col("last"))\n            .agg(F.count("*").alias("groups"), F.sum("rows").alias("rows")).first().asDict())\n    near["eligible_sample_rows"] = near_sample.count()\n    return {\n        "raw_line_duplicates_sha256": {key: value or 0 for key, value in duplicates.items()},\n        "label_conflicting_features_sha256": {key: value or 0 for key, value in conflicts.items()},\n        "near_sample_buckets": {key: value or 0 for key, value in near.items()},\n    }\n\n\ndef exact_duplicate_summary(spark, root, inventory):\n    """Group full parsed string arrays within identical original headers."""\n    from pyspark.sql import functions as F\n\n    schemas = {}\n    complete = []\n    for item in inventory:\n        if item["unexpected"] or item["access"] != "readable" or not item["size_matches"]:\n            raise ValueError("Exact duplicate measurement requires the verified inventory.")\n        path = root / item["path"]\n        with path.open("rb") as source:\n            header_line = source.readline(65536).decode("utf-8").rstrip("\\r\\n").lstrip("\\ufeff")\n        header = tuple(next(csv.reader([header_line], strict=True)))\n        schema_key = schemas.setdefault(header, len(schemas))\n        frame = record_frame(spark, path, header_line, len(header))\n        complete.append(frame.where(F.col("kind") == "parsed").select(\n            F.lit(schema_key).alias("schema"),\n            F.array(*[F.col(f"c{i}") for i in range(len(header))]).alias("values"),\n            F.lit(item["capture_day"]).alias("day"),\n        ))\n    combined = complete[0]\n    for frame in complete[1:]:\n        combined = combined.unionByName(frame)\n    groups = combined.groupBy("schema", "values").agg(\n        F.count("*").alias("count"), F.min("day").alias("first_day"), F.max("day").alias("last_day"),\n    )\n    result = groups.agg(\n        F.sum("count").alias("parsed_rows"),\n        F.count("*").alias("distinct_rows"),\n        count_if(F.col("count") > 1).alias("duplicate_groups"),\n        F.sum(F.col("count") - 1).alias("excess_copies"),\n        count_if(F.col("first_day") != F.col("last_day")).alias("cross_day_groups"),\n    ).first().asDict()\n    return result | {"files": len(complete), "original_schemas": len(schemas)}\n\n\ndef audit_exact_duplicates(config, root, output, scratch):\n    root, output, scratch = map(Path, (root, output, scratch))\n    if not root.is_dir() or any(path.resolve().is_relative_to(root.resolve()) for path in (output, scratch)):\n        raise ValueError("Use a mounted dataset root and separate writable output and scratch directories")\n    output.mkdir(parents=True, exist_ok=True)\n    scratch.mkdir(parents=True, exist_ok=True)\n    inventory = read_inventory(root, config)\n    atomic_json(output / "inventory.json", {\n        "dataset": {key: config[key] for key in ("owner", "slug", "version")},\n        "expected_bytes": sum(config["files"].values()),\n        "actual_total_bytes": sum(item["bytes"] or 0 for item in inventory),\n        "files": inventory,\n    })\n    spark = spark_session(config, scratch)\n    spark.sparkContext.setLogLevel("WARN")\n    try:\n        started = time.monotonic()\n        print("Measuring exact duplicate field arrays across the mounted captures.", flush=True)\n        result = exact_duplicate_summary(spark, root, inventory)\n        result["elapsed_seconds"] = round(time.monotonic() - started, 3)\n        result["spark_version"] = spark.version\n        result["spark"] = config["spark"]\n        atomic_json(output / "exact_duplicates.json", result)\n        print(f"Exact duplicate aggregation completed in {result[\'elapsed_seconds\']} s", flush=True)\n    finally:\n        spark.stop()\n\n\ndef measure_file(spark, path, item, output, cache_dir, config, result):\n    from pyspark import StorageLevel\n    from pyspark.sql import functions as F\n\n    with path.open("rb") as source:\n        header_bytes = source.readline(65536)\n    if not header_bytes.endswith(b"\\n"):\n        raise ValueError("Missing header newline or header exceeds 64 KiB")\n    header_line = header_bytes.decode("utf-8").rstrip("\\r\\n")\n    schema = inspect_schema(next(csv.reader([header_line.lstrip("\\ufeff")], strict=True)))\n    schema["utf8_bom"] = header_bytes.startswith(b"\\xef\\xbb\\xbf")\n    result["schema"] = schema\n    cache_path = cache_dir / (item["name"] + ".parquet")\n    stage = output / (item["name"] + ".json")\n    frame = record_frame(spark, path, header_line.lstrip("\\ufeff"), len(schema["original"])).persist(StorageLevel.DISK_ONLY)\n    try:\n        started = time.monotonic()\n        counts = count_records(frame)\n        if not counts_reconcile(counts) or not counts["headers"]:\n            raise ValueError("Header or row counts failed reconciliation")\n        result["counts"] = counts\n        result["stage_seconds"]["parse_and_count"] = round(time.monotonic() - started, 3)\n        atomic_json(stage, result)\n        valid = frame.where(F.col("kind") == "parsed")\n        started = time.monotonic()\n        if len(schema["label_indices"]) == 1:\n            label_col = F.col(f\'c{schema["label_indices"][0]}\')\n            labels = valid.groupBy(label_col.alias("label")).count().orderBy("label").limit(501).collect()\n            if len(labels) > 500:\n                raise ValueError("Label cardinality exceeds compact audit limit")\n            result["labels"] = [row.asDict() for row in labels]\n            if not counts_reconcile(counts, result["labels"]):\n                raise ValueError("Label counts do not reconcile")\n        result["timestamps"] = summarize_time(valid, schema, config["timestamp_formats"], item["capture_day"])\n        result["stage_seconds"]["labels_and_time"] = round(time.monotonic() - started, 3)\n        atomic_json(stage, result)\n        started = time.monotonic()\n        result["quality"] = profile_columns(valid, schema, config)\n        result["stage_seconds"]["quality"] = round(time.monotonic() - started, 3)\n        atomic_json(stage, result)\n        started = time.monotonic()\n        fingerprints = fingerprint_frame(valid, schema, item["path"], config)\n        if fingerprints is not None:\n            fingerprints.write.mode("overwrite").parquet(str(cache_path))\n            result["fingerprints"] = True\n        result["stage_seconds"]["fingerprints"] = round(time.monotonic() - started, 3)\n        return cache_path if fingerprints is not None else None\n    finally:\n        frame.unpersist(blocking=True)\n\n\ndef audit_dataset(config, root, output, scratch, only_days=None):\n    root, output, scratch = map(Path, (root, output, scratch))\n    if not root.is_dir() or any(path.resolve().is_relative_to(root.resolve()) for path in (output, scratch)):\n        raise ValueError("Use a mounted dataset root and separate writable output and scratch directories")\n    output.mkdir(parents=True, exist_ok=True)\n    scratch.mkdir(parents=True, exist_ok=True)\n    fingerprints_dir = scratch / "fingerprints"\n    fingerprints_dir.mkdir(exist_ok=True)\n    (scratch / "spark").mkdir(exist_ok=True)\n    inventory = read_inventory(root, config)\n    atomic_json(output / "inventory.json", {\n        "dataset": {key: config[key] for key in ("owner", "slug", "version")},\n        "expected_bytes": sum(config["files"].values()),\n        "actual_total_bytes": sum(item["bytes"] or 0 for item in inventory),\n        "files": inventory,\n    })\n    spark = spark_session(config, scratch / "spark")\n    spark.sparkContext.setLogLevel("WARN")\n    run = {\n        "spark": config["spark"] | {\n            "version": spark.version, "timezone": "UTC", "timestamp_type": "TIMESTAMP_NTZ",\n            "executor_cores": spark.sparkContext.defaultParallelism, "spark_application": spark.sparkContext.appName,\n        },\n        "timestamp_formats": config["timestamp_formats"], "files": {},\n    }\n    started = time.monotonic()\n    atomic_json(output / "run_summary.json", run)\n    fingerprints = []\n    try:\n        # Complete the smaller files first so expensive captures do not prevent partial findings.\n        for item in sorted(inventory, key=lambda entry: entry["bytes"] or 0):\n            if item["unexpected"] or (only_days is not None and item["capture_day"] not in only_days):\n                continue\n            name = item["name"]\n            result = {"name": name, "stage_seconds": {}, "status": "partial"}\n            stage = output / (name + ".json")\n            print(f"Audit start {name}", flush=True)\n            try:\n                if item["access"] != "readable" or not item["size_matches"]:\n                    raise ValueError("Missing, inaccessible or changed source file")\n                path = measure_file(spark, root / item["path"], item, output, fingerprints_dir, config, result)\n                if path is not None:\n                    fingerprints.append(path)\n                result["status"] = "complete"\n            except Exception as error:\n                result["error"] = f"{type(error).__name__}: {str(error)[:900]}"\n                print(result["error"], flush=True)\n            atomic_json(stage, result)\n            run["files"][name] = {"status": result["status"], "stage_seconds": result["stage_seconds"]}\n            atomic_json(output / "run_summary.json", run)\n            print(f"Audit {result[\'status\']} {name}: {result.get(\'counts\', {}).get(\'parsed\')} parsed", flush=True)\n\n        if fingerprints:\n            fingerprint_started = time.monotonic()\n            try:\n                run["fingerprint_summary"] = summarize_fingerprints(spark, fingerprints)\n                run["fingerprint_files"] = len(fingerprints)\n            except Exception as error:\n                run["fingerprint_error"] = f"{type(error).__name__}: {str(error)[:900]}"\n                print(run["fingerprint_error"], flush=True)\n            run["fingerprint_seconds"] = round(time.monotonic() - fingerprint_started, 3)\n    finally:\n        run["elapsed_seconds"] = round(time.monotonic() - started, 3)\n        atomic_json(output / "run_summary.json", run)\n        spark.stop()\n    return run\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--config", type=Path, default=Path("configs/dataset.yaml"))\n    parser.add_argument("--root", type=Path, required=True)\n    parser.add_argument("--output", type=Path, required=True)\n    parser.add_argument("--scratch", type=Path, required=True)\n    parser.add_argument("--day", action="append", help="Audit only this ISO capture day; repeat for more days.")\n    parser.add_argument("--exact-duplicates", action="store_true", help="Run only the exact duplicate aggregation.")\n    args = parser.parse_args()\n    config = json.loads(args.config.read_text())\n    if args.exact_duplicates:\n        audit_exact_duplicates(config, args.root, args.output, args.scratch)\n    else:\n        audit_dataset(config, args.root, args.output, args.scratch, args.day)\n\n\nif __name__ == "__main__":\n    main()\n', 'src/ingest/schema_contract.py': '"""Versioned schema, record identity and strict Phase 1 source reconciliation."""\n\nimport csv\nfrom dataclasses import dataclass\nimport json\nfrom pathlib import Path\n\nfrom pyspark.sql import functions as F\n\nfrom src.ingest.dataset_audit import normalize_column, record_frame\n\n\nEXTRA_FIELDS = ("flow_id", "src_ip", "src_port", "dst_ip")\nTEXT_METADATA = ("flow_id", "src_ip", "dst_ip")\nSOURCE_COUNTS = ("parsed", "headers", "blank", "bad_width_or_quotes", "spark_corrupt")\nFEATURE_CONTEXT = ("record_fingerprint", "split_feature_key", "source_day", "temporal_eligibility")\nTARGETS = ("original_label", "normalized_label", "binary_label", "attack_family")\n\n\n@dataclass\nclass DataContract:\n    policy: dict\n\n    @classmethod\n    def load(cls, path):\n        return cls(json.loads(Path(path).read_text(encoding="utf-8")))\n\n    @property\n    def common_values(self):\n        return tuple(name for name in map(normalize_column, self.policy["headers"]["common_80"])\n                     if name not in {"label", "timestamp"})\n\n    @property\n    def feature_names(self):\n        return tuple(name for name in self.common_values if name not in self.policy["excluded_shared_features"])\n\n    def source_variant(self, source_day):\n        return "extended_84" if source_day == self.policy["extended_day"] else "common_80"\n\n    def verified_header(self, path, source_day):\n        variant = self.source_variant(source_day)\n        with path.open("rb") as source:\n            line = source.readline(65536)\n        if not line.endswith(b"\\n"):\n            raise ValueError(f"Missing or oversized header in {path.name}")\n        decoded = line.decode("utf-8-sig").rstrip("\\r\\n")\n        header = next(csv.reader([decoded], strict=True))\n        if header != self.policy["headers"][variant]:\n            raise ValueError(f"Unexpected original header/order in {path.name} (expected {variant})")\n        return variant, decoded, tuple(map(normalize_column, header))\n\n    def label_lookup(self):\n        # Match the exact audited originals. Normalization is for display only.\n        pairs = []\n        for original, values in self.policy["label_mapping"].items():\n            pairs.extend((F.lit(original), F.struct(\n                F.lit(values["normalized"]).alias("normalized"),\n                F.lit(values["binary"]).cast("byte").alias("binary"),\n                F.lit(values["family"]).alias("family"),\n            )))\n        return F.create_map(*pairs)\n\n\ndef source_records(spark, contract, path, source_day):\n    variant, line, names = contract.verified_header(path, source_day)\n    frame = record_frame(spark, path, line, len(names))\n    return variant, frame.select("raw", "kind", *[F.col(f"c{i}").alias(name) for i, name in enumerate(names)])\n\n\ndef record_key(raw, variant):\n    # Use exact UTF-8 source-line text (excluding line break), scoped by header variant.\n    return F.unhex(F.sha2(F.concat(F.lit(variant + "\\u0000"), raw), 256))\n\n\ndef reconcile_source(observed, expected):\n    for name in SOURCE_COUNTS:\n        if observed[name] != expected[name]:\n            raise ValueError(f"Source count mismatch for {name}: {observed[name]} vs {expected[name]}")\n    if observed["physical_lines"] != expected["physical_lines"]:\n        raise ValueError("Source physical lines do not reconcile")\n    return observed | {"repeated_headers": max(0, observed["headers"] - 1)}\n\n\ndef safe_union(frames):\n    """Align only frames that already have matching canonical names and column types."""\n    result = frames[0]\n    expected = {field.name: field.dataType for field in result.schema}\n    for frame in frames[1:]:\n        actual = {field.name: field.dataType for field in frame.schema}\n        if actual != expected:\n            raise ValueError("Canonical frames have different columns or types")\n        result = result.unionByName(frame)\n    return result\n', 'src/ingest/cleaning.py': '"""Phase 2 value-level diagnostics and policy transformations."""\n\nfrom pyspark.sql import Window, functions as F\n\nfrom src.ingest.dataset_audit import NONNEGATIVE, SENTINELS, timestamp_columns\nfrom src.ingest.schema_contract import EXTRA_FIELDS, FEATURE_CONTEXT, TARGETS, TEXT_METADATA, record_key\n\n\ndef issue_rule(item):\n    text, value = item["text"], item["value"]\n    missing = text.isNull() | (F.trim(text) == "")\n    inf = F.lower(F.trim(text)).isin(*sorted(SENTINELS)) | F.coalesce(F.abs(value) == float("inf"), F.lit(False))\n    return (\n        F.when(missing, "source_null")\n        .when(F.isnan(value), "nan")\n        .when(inf & (F.trim(text).startswith("-") | (value < 0)), "negative_infinity")\n        .when(inf, "positive_infinity")\n        .when(value.isNull(), "cast_failure")\n        .when(F.array_contains(item["sentinels"], value), "sentinel")\n        .when(item["nonnegative"] & (value < 0), "invalid_negative")\n    )\n\n\ndef canonical_frame(frame, contract, variant, source_name, source_day, timestamp_formats):\n    numeric_names = tuple(name for name in (*contract.common_values, "src_port") if name in frame.columns)\n    raw_values = F.array(*[F.col(name) for name in numeric_names])\n    config = contract.policy\n    policies = config["numeric_actions"]\n    policy_pairs = [F.lit(v) for pair in policies.items() for v in pair]\n    actions = F.create_map(*policy_pairs)\n    descriptors = F.array(*[\n        F.struct(F.lit(name).alias("field"), F.lit(bool(NONNEGATIVE.search(name) and not name.startswith("init_"))).alias("nonnegative"),\n                 F.array(*[F.lit(value).cast("double") for value in config["sentinel_values"].get(name, [])])\n                 .cast("array<double>").alias("sentinels"))\n        for name in numeric_names\n    ])\n    parsed, _, _ = timestamp_columns(F.col("timestamp"), timestamp_formats)\n    base = (\n        frame.where(F.col("kind") == "parsed")\n        .withColumn("_text", raw_values)\n        .withColumn("_numbers", F.transform("_text", lambda value: F.trim(value).try_cast("double")))\n        .withColumn("_rules", descriptors)\n        .withColumn("_tokens", F.zip_with(F.arrays_zip("_text", "_numbers"), "_rules",\n                                         lambda x, r: F.struct(r["field"].alias("field"), x["_text"].alias("text"),\n                                                              x["_numbers"].alias("value"), r["nonnegative"],\n                                                              r["sentinels"])))\n        .withColumn("_issues", F.transform("_tokens", lambda token: F.struct(\n            token["field"], token["text"].alias("source_text"), issue_rule(token).alias("reason"),\n        )))\n        .withColumn("quality_issues", F.transform(F.filter("_issues", lambda item: item["reason"].isNotNull()),\n                                                 lambda item: F.struct(item["field"], item["source_text"], item["reason"],\n                                                                       actions[item["reason"]].alias("action"))))\n        .withColumn("record_fingerprint", record_key(F.col("raw"), variant))\n        .withColumn("_label", contract.label_lookup()[F.col("label")])\n        .withColumn("parsed_event_time", parsed)\n    )\n    match = F.to_date("parsed_event_time") == F.lit(source_day).cast("date")\n    status = (F.when(F.col("timestamp").isNull() | (F.trim("timestamp") == ""), "missing")\n              .when(F.col("parsed_event_time").isNull(), "unparseable")\n              .when(match, "matched")\n              .otherwise("outside_capture_day"))\n    fields = {name: F.col("_numbers")[i].alias(name) for i, name in enumerate(numeric_names)}\n    for name in EXTRA_FIELDS:\n        if name in TEXT_METADATA:\n            fields[name] = (F.col(name) if name in frame.columns else F.lit(None).cast("string")).alias(name)\n        elif name not in fields:\n            fields[name] = F.lit(None).cast("double").alias(name)\n    missing_fields = [name for name in EXTRA_FIELDS if name not in frame.columns]\n    return base.select(\n        *fields.values(),\n        F.col("label").alias("original_label"), F.col("_label.normalized").alias("normalized_label"),\n        F.col("_label.binary").alias("binary_label"), F.col("_label.family").alias("attack_family"),\n        F.col("timestamp").alias("original_timestamp"), "parsed_event_time",\n        F.lit(source_name).alias("source_file"), F.lit(source_day).alias("source_day"),\n        "record_fingerprint", F.lit(variant).alias("schema_variant"),\n        F.lit(config["version"]).alias("cleaning_policy_version"),\n        F.lit(config["dataset_version"]).cast("int").alias("dataset_version"),\n        status.alias("timestamp_parse_status"), match.alias("timestamp_capture_day_match"),\n        F.coalesce(match, F.lit(False)).alias("temporal_eligibility"),\n        F.array(*[F.lit(name) for name in missing_fields]).cast("array<string>").alias("structural_missing_fields"),\n        "quality_issues",\n    )\n\n\ndef clean_frame(canonical, contract):\n    null_fields = F.transform(F.filter("quality_issues", lambda issue: issue["action"] == "nullify"),\n                              lambda issue: issue["field"])\n    base = canonical.withColumn("_null_fields", null_fields)\n    numeric = {*contract.common_values, "src_port"}\n    return base.select(*[\n        F.when(F.array_contains("_null_fields", name), F.lit(None).cast("double"))\n        .otherwise(F.col(name)).alias(name) if name in numeric else F.col(name)\n        for name in canonical.columns\n    ])\n\n\ndef feature_frame(clean, contract):\n    # Hash the shared cleaned feature vector in fixed contract order, with a policy version prefix.\n    values = F.array(*[F.col(name) for name in contract.feature_names])\n    split_key = F.unhex(F.sha2(F.concat(F.lit(contract.policy["version"] + "\\u0000"), F.to_json(values)), 256))\n    with_key = clean.withColumn("split_feature_key", split_key)\n    return with_key.select(*contract.feature_names, *TARGETS, *FEATURE_CONTEXT)\n\n\ndef exact_groups(frame):\n    return (frame.groupBy("record_fingerprint")\n            .agg(F.count("*").alias("multiplicity"), F.min("source_file").alias("representative_source"))\n            .where(F.col("multiplicity") > 1))\n\n\ndef conflict_groups(frame):\n    return (frame.groupBy("split_feature_key")\n            .agg(F.count("*").alias("rows"), F.count_distinct("original_label").alias("label_count"))\n            .where(F.col("label_count") > 1))\n\n\ndef apply_duplicate_policy(frame, mode):\n    if mode in {"preserve_all", "diagnostic"}:\n        return frame\n    if mode != "exact_representative":\n        raise ValueError(f"Unsupported duplicate policy: {mode}")\n    # Rows tied on source_file and digest are identical visible occurrences in that source.\n    window = Window.partitionBy("record_fingerprint").orderBy("source_file")\n    return frame.withColumn("_representative", F.row_number().over(window)).where("_representative = 1").drop("_representative")\n', 'src/ingest/parquet_pipeline.py': '"""Execute, reconcile and checkpoint the Phase 2 data layers."""\n\nimport argparse\nfrom collections import Counter\nimport csv\nimport json\nimport os\nfrom pathlib import Path\nimport shutil\nimport time\n\nfrom pyspark import StorageLevel\nfrom pyspark.sql import Observation, functions as F\n\nfrom src.ingest.cleaning import apply_duplicate_policy, canonical_frame, clean_frame, conflict_groups, exact_groups, feature_frame\nfrom src.ingest.dataset_audit import atomic_json, count_if, read_inventory, spark_session\nfrom src.ingest.schema_contract import DataContract, SOURCE_COUNTS, reconcile_source, source_records\nfrom src.sql.aggregation_demo import run_aggregation_demo\n\n\nLAYERS = ("raw_aligned", "clean", "feature_ready")\nKINDS = ("nan", "positive_infinity", "negative_infinity", "invalid_negative", "sentinel", "cast_failure", "source_null")\n\n\ndef canonical_metrics():\n    issues = F.col("quality_issues")\n    metrics = [\n        F.count("*").alias("canonical_rows"),\n        count_if(F.col("binary_label") == 0).alias("benign"),\n        count_if(F.col("binary_label") == 1).alias("attack"),\n        count_if(F.col("binary_label").isNull()).alias("unknown_labels"),\n        count_if(F.col("timestamp_parse_status") == "outside_capture_day").alias("timestamp_outlier"),\n        count_if(F.col("timestamp_parse_status") == "unparseable").alias("timestamp_unparseable"),\n        count_if(F.col("timestamp_parse_status") == "missing").alias("timestamp_missing"),\n        count_if(F.col("temporal_eligibility")).alias("temporal_eligible"),\n        count_if(F.size(issues) > 0).alias("quality_affected_rows"),\n        F.coalesce(F.sum(F.size(issues)), F.lit(0)).alias("quality_affected_cells"),\n        count_if(F.size("structural_missing_fields") > 0).alias("structural_missing_rows"),\n        F.coalesce(F.sum(F.size("structural_missing_fields")), F.lit(0)).alias("structural_missing_cells"),\n    ]\n    for reason in KINDS:\n        selected = F.size(F.filter(issues, lambda issue: issue["reason"] == reason))\n        metrics.extend([\n            count_if(selected > 0).alias(reason + "_rows"),\n            F.coalesce(F.sum(selected), F.lit(0)).alias(reason + "_cells"),\n        ])\n    return metrics\n\n\ndef parquet_footers(path):\n    """Verify files and row counts via footer metadata only."""\n    import pyarrow.parquet as parquet\n\n    files = sorted(path.rglob("part-*.parquet"))\n    if not files:\n        raise ValueError(f"No Parquet data files under {path.name}")\n    rows, size = 0, 0\n    codecs = set()\n    for file in files:\n        metadata = parquet.read_metadata(file)\n        rows += metadata.num_rows\n        size += file.stat().st_size\n        for group in range(metadata.num_row_groups):\n            row_group = metadata.row_group(group)\n            codecs.update(row_group.column(i).compression for i in range(row_group.num_columns))\n    if codecs != {"SNAPPY"}:\n        raise ValueError(f"Parquet compression mismatch in {path.name}: {sorted(codecs)}")\n    return {"records": rows, "bytes": size, "files": len(files), "compression": "SNAPPY"}\n\n\ndef write_day(frame, path, number_of_files, expected_rows):\n    started = time.monotonic()\n    (frame.drop("source_day").coalesce(number_of_files).write.mode("overwrite")\n     .option("compression", "snappy").parquet(str(path)))\n    write_s = round(time.monotonic() - started, 3)\n    observed = parquet_footers(path)\n    if observed["records"] != expected_rows:\n        raise ValueError(f"Parquet row-count mismatch at {path.name}")\n    return observed | {"write_seconds": write_s}\n\n\ndef driver_peak_bytes(spark):\n    pid = spark._jvm.java.lang.ProcessHandle.current().pid()\n    status = Path(f"/proc/{pid}/status")\n    if status.is_file():\n        line = next((line for line in status.read_text().splitlines() if line.startswith("VmHWM:")), None)\n        if line is not None:\n            return int(line.split()[1]) * 1024\n    return None\n\n\ndef process_capture(spark, root, stores, contract, source, formats):\n    source_name, day = source["name"], source["capture_day"]\n    path = root / source["path"]\n    expected = source["counts"]\n    start = time.monotonic()\n    variant, frame = source_records(spark, contract, path, day)\n    physical = Observation()\n    measured = frame.observe(\n        physical, F.count("*").alias("physical_lines"),\n        *[count_if(F.col("kind") == name).alias(name) for name in SOURCE_COUNTS],\n    )\n    normalized = Observation()\n    canonical = (canonical_frame(measured, contract, variant, source_name, day, formats)\n                 .observe(normalized, *canonical_metrics()).persist(StorageLevel.DISK_ONLY))\n    target = contract.policy["target_source_bytes_per_part"]\n    parts = max(1, (source["bytes"] + target - 1) // target)\n    try:\n        materialized = canonical.count()\n        parse_seconds = round(time.monotonic() - start, 3)\n        reconciliation = reconcile_source(physical.get, expected)\n        stats = normalized.get\n        if stats["canonical_rows"] != expected["parsed"] or materialized != expected["parsed"]:\n            raise ValueError("Canonical rows failed reconciliation")\n        if stats["unknown_labels"] or stats["benign"] + stats["attack"] != materialized:\n            raise ValueError(f"Unmapped labels in {source_name}: {stats[\'unknown_labels\']}")\n        if stats["timestamp_outlier"] + stats["timestamp_missing"] + stats["timestamp_unparseable"] + stats["temporal_eligible"] != materialized:\n            raise ValueError("Canonical timestamp statuses failed reconciliation")\n        raw_output = write_day(canonical, stores["raw_aligned"] / f"source_day={day}", parts, materialized)\n        clean_output = write_day(clean_frame(canonical, contract), stores["clean"] / f"source_day={day}", parts, materialized)\n    finally:\n        canonical.unpersist(blocking=True)\n    # The third layer is built from the persisted clean data.\n    clean_path = stores["clean"] / f"source_day={day}"\n    feature_source = spark.read.parquet(str(clean_path)).withColumn("source_day", F.lit(day))\n    feature_output = write_day(feature_frame(feature_source, contract), stores["feature_ready"] / f"source_day={day}", parts, materialized)\n    return {\n        "source_file": source_name, "source_day": day, "schema_variant": variant,\n        "source_bytes": source["bytes"], "expected_records": expected["parsed"],\n        "expected_headers": expected["headers"], "read_transform_seconds": parse_seconds,\n        "effective_read_mib_s": round(source["bytes"] / (1024 ** 2) / max(parse_seconds, 0.001), 3),\n        "driver_peak_rss_bytes": driver_peak_bytes(spark),\n        "counts": reconciliation | stats,\n        "layers": dict(zip(LAYERS, (raw_output, clean_output, feature_output))),\n    }\n\n\ndef reconcile_full(summary, expected):\n    counts = summary["totals"]\n    observed = {key: counts[key] for key in ("benign", "attack", "timestamp_outlier")}\n    observed.update(nan=counts["nan_cells"], infinity=counts["positive_infinity_cells"] + counts["negative_infinity_cells"],\n                    invalid_negative=counts["invalid_negative_cells"])\n    if any(observed[key] != expected[key] for key in observed):\n        raise ValueError(f"Phase 1 totals disagree: {observed}")\n    if counts["canonical_rows"] != counts["parsed"] or any(counts["canonical_rows"] != summary["layers"][name]["records"] for name in LAYERS):\n        raise ValueError("Layer totals do not reconcile")\n\n\ndef layer_summary(files, data_root):\n    output = {}\n    for name in LAYERS:\n        metrics = [row["layers"][name] for row in files]\n        output[name] = {\n            "path": (data_root / name).as_posix(), "partition_column": "source_day",\n            "partitions": len(metrics), "format": "parquet", "compression": "SNAPPY",\n            **{key: sum(item[key] for item in metrics) for key in ("records", "bytes", "files")},\n            "write_seconds": round(sum(item["write_seconds"] for item in metrics), 3),\n        }\n    return output\n\n\ndef write_csv(path, rows, fields):\n    with path.open("w", encoding="utf-8", newline="") as stream:\n        writer = csv.DictWriter(stream, fields, lineterminator="\\n")\n        writer.writeheader()\n        writer.writerows(rows)\n\n\ndef write_evidence(evidence, summary, policy):\n    atomic_json(evidence / "validation_summary.json", summary)\n    atomic_json(evidence / "parquet_layers.json", {\n        key: summary[key] for key in ("status", "dataset", "policy_version", "layers", "duplicate_policy", "duplicate_diagnostics")\n        if key in summary\n    })\n    captured = summary["completed_files"]\n    counts = [{\n        "source_file": item["source_file"], "source_day": item["source_day"],\n        "expected_records": item["expected_records"], "observed_records": item["counts"]["parsed"],\n        "physical_lines": item["counts"]["physical_lines"], "expected_headers": item["expected_headers"],\n        "headers": item["counts"]["headers"], "repeated_headers": item["counts"]["repeated_headers"],\n        "canonical": item["counts"]["canonical_rows"], "clean": item["layers"]["clean"]["records"],\n        "feature_ready": item["layers"]["feature_ready"]["records"], "benign": item["counts"]["benign"],\n        "attack": item["counts"]["attack"], "timestamp_outlier": item["counts"]["timestamp_outlier"],\n    } for item in captured]\n    timing = [{\n        "source_file": item["source_file"], "source_bytes": item["source_bytes"],\n        "read_transform_seconds": item["read_transform_seconds"], "effective_read_mib_s": item["effective_read_mib_s"],\n        "driver_peak_rss_bytes": item["driver_peak_rss_bytes"],\n        **{name + "_write_seconds": item["layers"][name]["write_seconds"] for name in LAYERS},\n    } for item in captured]\n    quality = [{\n        "source_file": item["source_file"], "reason": reason,\n        "affected_rows": item["counts"][reason + "_rows"], "affected_cells": item["counts"][reason + "_cells"],\n        "action": policy["numeric_actions"].get(reason, "structural_null" if reason == "structural_missing" else "all_quality_issues"),\n    } for item in captured for reason in (*KINDS, "structural_missing", "quality_affected")]\n    if counts:\n        write_csv(evidence / "phase2_count_reconciliation.csv", counts, list(counts[0]))\n        write_csv(evidence / "phase2_performance.csv", timing, list(timing[0]))\n        write_csv(evidence / "phase2_quality_actions.csv", quality, list(quality[0]))\n\n\ndef run_duplicate_mode(spark, stores, policy, data_root, expected_excess):\n    mode = policy["duplicate_policy"]\n    if mode == "preserve_all":\n        return {"records_removed_primary": 0}\n    if mode not in {"diagnostic", "exact_representative"}:\n        raise ValueError(f"Unknown duplicate policy {mode}")\n    started = time.monotonic()\n    folder = data_root / "diagnostics"\n    folder.mkdir(exist_ok=True)\n    aligned = spark.read.parquet(str(stores["raw_aligned"])).select("record_fingerprint", "source_file")\n    keys = folder / "exact_groups"\n    exact_groups(aligned).write.mode("overwrite").option("compression", "snappy").parquet(str(keys))\n    exact = spark.read.parquet(str(keys)).agg(F.count("*").alias("groups"),\n                                              F.coalesce(F.sum(F.col("multiplicity") - 1), F.lit(0)).alias("excess")).first().asDict()\n    if expected_excess is not None and exact["excess"] != expected_excess:\n        raise ValueError("Exact raw-line duplicate excess differs from Phase 1")\n    features = spark.read.parquet(str(stores["feature_ready"])).select("split_feature_key", "original_label")\n    conflicts_path = folder / "label_conflicts"\n    conflict_groups(features).write.mode("overwrite").option("compression", "snappy").parquet(str(conflicts_path))\n    conflicts = spark.read.parquet(str(conflicts_path)).agg(F.count("*").alias("groups"),\n        F.coalesce(F.sum("rows"), F.lit(0)).alias("participating_rows")).first().asDict()\n    result = {"records_removed_primary": 0, "exact": exact, "cleaned_feature_conflicts": conflicts,\n              "diagnostics_path": folder.as_posix()}\n    if mode == "exact_representative":\n        clean = spark.read.parquet(str(stores["clean"]))\n        target = data_root / "exact_representatives"\n        (apply_duplicate_policy(clean, mode).write.mode("overwrite").option("compression", "snappy")\n         .partitionBy("source_day").parquet(str(target)))\n        measured = parquet_footers(target)\n        if measured["records"] != sum(parquet_footers(p)["records"] for p in stores["clean"].glob("source_day=*")) - exact["excess"]:\n            raise ValueError("Representative output failed multiplicity reconciliation")\n        result["representatives"] = measured | {"path": target.as_posix()}\n    return result | {"seconds": round(time.monotonic() - started, 3)}\n\n\ndef resource_limits(spark, policy, path):\n    metrics = {"spark_version": spark.version, **policy["spark"], "default_parallelism": spark.sparkContext.defaultParallelism,\n               "cpu_affinity": len(os.sched_getaffinity(0)), "scratch_output_free_bytes_start": shutil.disk_usage(path).free}\n    for name in ("memory.max", "cpu.max"):\n        resource = Path("/sys/fs/cgroup") / name\n        if resource.exists():\n            metrics[name] = resource.read_text().strip()\n    return metrics\n\n\ndef run_pipeline(contract, source_config, phase1, root, output, scratch, only_days=None):\n    root, output, scratch = map(Path, (root, output, scratch))\n    if not root.is_dir() or any(p.resolve().is_relative_to(root.resolve()) for p in (output, scratch)):\n        raise ValueError("The verified mount and writable outputs must be separate")\n    output.mkdir(parents=True, exist_ok=True)\n    scratch.mkdir(parents=True, exist_ok=True)\n    actual = read_inventory(root, source_config)\n    audited = {row["path"]: row for row in phase1["files"]}\n    if len(actual) != len(audited) or any(\n        row["path"] not in audited or row["bytes"] != audited[row["path"]]["bytes"]\n        or row["access"] != "readable" or row["unexpected"] or row["duplicate_basename"] for row in actual\n    ):\n        raise ValueError("Mounted inventory differs from Phase 1")\n    policy = contract.policy\n    data_root = output / policy["version"]\n    stores = {name: data_root / name for name in LAYERS}\n    evidence = output / "evidence"\n    evidence.mkdir(exist_ok=True)\n    spark = spark_session({"spark": policy["spark"]}, scratch)\n    spark.conf.set("spark.sql.sources.partitionColumnTypeInference.enabled", "false")\n    spark.sparkContext.setLogLevel("WARN")\n    summary = {\n        "status": "running", "dataset": phase1["dataset"], "policy_version": policy["version"],\n        "duplicate_policy": policy["duplicate_policy"], "expected_files": len(audited),\n        "resources": resource_limits(spark, policy, output), "completed_files": [], "failures": [],\n        "phase1_conflict_groups": 22129,\n    }\n    started = time.monotonic()\n    totals = Counter()\n    try:\n        sources = sorted(audited.values(), key=lambda file: file["bytes"])\n        for source in sources:\n            day = source["capture_day"]\n            if only_days is not None and day not in only_days:\n                continue\n            try:\n                marker = evidence / f"capture-{day}.json"\n                if marker.is_file():\n                    result = json.loads(marker.read_text())\n                    for name in LAYERS:\n                        metrics = parquet_footers(stores[name] / f"source_day={day}")\n                        if any(result["layers"][name][key] != value for key, value in metrics.items()):\n                            raise ValueError("Existing checkpoint no longer matches its data files")\n                else:\n                    print(f"PHASE2 start {day} {source[\'bytes\']} bytes", flush=True)\n                    result = process_capture(spark, root, stores, contract, source, source_config["timestamp_formats"])\n                    atomic_json(marker, result)\n                summary["completed_files"].append(result)\n                totals.update(result["counts"])\n                print(f"PHASE2 completed {day} {result[\'counts\'][\'canonical_rows\']} rows "\n                      f"in {result[\'read_transform_seconds\']} s read and transform", flush=True)\n            except Exception as error:\n                summary["failures"].append({"source_day": day, "stage": "capture", "type": type(error).__name__})\n                print(f"PHASE2 failed {day}: {type(error).__name__}: {str(error)[:1000]}", flush=True)\n            summary.update(totals=dict(totals), layers=layer_summary(summary["completed_files"], data_root))\n            write_evidence(evidence, summary, policy)\n        full = len(summary["completed_files"]) == len(sources)\n        if full:\n            reconcile_full(summary, policy["expected_full"])\n            stored = stores["feature_ready"] / f"source_day={sources[0][\'capture_day\']}"\n            summary["aggregation_demo"] = run_aggregation_demo(spark.read.parquet(str(stored)))\n            summary["duplicate_diagnostics"] = run_duplicate_mode(\n                spark, stores, policy, data_root, policy["expected_full"]["duplicate_excess"])\n        summary["status"] = "validated" if full else "partial"\n    except Exception as error:\n        summary["status"] = "partial"\n        summary["failures"].append({"stage": "global_validation", "type": type(error).__name__})\n        print(f"PHASE2 validation failed: {type(error).__name__}: {str(error)[:1000]}", flush=True)\n    finally:\n        summary["elapsed_seconds"] = round(time.monotonic() - started, 3)\n        summary["resources"]["driver_peak_rss_bytes"] = driver_peak_bytes(spark)\n        write_evidence(evidence, summary, policy)\n        print(f"PHASE2 final {summary[\'status\']} in {summary[\'elapsed_seconds\']} seconds", flush=True)\n        # Persist compact evidence in logs as well as the notebook output.\n        print("PHASE2_EVIDENCE " + json.dumps(summary, separators=(",", ":")), flush=True)\n        spark.stop()\n    return summary\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--root", type=Path, required=True)\n    parser.add_argument("--output", type=Path, required=True)\n    parser.add_argument("--scratch", type=Path, required=True)\n    parser.add_argument("--day", action="append")\n    args = parser.parse_args()\n    contract = DataContract.load("configs/cleaning.yaml")\n    load = lambda path: json.loads(Path(path).read_text())\n    summary = run_pipeline(contract, load("configs/dataset.yaml"), load("manifests/dataset_inventory.json"),\n                           args.root, args.output, args.scratch, args.day)\n    raise SystemExit(0 if summary["status"] == "validated" else 1)\n\n\nif __name__ == "__main__":\n    main()\n', 'src/sql/aggregation_demo.py': '"""Compare equivalent SQL, DataFrame and RDD aggregations on a bounded sample."""\n\nfrom pyspark.sql import functions as F\n\n\ndef run_aggregation_demo(features):\n    sample = (features.select("protocol", "binary_label", "record_fingerprint")\n              .where(F.pmod(F.xxhash64("record_fingerprint"), F.lit(1024)) == 0)\n              .select("protocol", "binary_label").cache())\n    sample.createOrReplaceTempView("phase2_demo_sample")\n    try:\n        sql = features.sparkSession.sql(\n            "SELECT protocol, count(*) AS records, sum(binary_label) AS attacks "\n            "FROM phase2_demo_sample GROUP BY protocol")\n        dataframe = sample.groupBy("protocol").agg(F.count("*").alias("records"), F.sum("binary_label").alias("attacks"))\n        rdd = (sample.rdd.map(lambda row: (row.protocol, (1, row.binary_label)))\n               .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1])))\n        result = {row.protocol: (row.records, row.attacks) for row in sql.collect()}\n        if result != {row.protocol: (row.records, row.attacks) for row in dataframe.collect()} or result != dict(rdd.collect()):\n            raise ValueError("SQL, DataFrame and RDD aggregation disagree")\n        if not result:\n            raise ValueError("The demonstration sample is empty")\n        return {\n            "sample_rule": "smallest capture by source bytes; pmod(xxhash64(record_fingerprint), 1024) == 0",\n            "rows": sum(count for count, _ in result.values()),\n            "groups": [{"protocol": key, "records": value[0], "attacks": value[1]}\n                       for key, value in sorted(result.items())],\n            "sql_plan": sql._jdf.queryExecution().executedPlan().toString(),\n            "dataframe_plan": dataframe._jdf.queryExecution().executedPlan().toString(),\n            "rdd_dag": rdd.toDebugString().decode("utf-8"),\n        }\n    finally:\n        features.sparkSession.catalog.dropTempView("phase2_demo_sample")\n        sample.unpersist()\n', 'src/sql/traffic_analytics.py': '"""Validated input and reusable SQL aggregations for Phase 3."""\n\nfrom collections import Counter\nimport csv\nfrom hashlib import sha256\nimport json\nimport math\nfrom pathlib import Path\nimport re\nimport time\n\nfrom pyspark.sql import functions as F\n\nfrom src.ingest.dataset_audit import atomic_json\nfrom src.ingest.parquet_pipeline import parquet_footers\nfrom src.ingest.schema_contract import FEATURE_CONTEXT, TARGETS\n\n\ndef identifier(name):\n    if not re.fullmatch(r"[a-z][a-z0-9_]*", name):\n        raise ValueError(f"Invalid configured SQL identifier {name!r}")\n    return name\n\n\ndef locate_producer(input_root, expected_sha):\n    relative = "mmds_phase2/evidence/validation_summary.json"\n    root = Path(input_root)\n    candidates = set(root.rglob(relative))\n    for depth in (1, 2, 3, 4):\n        candidates.update(root.glob("*/" * depth + relative))\n    matched = {path.resolve(): path for path in sorted(candidates)\n               if sha256(path.read_bytes()).hexdigest() == expected_sha}\n    if len(matched) != 1:\n        raise ValueError("Expected one mounted input matching the Phase 2 receipt")\n    return next(iter(matched.values()))\n\n\ndef require(condition, message):\n    if not condition:\n        raise ValueError(message)\n\n\ndef label_grouping(frame):\n    return frame.groupBy("source_day", *TARGETS, "temporal_eligibility").count()\n\n\ndef compact(frame, maximum):\n    rows = frame.limit(maximum + 1).collect()\n    require(len(rows) <= maximum, "Aggregate exceeds compact output limit")\n    def scalar(value):\n        if isinstance(value, float):\n            return value if math.isfinite(value) else None\n        return value\n\n    return [{k: scalar(v) for k, v in row.asDict().items()} for row in rows]\n\n\ndef validate_input(spark, receipt, manifest, policy):\n    summary = json.loads(receipt.read_text())\n    expected_sha = manifest["independent_mount"]["summary_sha256"]\n    require(sha256(receipt.read_bytes()).hexdigest() == expected_sha, "Phase 2 receipt checksum differs")\n    require(summary["status"] == "validated" and summary["dataset"] == manifest["dataset"], "Wrong producer dataset")\n    require(summary["policy_version"] == manifest["policy_version"] == policy["version"], "Wrong cleaning policy")\n    require(summary["totals"] == manifest["totals"], "Wrong producer totals")\n    root = receipt.parent.parent / summary["policy_version"]\n    frames = {}\n    evidence = {}\n    for name in ("feature_ready", "clean", "raw_aligned"):\n        directory = root / name\n        actual = parquet_footers(directory)\n        expected = manifest["layers"][name]\n        days = sorted(path.name.split("=", 1)[1] for path in directory.glob("source_day=*") if path.is_dir())\n        require(all(expected[key] == value for key, value in actual.items()), f"{name} is unreadable or differs from manifest")\n        require(len(days) == expected["partitions"] == 10, f"{name} partition count differs")\n        require(days == sorted(row["source_day"] for row in summary["completed_files"]), f"{name} day list differs")\n        frames[name] = spark.read.parquet(str(directory))\n        evidence[name] = actual | {"path": str(directory), "partitions": len(days)}\n    features, clean = frames["feature_ready"], frames["clean"]\n    require(set(features.columns) == set(manifest["feature_columns"]) | set(FEATURE_CONTEXT) | set(TARGETS), "Feature schema differs")\n    for frame in (features, clean):\n        types = dict(frame.dtypes)\n        require(all(types.get(name) == "double" for name in manifest["feature_columns"]), "Feature types differ")\n        require(types.get("source_day") == "string" and types.get("binary_label") == "tinyint", "Target or partition type differs")\n        require(types.get("record_fingerprint") == "binary" and types.get("temporal_eligibility") == "boolean", "Provenance types differ")\n    require(dict(clean.dtypes).get("parsed_event_time") == "timestamp_ntz", "Wall-clock timestamp is unavailable")\n    require({row[0] for row in clean.select("cleaning_policy_version").distinct().collect()} == {policy["version"]}, "Stored cleaning policy differs")\n    clean_groups = compact(label_grouping(clean).orderBy("source_day", "original_label", "temporal_eligibility"), 500)\n    feature_groups = compact(label_grouping(features).orderBy("source_day", "original_label", "temporal_eligibility"), 500)\n    require(clean_groups == feature_groups, "Layer targets or temporal flags differ")\n    expected_totals = summary["totals"]\n    labels, day_counts = Counter(), Counter()\n    ineligible = 0\n    for row in clean_groups:\n        label = policy["label_mapping"].get(row["original_label"])\n        require(label is not None and (row["normalized_label"], row["binary_label"], row["attack_family"]) ==\n                (label["normalized"], label["binary"], label["family"]), "Observed label mapping differs")\n        require(row["temporal_eligibility"] in (True, False), "Null temporal flag")\n        labels[row["binary_label"]] += row["count"]\n        day_counts[row["source_day"]] += row["count"]\n        if not row["temporal_eligibility"]:\n            ineligible += row["count"]\n    require(dict(labels) == {0: expected_totals["benign"], 1: expected_totals["attack"]}, "Binary counts differ")\n    require(sum(labels.values()) == expected_totals["canonical_rows"] == 16232943 and ineligible == 14, "Row or temporal counts differ")\n    require(dict(day_counts) == {row["source_day"]: row["counts"]["canonical_rows"] for row in summary["completed_files"]}, "Per-day counts differ")\n    bad_time = clean.where(\n        ~F.col("temporal_eligibility").eqNullSafe(\n            F.col("parsed_event_time").isNotNull() & (F.to_date("parsed_event_time") == F.col("source_day").cast("date"))))\n    require(bad_time.limit(1).count() == 0, "Temporal eligibility disagrees with stored event time")\n    return frames, root, {\n        "status": "validated", "dataset": summary["dataset"], "phase2_receipt_sha256": expected_sha,\n        "policy_version": summary["policy_version"], "producer": manifest["producer"], "layers": evidence,\n        "records": sum(labels.values()), "benign": labels[0], "attack": labels[1], "temporal_ineligible": ineligible,\n        "source_days": sorted(day_counts), "features": manifest["feature_columns"],\n    }\n\n\ndef finite(column):\n    name = identifier(column)\n    return f"CASE WHEN {name} IS NOT NULL AND NOT isnan({name}) AND abs({name}) < CAST(\'Infinity\' AS DOUBLE) THEN {name} END"\n\n\ndef register_traffic(clean, metrics):\n    """Project only the columns needed by exploratory queries."""\n    selected = set(metrics) - {"total_packets", "total_bytes"}\n    columns = ["source_day", *TARGETS, "temporal_eligibility", "parsed_event_time",\n               *sorted(selected | {"protocol", "dst_port", "fin_flag_cnt"})]\n    projected = (clean.select(*columns, "quality_issues")\n                 .withColumn("quality_cells", F.size("quality_issues"))\n                 .withColumn("invalid_cells", F.size(F.filter("quality_issues", lambda issue: issue["reason"] != "sentinel")))\n                 .drop("quality_issues"))\n    return (\n        projected.withColumn("population", F.when(F.col("binary_label") == 0, "benign").otherwise("attack"))\n        .withColumn("total_packets", F.col("tot_fwd_pkts") + F.col("tot_bwd_pkts"))\n        .withColumn("total_bytes", F.col("totlen_fwd_pkts") + F.col("totlen_bwd_pkts"))\n    )\n\n\nDAILY_SQL = """SELECT source_day, count(*) AS records, count_if(binary_label = 0) AS benign,\n count_if(binary_label = 1) AS attacks, avg(CAST(binary_label AS DOUBLE)) AS attack_share,\n sum(total_packets) AS packet_count, sum(total_bytes) AS byte_count\n FROM traffic GROUP BY source_day ORDER BY source_day"""\n\nTOTAL_SQL = """SELECT population, count(*) AS records, sum(total_packets) AS packet_count,\n sum(total_bytes) AS byte_count, count(total_packets) AS packet_observations,\n count(total_bytes) AS byte_observations FROM traffic GROUP BY population ORDER BY population"""\n\nFAMILY_SQL = """SELECT attack_family, count(*) AS records, count(*) / sum(count(*)) OVER () AS share_of_attacks,\n count(DISTINCT source_day) AS capture_days,\n count_if(quality_cells > 0) AS quality_affected_rows, avg(CAST(quality_cells > 0 AS DOUBLE)) AS quality_affected_share,\n count_if(invalid_cells > 0) AS invalid_affected_rows, avg(CAST(invalid_cells > 0 AS DOUBLE)) AS invalid_affected_share\n FROM traffic WHERE binary_label = 1 GROUP BY attack_family ORDER BY records DESC, attack_family"""\n\nLABEL_SQL = """SELECT attack_family, original_label, count(*) AS records, count(DISTINCT source_day) AS capture_days\n FROM traffic GROUP BY attack_family, original_label ORDER BY attack_family, original_label"""\n\nPROTOCOL_SQL = """SELECT population, attack_family, protocol, count(*) AS records\n FROM traffic GROUP BY population, attack_family, protocol ORDER BY population, attack_family, protocol"""\n\nPORT_SQL = """WITH grouped AS (\n SELECT population, coalesce(attack_family, \'All families\') AS attack_family,\n CASE WHEN grouping(attack_family) = 1 THEN \'population\' ELSE \'family\' END AS scope,\n dst_port, count(*) AS records FROM traffic\n GROUP BY GROUPING SETS ((population, attack_family, dst_port), (population, dst_port))),\n ranked AS (SELECT *, row_number() OVER (PARTITION BY scope, population, attack_family ORDER BY records DESC, dst_port) AS rank,\n sum(records) OVER (PARTITION BY scope, population, attack_family) AS population_records FROM grouped)\n SELECT * FROM ranked WHERE rank <= {limit} ORDER BY scope, population, attack_family, rank"""\n\ndef feature_statistics_sql(features, group, accuracy):\n    group = identifier(group)\n    terms = []\n    for feature in features:\n        x = finite(feature)\n        terms.append(\n            f"named_struct(\'feature\', \'{feature}\', \'valid\', count({x}), \'population_count\', count(*), "\n            f"\'mean\', avg({x}), \'stddev\', stddev_samp({x}), \'min\', min({x}), \'max\', max({x}), "\n            f"\'quantiles\', percentile_approx({x}, array(0.1D, 0.5D, 0.9D, 0.99D), {int(accuracy)}))"\n        )\n    return (f"SELECT {group}, m.* FROM (SELECT {group}, explode(array({\', \'.join(terms)})) AS m "\n            f"FROM traffic GROUP BY {group}) ORDER BY {group}, m.feature")\n\n\ndef missing_sql(features):\n    terms = [\n        f"named_struct(\'feature\', \'{name}\', \'missing\', count(*) - count({finite(name)}), \'records\', count(*))"\n        for name in features\n    ]\n    return f"SELECT m.* FROM (SELECT explode(array({\', \'.join(terms)})) AS m FROM features) ORDER BY m.feature"\n\n\ndef correlations_sql(pairs):\n    terms = []\n    for x, y in pairs:\n        a, b = finite(x), finite(y)\n        terms.append(\n            f"named_struct(\'x\', \'{x}\', \'y\', \'{y}\', \'pairs\', count_if(({a}) IS NOT NULL AND ({b}) IS NOT NULL), "\n            f"\'pearson\', corr({a}, {b}))"\n        )\n    return (f"SELECT population, m.* FROM (SELECT population, explode(array({\', \'.join(terms)})) AS m "\n            "FROM traffic GROUP BY population) ORDER BY population, m.x, m.y")\n\n\ndef write_rows(destination, rows):\n    if not rows:\n        raise ValueError(f"Unexpected empty result {destination.name}")\n    stage = destination.with_suffix(".tmp")\n    with stage.open("w", encoding="utf-8", newline="") as stream:\n        writer = csv.DictWriter(stream, fieldnames=list(rows[0]), lineterminator="\\n")\n        writer.writeheader()\n        writer.writerows({key: json.dumps(value, separators=(",", ":")) if isinstance(value, (list, dict)) else value\n                          for key, value in row.items()} for row in rows)\n    stage.replace(destination)\n\n\ndef timed_sql(spark, query, limit, name, output, performance):\n    started = time.monotonic()\n    rows = compact(spark.sql(query), limit)\n    seconds = round(time.monotonic() - started, 6)\n    write_rows(output / f"phase3_{name}.csv", rows)\n    performance.append({"workload": name, "api": "SQL", "iteration": 1, "seconds": seconds, "result_rows": len(rows)})\n    print(f"PHASE3 query {name}: {len(rows)} result rows in {seconds:.3f} s", flush=True)\n    return rows\n\n\ndef write_checkpoint(output, validation, completed, performance):\n    atomic_json(output / "phase3_validation.json", validation | {"completed_queries": list(completed)})\n    if performance:\n        write_rows(output / "phase3_query_performance.csv", performance)\n', 'src/ml/feature_registry.py': '"""Phase 4 predictor contract and the predictor-equivalence key."""\n\nfrom pyspark.sql import functions as F\n\n\nMETA = ("source_day", "binary_label", "attack_family", "original_label", "temporal_eligibility")\nFORBIDDEN = {\n    *META, "normalized_label", "dst_port", "src_port", "src_ip", "dst_ip", "flow_id",\n    "source_file", "parsed_event_time", "original_timestamp", "record_fingerprint",\n    "split_feature_key", "predictor_key", "schema_variant", "quality_issues", "protocol",\n    "init_fwd_win_byts", "init_bwd_win_byts",\n}\n\n\ndef registry(config, upstream):\n    names = config["numerical_features"]\n    if len(names) != len(set(names)) or not names or set(names) & FORBIDDEN or not set(names) <= set(upstream):\n        raise ValueError("Invalid Phase 4 predictor registry")\n    return list(names)\n\n\ndef finite(column):\n    value = F.col(column).cast("double")\n    return F.when(value.isNotNull() & ~F.isnan(value) & (F.abs(value) < float("inf")),\n                  F.when(value == 0.0, F.lit(0.0)).otherwise(value))\n\n\ndef predictor_input(frame, names, version):\n    """Hash the actual primary scalar vector before training-only transformations."""\n    data = frame.select(*META, "record_fingerprint", "split_feature_key", *map(lambda name: finite(name).alias(name), names))\n    return data.withColumn("predictor_key", F.unhex(F.sha2(\n        F.concat(F.lit(version + "\\u0000"), F.to_json(F.array(*names))), 256,\n    )))\n', 'src/ml/split_protocols.py': '"""Deterministic disjoint predictor cohorts and ordered capture-day policies."""\n\nfrom pyspark.sql import functions as F\n\n\nSPLITS = ("train", "validation", "test")\n\n\ndef check_days(protocol, design, days):\n    selected = [day for name in SPLITS for day in design[name]]\n    if len(set(selected)) != len(selected) or set(selected) != set(days) or any(not design[name] for name in SPLITS):\n        raise ValueError("Capture days must map exactly once to a nonempty partition")\n    if protocol == "B" and not (max(design["train"]) < min(design["validation"])\n                                and max(design["validation"]) < min(design["test"])):\n        raise ValueError("Chronological holdout must be strictly ordered")\n\n\ndef assign_population(frame, protocol, config, days):\n    if protocol == "A":\n        ratios = config["random_permyriad"]\n        if sum(ratios.values()) != 10000 or any(ratios[name] <= 0 for name in SPLITS):\n            raise ValueError("Invalid random split proportions")\n        index = F.pmod(F.xxhash64(F.lit(config["seed"]), "predictor_key"), F.lit(10000))\n        return frame.withColumn("split_rank", F.when(index < ratios["train"], 0)\n                                .when(index < ratios["train"] + ratios["validation"], 1).otherwise(2))\n    if protocol not in ("B", "C"):\n        raise ValueError(f"Unknown protocol {protocol}")\n    design = config["protocols"][protocol]\n    check_days(protocol, design, days)\n    pairs = [value for rank, name in enumerate(SPLITS) for day in design[name] for value in (F.lit(day), F.lit(rank))]\n    return frame.withColumn("split_rank", F.create_map(*pairs)[F.col("source_day")])\n\n\ndef crossing_groups(frame, key):\n    return (frame.groupBy(key).agg(F.min("split_rank").alias("first_rank"), F.max("split_rank").alias("last_rank"),\n                                   F.count("*").alias("occurrences"))\n            .where("first_rank <> last_rank"))\n\n\ndef mark_retention(assigned, crossing, count):\n    groups = crossing.select("predictor_key", "first_rank")\n    # Bound executor replication and use Spark\'s normal join above the cap.\n    groups = F.broadcast(groups) if count <= 400000 else groups\n    return (assigned.join(groups, "predictor_key", "left")\n            .withColumn("retained", F.col("first_rank").isNull() | (F.col("first_rank") == F.col("split_rank")))\n            .drop("first_rank"))\n\n\ndef group_sample(frame, config):\n    fraction = config["group_sample_permyriad"]\n    if not 0 < fraction <= 10000:\n        raise ValueError("Invalid grouped sampling fraction")\n    return frame.withColumn("sampled", F.pmod(F.xxhash64(F.lit(config["seed"]), F.lit("training-sample"), "predictor_key"),\n                                             F.lit(10000)) < fraction)\n\n\ndef group_summary(groups):\n    row = groups.agg(F.count("*").alias("groups"), F.coalesce(F.sum("occurrences"), F.lit(0)).alias("rows")).first()\n    return row.asDict()\n\n\ndef label_conflicts(frame):\n    return (frame.groupBy("predictor_key")\n            .agg(F.min("binary_label").alias("low"), F.max("binary_label").alias("high"),\n                 F.count("*").alias("occurrences"))\n            .where("low <> high"))\n', 'src/ml/preprocessing.py': '"""Spark MLlib preprocessing with an explicit training-only fit boundary."""\n\nfrom pyspark.ml import Pipeline, PipelineModel\nfrom pyspark.ml.feature import Imputer, StandardScaler, VectorAssembler\nfrom pyspark.sql import functions as F\n\n\ndef fit_preprocessing(train, names):\n    if train.where("split_rank <> 0 OR split_rank IS NULL").limit(1).count():\n        raise ValueError("Preprocessing requires train rows only")\n    metrics = train.agg(*[\n        expr.alias(f"{name}_{key}")\n        for name in names for key, expr in (("min", F.min(name)), ("max", F.max(name)))\n    ]).first().asDict()\n    active = [name for name in names if metrics[f"{name}_min"] is not None and metrics[f"{name}_min"] != metrics[f"{name}_max"]]\n    if not active:\n        raise ValueError("Training population has no varying finite predictors")\n    imputed = [name + "_imputed" for name in active]\n    model = Pipeline(stages=[\n        Imputer(strategy="median", relativeError=0.001, inputCols=active, outputCols=imputed),\n        VectorAssembler(inputCols=imputed, outputCol="features", handleInvalid="error"),\n    ]).fit(train)\n    scaler = StandardScaler(inputCol="features", outputCol="scaled_features", withStd=True,\n                            withMean=False).fit(model.transform(train))\n    scaled = PipelineModel(stages=[*model.stages, scaler])\n    medians = model.stages[0].surrogateDF.first().asDict()\n    return model, scaled, {"active": active, "dropped_train_constant_or_empty": [name for name in names if name not in active],\n                           "training_medians": medians}\n\n\ndef verify_vectors(assembled, active_count):\n    from pyspark.ml.functions import vector_to_array\n\n    for name in ("features", "scaled_features"):\n        values = vector_to_array(name)\n        invalid = (F.size(values) != active_count) | F.exists(\n            values, lambda item: item.isNull() | F.isnan(item) | (F.abs(item) == float("inf")))\n        if assembled.where(invalid).limit(1).count():\n            raise ValueError(f"Nonfinite or invalid {name} supplied to estimators")\n', 'src/ml/baselines.py': '"""Two prespecified binary baselines using training data only."""\n\nfrom pyspark.sql import functions as F\n\n\ndef fit_baselines(train, benign, attacks):\n    if train.where("split_rank <> 0 OR split_rank IS NULL").limit(1).count():\n        raise ValueError("Baseline calibration requires train rows only")\n    threshold = train.where("binary_label = 0").agg(F.percentile_approx("flow_duration", 0.5, 4096)).first()[0]\n    if threshold is None:\n        raise ValueError("Benign reference is unavailable for duration baseline")\n    return {"majority": {"attack_prediction": int(attacks > benign)}, "duration_rule": {"benign_training_median_us": threshold}}\n\n\ndef baseline_score(frame, name, parameters):\n    if name == "majority":\n        score = F.lit(float(parameters["attack_prediction"]))\n    elif name == "duration_rule":\n        score = F.when(F.col("flow_duration") <= parameters["benign_training_median_us"], 1.0).otherwise(0.0)\n    else:\n        raise ValueError(f"Unknown baseline {name}")\n    return frame.withColumn("score", score)\n', 'src/ml/evaluate_supervised.py': '"""Distributed exact confusion counts and bounded score-bin AUC estimates."""\n\nfrom collections import Counter\nimport time\n\nfrom pyspark.ml.functions import vector_to_array\nfrom pyspark.sql import functions as F\n\n\ndef rate(numerator, denominator):\n    return numerator / denominator if denominator else None\n\n\ndef confusion_metrics(tp, tn, fp, fn):\n    recall, specificity = rate(tp, tp + fn), rate(tn, tn + fp)\n    return {"tp": tp, "tn": tn, "fp": fp, "fn": fn, "rows": tp + tn + fp + fn,\n            "accuracy": rate(tp + tn, tp + tn + fp + fn), "precision": rate(tp, tp + fp), "recall": recall,\n            "specificity": specificity, "f1": rate(2 * tp, 2 * tp + fp + fn),\n            "balanced_accuracy": (recall + specificity) / 2 if recall is not None and specificity is not None else None}\n\n\ndef binned_auc(histogram):\n    positives = sum(counts[1] for counts in histogram.values())\n    negatives = sum(counts[0] for counts in histogram.values())\n    if not positives or not negatives:\n        return {"roc_auc": None, "pr_auc": None}\n    grown_pos = grown_neg = 0\n    last_tpr = last_fpr = 0.0\n    roc = pr = 0.0\n    for score_bin in sorted(histogram, reverse=True):\n        counts = histogram[score_bin]\n        grown_neg += counts[0]\n        grown_pos += counts[1]\n        tpr, fpr = grown_pos / positives, grown_neg / negatives\n        precision = grown_pos / (grown_pos + grown_neg)\n        roc += (fpr - last_fpr) * (tpr + last_tpr) / 2\n        pr += (tpr - last_tpr) * precision\n        last_tpr, last_fpr = tpr, fpr\n    return {"roc_auc": roc, "pr_auc": pr}\n\n\ndef estimator_score(model, frame):\n    return model.transform(frame).withColumn("score", vector_to_array("probability")[1])\n\n\ndef evaluate(scored, config):\n    """One Spark action scores rows and gathers only small aggregated histograms."""\n    started = time.monotonic()\n    value = F.col("score")\n    bins = config["score_bins"]\n    invalid = value.isNull() | F.isnan(value) | (value < 0.0) | (value > 1.0)\n    rows = (scored.select("binary_label", "attack_family", "original_label",\n                         F.when(invalid, -1).otherwise(F.least(F.floor(value * bins), F.lit(bins - 1))).alias("score_bin"),\n                         F.when(value >= config["threshold"], 1).otherwise(0).alias("pred"))\n            .groupBy("binary_label", "attack_family", "original_label", "score_bin", "pred").count()\n            .limit(65001).collect())\n    seconds = round(time.monotonic() - started, 3)\n    if len(rows) >= 65001 or any(row.score_bin < 0 or row.binary_label not in (0, 1) for row in rows):\n        raise ValueError("Invalid model scores or aggregate exceeded the compact limit")\n    histogram, conf, groups = {}, Counter(), {}\n    for row in rows:\n        histogram.setdefault(row.score_bin, Counter())[row.binary_label] += row["count"]\n        conf[row.binary_label, row.pred] += row["count"]\n        groups.setdefault((row.binary_label, row.attack_family, row.original_label), Counter())[row.pred] += row["count"]\n    metrics = confusion_metrics(conf[1, 1], conf[0, 0], conf[0, 1], conf[1, 0])\n    return metrics | binned_auc(histogram) | {"prediction_seconds": seconds}, groups\n\n\ndef family_recall(groups, train_families, train_labels, families, original_labels, minimum):\n    observed = {}\n    for (binary, family, label), pred in groups.items():\n        observed.setdefault(("family", family, binary), Counter()).update(pred)\n        observed.setdefault(("original_label", label, binary), Counter()).update(pred)\n    rows = []\n    for level, catalog, train in (("family", families, train_families), ("original_label", original_labels, train_labels)):\n        for name, binary in sorted(catalog.items()):\n            counts = observed.get((level, name, binary), Counter())\n            support = counts[0] + counts[1]\n            rows.append({"level": level, "category": name, "binary_label": binary,\n                         "evaluated_rows": support, "tp": counts[1] if binary else None,\n                         "fn": counts[0] if binary else None, "fp": counts[1] if not binary else None,\n                         "recall": rate(counts[1], support) if binary else None,\n                         "benign_false_positive_rate": rate(counts[1], support) if not binary else None,\n                         "seen_in_training": train.get(name, 0) > 0, "training_rows": train.get(name, 0),\n                         "support_status": "absent" if not support else "low" if support < minimum else "reported"})\n    return rows\n', 'src/ml/model_explanations.py': '"""Map fitted coefficients or split importances to the exact vector order."""\n\n\ndef explain(estimator, model_name, active):\n    if model_name == "logistic_regression":\n        method, values = "scaled_lr_coefficient", model_values(estimator.coefficients)\n    else:\n        method, values = "impurity_split_importance", model_values(estimator.featureImportances)\n    if len(values) != len(active):\n        raise ValueError("Fitted vector and registry dimensions disagree")\n    return [{"feature": feature, "method": method, "value": value}\n            for feature, value in sorted(zip(active, values), key=lambda item: (-abs(item[1]), item[0]))]\n\n\ndef model_values(vector):\n    return [float(value) for value in vector.toArray()]\n', 'src/ml/train_supervised.py': '"""Execute one predeclared Phase 4 protocol over the verified Kaggle input."""\n\nfrom collections import Counter\nimport json\nfrom hashlib import sha256\nfrom pathlib import Path\nimport time\n\nfrom pyspark import StorageLevel\nfrom pyspark.ml import PipelineModel\nfrom pyspark.ml.classification import GBTClassifier, LogisticRegression, RandomForestClassifier\nfrom pyspark.ml.functions import vector_to_array\nfrom pyspark.sql import functions as F\n\nfrom src.ingest.dataset_audit import atomic_json, spark_session\nfrom src.ingest.parquet_pipeline import driver_peak_bytes, parquet_footers, resource_limits\nfrom src.ml.baselines import baseline_score, fit_baselines\nfrom src.ml.evaluate_supervised import estimator_score, evaluate, family_recall\nfrom src.ml.feature_registry import predictor_input, registry\nfrom src.ml.model_explanations import explain\nfrom src.ml.preprocessing import fit_preprocessing, verify_vectors\nfrom src.ml.split_protocols import (\n    SPLITS, assign_population, crossing_groups, group_sample, group_summary, label_conflicts, mark_retention,\n)\nfrom src.sql.traffic_analytics import compact, label_grouping, locate_producer, require, write_rows\n\n\nAUDIT_KEYS = ("predictor_key", "record_fingerprint", "split_feature_key")\n\n\ndef read_json(path):\n    return json.loads(path.read_text())\n\n\ndef verified_input(spark, receipt, manifest, clean_policy):\n    root = receipt.parent.parent\n    evidence = read_json(receipt)\n    require(evidence["status"] == "validated" and evidence["dataset"] == manifest["dataset"], "Producer dataset differs")\n    require(evidence["policy_version"] == clean_policy["version"] == manifest["policy_version"], "Cleaning policy differs")\n    path = root / manifest["policy_version"] / "feature_ready"\n    footer = parquet_footers(path)\n    expected = manifest["layers"]["feature_ready"]\n    require(all(footer[key] == expected[key] for key in footer), "Saved feature-ready Parquet differs")\n    days = sorted(x.name.split("=", 1)[1] for x in path.glob("source_day=*") if x.is_dir())\n    require(len(days) == expected["partitions"] == 10, "Verified input must contain ten days")\n    frame = spark.read.parquet(str(path))\n    extras = {"source_day", "binary_label", "original_label", "normalized_label", "attack_family",\n              "temporal_eligibility", "record_fingerprint", "split_feature_key"}\n    require(set(frame.columns) == set(manifest["feature_columns"]) | extras, "Input schema differs")\n    types = dict(frame.dtypes)\n    require(all(types.get(name) == "double" for name in manifest["feature_columns"]), "Input feature types differ")\n    require(types["binary_label"] == "tinyint" and types["source_day"] == "string", "Label or day type differs")\n    label_totals, day_totals = Counter(), Counter()\n    ineligible = 0\n    for row in compact(label_grouping(frame), 500):\n        mapping = clean_policy["label_mapping"].get(row["original_label"])\n        require(mapping and (mapping["binary"], mapping["family"], mapping["normalized"]) ==\n                (row["binary_label"], row["attack_family"], row["normalized_label"]), "Label integrity check failed")\n        require(row["temporal_eligibility"] in (True, False), "Temporal flag is unavailable")\n        label_totals[row["binary_label"]] += row["count"]\n        day_totals[row["source_day"]] += row["count"]\n        ineligible += 0 if row["temporal_eligibility"] else row["count"]\n    require(dict(label_totals) == {0: 13484708, 1: 2748235} and sum(label_totals.values()) == 16232943,\n            "Input row or label totals differ")\n    require(ineligible == 14, "Timestamp eligibility count differs")\n    require(dict(day_totals) == {row["source_day"]: row["counts"]["canonical_rows"] for row in evidence["completed_files"]},\n            "Capture-day totals differ")\n    return frame, {"path": str(path), "days": days, "totals": dict(label_totals), "rows": footer["records"],\n                   "temporally_flagged": ineligible, "policy_version": manifest["policy_version"],\n                   "receipt_sha256": sha256(receipt.read_bytes()).hexdigest(), "producer": manifest["producer"]}\n\n\ndef split_rows(frame, config):\n    return compact(frame.groupBy("split_rank", "retained", "sampled", "source_day", "binary_label",\n                                 "attack_family", "original_label").count()\n                   .orderBy("split_rank", "retained", "sampled", "source_day", "original_label"), 1500)\n\n\ndef supports(rows, rank, sample_only=False):\n    labels, families, originals = Counter(), Counter(), Counter()\n    for row in rows:\n        if row["split_rank"] == rank and row["retained"] and (not sample_only or row["sampled"]):\n            labels[row["binary_label"]] += row["count"]\n            families[row["attack_family"]] += row["count"]\n            originals[row["original_label"]] += row["count"]\n    return {"labels": dict(labels), "families": dict(families), "original_labels": dict(originals), "rows": sum(labels.values())}\n\n\ndef create_estimator(name, params, config):\n    common = {"labelCol": "binary_label"}\n    if name == "logistic_regression":\n        return LogisticRegression(**common, **params, featuresCol="scaled_features", standardization=False, tol=1e-5)\n    trees = {"featuresCol": "features", "seed": config["seed"], "maxBins": 24, "maxMemoryInMB": 192, **common}\n    if name == "random_forest":\n        return RandomForestClassifier(**trees, **params, featureSubsetStrategy="sqrt", subsamplingRate=1.0)\n    if name == "gradient_boosted_trees":\n        return GBTClassifier(**trees, **params, subsamplingRate=1.0, minInstancesPerNode=30)\n    raise ValueError(f"Unknown estimator {name}")\n\n\ndef select_trial(trials):\n    candidates = [row for row in trials if row["pr_auc"] is not None]\n    if not candidates:\n        raise ValueError("No trial has a defined validation PR-AUC")\n    return max(candidates, key=lambda row: (row["pr_auc"], -row["trial"]))\n\n\ndef save_and_reload(estimator, preprocessor, model_name, dataset, directory):\n    pipeline = PipelineModel(stages=[*preprocessor.stages, estimator])\n    directory.parent.mkdir(parents=True, exist_ok=True)\n    pipeline.write().save(str(directory))\n    reopened = PipelineModel.load(str(directory))\n    require(len(reopened.stages) == len(pipeline.stages), "Reloaded pipeline has a different stage count")\n    def scores(model):\n        return sorted(row[0] for row in model.transform(dataset).select(vector_to_array("probability")[1]).collect())\n\n    probe = dataset.orderBy("predictor_key", "binary_label").limit(24).persist()\n    try:\n        before, after = scores(pipeline), scores(reopened)\n        require(bool(before) and before == after, "Reloaded probabilities differ")\n    finally:\n        probe.unpersist()\n    files = sorted(x for x in directory.rglob("*") if x.is_file() and not x.name.endswith(".crc"))\n    return {"path": str(directory), "model": model_name, "reload_probe_rows": len(after),\n            "model_bytes": sum(x.stat().st_size for x in files), "model_files": len(files),\n            "sha256": {x.relative_to(directory).as_posix(): sha256(x.read_bytes()).hexdigest() for x in files}}\n\n\ndef run(project, input_root, output, scratch, protocol):\n    output.mkdir(parents=True, exist_ok=True)\n    scratch.mkdir(parents=True, exist_ok=True)\n    config = read_json(project / "configs/ml_experiments.yaml")\n    manifest = read_json(project / "manifests/parquet_layers.json")\n    policy = read_json(project / "configs/cleaning.yaml")\n    atomic_json(output / "phase4_config.json", config)\n    spark = spark_session(config, scratch)\n    spark.conf.set("spark.sql.sources.partitionColumnTypeInference.enabled", "false")\n    spark.sparkContext.setLogLevel("WARN")\n    start = time.monotonic()\n    cached = []\n\n    def disk(frame):\n        result = frame.persist(StorageLevel.DISK_ONLY)\n        cached.append(result)\n        return result\n\n    metrics, family_rows, performance = [], [], []\n    summary = {"protocol": protocol, "version": config["version"], "status": "running",\n               "design": config["protocols"][protocol], "resources": resource_limits(spark, config, output),\n               "models": [], "trial_selections": {}}\n    atomic_json(output / "phase4_summary.json", summary)\n\n    def checkpoint():\n        for name, rows in (("metrics", metrics), ("family_recall", family_rows), ("training_performance", performance)):\n            if rows:\n                write_rows(output / f"phase4_{name}.csv", rows)\n        atomic_json(output / "phase4_summary.json", summary)\n\n    try:\n        receipt = locate_producer(input_root, manifest["independent_mount"]["summary_sha256"])\n        frame, summary["input"] = verified_input(spark, receipt, manifest, policy)\n        names = registry(config, manifest["feature_columns"])\n        summary["registered_predictors"] = names\n        print(f"PHASE4 {protocol} verified {summary[\'input\'][\'rows\']} input rows", flush=True)\n        assigned = disk(assign_population(predictor_input(frame, names, config["version"]), protocol, config, summary["input"]["days"]))\n        require(assigned.count() == summary["input"]["rows"], "Assignment lost input rows")\n        groups = disk(crossing_groups(assigned, "predictor_key"))\n        before = group_summary(groups)\n        if protocol == "A":\n            require(before["groups"] == 0, "Grouped random assignment crossed an equivalence boundary")\n        retained = disk(group_sample(mark_retention(assigned, groups, before["groups"]), config))\n        counts = split_rows(retained, config)\n        require(sum(row["count"] for row in counts) == summary["input"]["rows"], "Split partition rows do not reconcile")\n        write_rows(output / "phase4_split_diagnostics.csv", counts)\n        active_source = retained.where("retained")\n        after = {key: group_summary(crossing_groups(active_source, key)) for key in AUDIT_KEYS}\n        require(all(count["groups"] == 0 for count in after.values()), "Retained partition keys overlap")\n        summary["overlap"] = {"predictor_before": before, "after": after,\n                              "purged_rows": sum(row["count"] for row in counts if not row["retained"])}\n        summary["retained_predictor_conflicts"] = group_summary(label_conflicts(active_source))\n        summary["split_support"] = {name: supports(counts, rank) for rank, name in enumerate(SPLITS)}\n        sample_support = supports(counts, 0, True)\n        summary["train_sample_support"] = sample_support\n        require(all(summary["split_support"][name]["rows"] > 0 for name in SPLITS), "A required partition is empty")\n        require(set(sample_support["labels"]) == {0, 1}, "Training sample lacks a binary class")\n        print(f"PHASE4 {protocol} retained split sizes {[(n, summary[\'split_support\'][n][\'rows\']) for n in SPLITS]}", flush=True)\n        assigned.unpersist(blocking=True)\n        groups.unpersist(blocking=True)\n        training = active_source.where("split_rank = 0")\n        sample = disk(training.where("sampled"))\n        calibration = fit_baselines(training, summary["split_support"]["train"]["labels"].get(0, 0),\n                                    summary["split_support"]["train"]["labels"].get(1, 0))\n        summary["baselines"] = calibration\n        eval_sources = {name: active_source.where(F.col("split_rank") == rank) for rank, name in enumerate(SPLITS) if rank}\n        catalogs = {name: values["binary"] for name, values in policy["label_mapping"].items()}\n        families = {values["family"]: values["binary"] for values in policy["label_mapping"].values()}\n\n        def record(scored, model, cohort, split, training_seconds):\n            result, groups = evaluate(scored, config)\n            require(result["rows"] == summary["split_support"][split]["rows"], f"{model} prediction population differs")\n            support = summary["train_sample_support"] if cohort == "group_sample" else summary["split_support"]["train"]\n            stamp = {"protocol": protocol, "model": model, "training_cohort": cohort, "split": split}\n            metrics.append(stamp | {"training_rows": support["rows"], "training_seconds": training_seconds} | result)\n            family_rows.extend(stamp | row for row in family_recall(\n                groups, support["families"], support["original_labels"], families, catalogs, config["rare_family_minimum"]))\n            checkpoint()\n            print(f"PHASE4 {protocol} {model} {cohort} {split}: {result[\'rows\']} scored", flush=True)\n            return result\n\n        # Baseline validation scores are established before fitting complex estimators.\n        for model, params in calibration.items():\n            record(baseline_score(eval_sources["validation"], model, params), model, "full_train_baseline", "validation", 0.0)\n\n        prep_started = time.monotonic()\n        common, scaled, summary["preprocessing"] = fit_preprocessing(sample, names)\n        summary["preprocess_seconds"] = round(time.monotonic() - prep_started, 3)\n        columns = ["split_rank", "sampled", "binary_label", "attack_family", "original_label", "features", "scaled_features"]\n        design = disk(scaled.transform(active_source).select(*columns))\n        require(design.count() == sum(v["rows"] for v in summary["split_support"].values()), "Transform lost retained rows")\n        verify_vectors(design, len(summary["preprocessing"]["active"]))\n        vector_keys = design.select("split_rank", F.unhex(F.sha2(F.to_json(vector_to_array("features")), 256)).alias("vector_key"))\n        summary["post_imputation_overlap"] = group_summary(crossing_groups(vector_keys, "vector_key"))\n        checkpoint()\n        sampled_vector = disk(design.where("split_rank = 0 AND sampled"))\n        sampled_vector.count()\n        vectors = {name: design.where(F.col("split_rank") == rank) for rank, name in enumerate(SPLITS)}\n        winners = []\n        for name, choices in config["trials"].items():\n            runs, fitted = [], {}\n            for i, params in enumerate(choices, 1):\n                fit_start = time.monotonic()\n                model = create_estimator(name, params, config).fit(sampled_vector)\n                fit_s = round(time.monotonic() - fit_start, 3)\n                measured, _ = evaluate(estimator_score(model, vectors["validation"]), config)\n                require(measured["rows"] == summary["split_support"]["validation"]["rows"], "Trial validation rows differ")\n                row = {"protocol": protocol, "model": name, "training_cohort": "group_sample", "trial": i,\n                       "parameters": params, "training_rows": sample_support["rows"], "training_seconds": fit_s,\n                       "validation_inference_seconds": measured["prediction_seconds"], "pr_auc": measured["pr_auc"],\n                       "roc_auc": measured["roc_auc"], "validation_f1": measured["f1"], "selected": False}\n                runs.append(row)\n                performance.append(row)\n                fitted[i] = model\n                checkpoint()\n                print(f"PHASE4 {protocol} {name} trial {i}: fit {fit_s:.3f}s, validation PR-AUC {measured[\'pr_auc\']}", flush=True)\n            selected = select_trial(runs)\n            selected["selected"] = True\n            summary["trial_selections"][name] = {"trial": selected["trial"], "parameters": selected["parameters"]}\n            winners.append((name, "group_sample", fitted[selected["trial"]], selected["training_seconds"]))\n\n        # Additional full-training LR uses fixed sample-selected settings and the same train-only transform.\n        name = "logistic_regression"\n        params = summary["trial_selections"][name]["parameters"]\n        fit_start = time.monotonic()\n        full_model = create_estimator(name, params, config).fit(vectors["train"])\n        full_time = round(time.monotonic() - fit_start, 3)\n        full_result, _ = evaluate(estimator_score(full_model, vectors["validation"]), config)\n        performance.append({"protocol": protocol, "model": name, "training_cohort": "full_train_refit", "trial": 1,\n                            "parameters": params, "training_rows": summary["split_support"]["train"]["rows"],\n                            "training_seconds": full_time, "validation_inference_seconds": full_result["prediction_seconds"],\n                            "pr_auc": full_result["pr_auc"], "roc_auc": full_result["roc_auc"],\n                            "validation_f1": full_result["f1"], "selected": True})\n        winners.append((name, "full_train_refit", full_model, full_time))\n        summary["selection_frozen"] = True\n        checkpoint()\n        print(f"PHASE4 {protocol} selections frozen before testing", flush=True)\n\n        for baseline, params in calibration.items():\n            record(baseline_score(eval_sources["test"], baseline, params), baseline, "full_train_baseline", "test", 0.0)\n        for name, cohort, model, fit_s in winners:\n            for split in ("validation", "test"):\n                record(estimator_score(model, vectors[split]), name, cohort, split, fit_s)\n            stored = save_and_reload(model, scaled if name == "logistic_regression" else common, name,\n                                     sample, output / "models" / f"{name}_{cohort}")\n            stored.update(training_cohort=cohort, training_seconds=fit_s,\n                          parameters=summary["trial_selections"][name]["parameters"],\n                          explanation=explain(model, name, summary["preprocessing"]["active"]))\n            summary["models"].append(stored)\n            checkpoint()\n        summary.update(status="validated", elapsed_seconds=round(time.monotonic() - start, 3))\n        summary["resources"]["driver_peak_rss_bytes"] = driver_peak_bytes(spark)\n        checkpoint()\n        print(f"PHASE4 {protocol} complete: {len(metrics)} evaluation rows, {len(summary[\'models\'])} reloaded pipelines "\n              f"in {summary[\'elapsed_seconds\']}s", flush=True)\n    except Exception as exc:\n        summary.update(status="partial", error=f"{type(exc).__name__}: {str(exc)[:500]}",\n                       elapsed_seconds=round(time.monotonic() - start, 3))\n        checkpoint()\n        raise\n    finally:\n        for item in reversed(cached):\n            item.unpersist()\n        spark.stop()\n', 'configs/cleaning.yaml': '{\n  "version": "phase2-v1",\n  "dataset_version": 1,\n  "headers": {\n    "common_80": [\n      "Dst Port",\n      "Protocol",\n      "Timestamp",\n      "Flow Duration",\n      "Tot Fwd Pkts",\n      "Tot Bwd Pkts",\n      "TotLen Fwd Pkts",\n      "TotLen Bwd Pkts",\n      "Fwd Pkt Len Max",\n      "Fwd Pkt Len Min",\n      "Fwd Pkt Len Mean",\n      "Fwd Pkt Len Std",\n      "Bwd Pkt Len Max",\n      "Bwd Pkt Len Min",\n      "Bwd Pkt Len Mean",\n      "Bwd Pkt Len Std",\n      "Flow Byts/s",\n      "Flow Pkts/s",\n      "Flow IAT Mean",\n      "Flow IAT Std",\n      "Flow IAT Max",\n      "Flow IAT Min",\n      "Fwd IAT Tot",\n      "Fwd IAT Mean",\n      "Fwd IAT Std",\n      "Fwd IAT Max",\n      "Fwd IAT Min",\n      "Bwd IAT Tot",\n      "Bwd IAT Mean",\n      "Bwd IAT Std",\n      "Bwd IAT Max",\n      "Bwd IAT Min",\n      "Fwd PSH Flags",\n      "Bwd PSH Flags",\n      "Fwd URG Flags",\n      "Bwd URG Flags",\n      "Fwd Header Len",\n      "Bwd Header Len",\n      "Fwd Pkts/s",\n      "Bwd Pkts/s",\n      "Pkt Len Min",\n      "Pkt Len Max",\n      "Pkt Len Mean",\n      "Pkt Len Std",\n      "Pkt Len Var",\n      "FIN Flag Cnt",\n      "SYN Flag Cnt",\n      "RST Flag Cnt",\n      "PSH Flag Cnt",\n      "ACK Flag Cnt",\n      "URG Flag Cnt",\n      "CWE Flag Count",\n      "ECE Flag Cnt",\n      "Down/Up Ratio",\n      "Pkt Size Avg",\n      "Fwd Seg Size Avg",\n      "Bwd Seg Size Avg",\n      "Fwd Byts/b Avg",\n      "Fwd Pkts/b Avg",\n      "Fwd Blk Rate Avg",\n      "Bwd Byts/b Avg",\n      "Bwd Pkts/b Avg",\n      "Bwd Blk Rate Avg",\n      "Subflow Fwd Pkts",\n      "Subflow Fwd Byts",\n      "Subflow Bwd Pkts",\n      "Subflow Bwd Byts",\n      "Init Fwd Win Byts",\n      "Init Bwd Win Byts",\n      "Fwd Act Data Pkts",\n      "Fwd Seg Size Min",\n      "Active Mean",\n      "Active Std",\n      "Active Max",\n      "Active Min",\n      "Idle Mean",\n      "Idle Std",\n      "Idle Max",\n      "Idle Min",\n      "Label"\n    ],\n    "extended_84": [\n      "Flow ID",\n      "Src IP",\n      "Src Port",\n      "Dst IP",\n      "Dst Port",\n      "Protocol",\n      "Timestamp",\n      "Flow Duration",\n      "Tot Fwd Pkts",\n      "Tot Bwd Pkts",\n      "TotLen Fwd Pkts",\n      "TotLen Bwd Pkts",\n      "Fwd Pkt Len Max",\n      "Fwd Pkt Len Min",\n      "Fwd Pkt Len Mean",\n      "Fwd Pkt Len Std",\n      "Bwd Pkt Len Max",\n      "Bwd Pkt Len Min",\n      "Bwd Pkt Len Mean",\n      "Bwd Pkt Len Std",\n      "Flow Byts/s",\n      "Flow Pkts/s",\n      "Flow IAT Mean",\n      "Flow IAT Std",\n      "Flow IAT Max",\n      "Flow IAT Min",\n      "Fwd IAT Tot",\n      "Fwd IAT Mean",\n      "Fwd IAT Std",\n      "Fwd IAT Max",\n      "Fwd IAT Min",\n      "Bwd IAT Tot",\n      "Bwd IAT Mean",\n      "Bwd IAT Std",\n      "Bwd IAT Max",\n      "Bwd IAT Min",\n      "Fwd PSH Flags",\n      "Bwd PSH Flags",\n      "Fwd URG Flags",\n      "Bwd URG Flags",\n      "Fwd Header Len",\n      "Bwd Header Len",\n      "Fwd Pkts/s",\n      "Bwd Pkts/s",\n      "Pkt Len Min",\n      "Pkt Len Max",\n      "Pkt Len Mean",\n      "Pkt Len Std",\n      "Pkt Len Var",\n      "FIN Flag Cnt",\n      "SYN Flag Cnt",\n      "RST Flag Cnt",\n      "PSH Flag Cnt",\n      "ACK Flag Cnt",\n      "URG Flag Cnt",\n      "CWE Flag Count",\n      "ECE Flag Cnt",\n      "Down/Up Ratio",\n      "Pkt Size Avg",\n      "Fwd Seg Size Avg",\n      "Bwd Seg Size Avg",\n      "Fwd Byts/b Avg",\n      "Fwd Pkts/b Avg",\n      "Fwd Blk Rate Avg",\n      "Bwd Byts/b Avg",\n      "Bwd Pkts/b Avg",\n      "Bwd Blk Rate Avg",\n      "Subflow Fwd Pkts",\n      "Subflow Fwd Byts",\n      "Subflow Bwd Pkts",\n      "Subflow Bwd Byts",\n      "Init Fwd Win Byts",\n      "Init Bwd Win Byts",\n      "Fwd Act Data Pkts",\n      "Fwd Seg Size Min",\n      "Active Mean",\n      "Active Std",\n      "Active Max",\n      "Active Min",\n      "Idle Mean",\n      "Idle Std",\n      "Idle Max",\n      "Idle Min",\n      "Label"\n    ]\n  },\n  "extended_day": "2018-02-20",\n  "label_mapping": {\n    "Benign": {\n      "normalized": "benign",\n      "binary": 0,\n      "family": "Benign"\n    },\n    "Bot": {\n      "normalized": "bot",\n      "binary": 1,\n      "family": "Bot"\n    },\n    "Brute Force -Web": {\n      "normalized": "brute force -web",\n      "binary": 1,\n      "family": "Web attack"\n    },\n    "Brute Force -XSS": {\n      "normalized": "brute force -xss",\n      "binary": 1,\n      "family": "Web attack"\n    },\n    "DDOS attack-HOIC": {\n      "normalized": "ddos attack-hoic",\n      "binary": 1,\n      "family": "DDoS"\n    },\n    "DDOS attack-LOIC-UDP": {\n      "normalized": "ddos attack-loic-udp",\n      "binary": 1,\n      "family": "DDoS"\n    },\n    "DDoS attacks-LOIC-HTTP": {\n      "normalized": "ddos attacks-loic-http",\n      "binary": 1,\n      "family": "DDoS"\n    },\n    "DoS attacks-GoldenEye": {\n      "normalized": "dos attacks-goldeneye",\n      "binary": 1,\n      "family": "DoS"\n    },\n    "DoS attacks-Hulk": {\n      "normalized": "dos attacks-hulk",\n      "binary": 1,\n      "family": "DoS"\n    },\n    "DoS attacks-SlowHTTPTest": {\n      "normalized": "dos attacks-slowhttptest",\n      "binary": 1,\n      "family": "DoS"\n    },\n    "DoS attacks-Slowloris": {\n      "normalized": "dos attacks-slowloris",\n      "binary": 1,\n      "family": "DoS"\n    },\n    "FTP-BruteForce": {\n      "normalized": "ftp-bruteforce",\n      "binary": 1,\n      "family": "Credential brute force"\n    },\n    "Infilteration": {\n      "normalized": "infilteration",\n      "binary": 1,\n      "family": "Infiltration"\n    },\n    "SQL Injection": {\n      "normalized": "sql injection",\n      "binary": 1,\n      "family": "Web attack"\n    },\n    "SSH-Bruteforce": {\n      "normalized": "ssh-bruteforce",\n      "binary": 1,\n      "family": "Credential brute force"\n    }\n  },\n  "sentinel_values": {\n    "init_fwd_win_byts": [\n      -1\n    ],\n    "init_bwd_win_byts": [\n      -1\n    ]\n  },\n  "numeric_actions": {\n    "nan": "nullify",\n    "positive_infinity": "nullify",\n    "negative_infinity": "nullify",\n    "invalid_negative": "nullify",\n    "sentinel": "nullify",\n    "cast_failure": "typed_null",\n    "source_null": "retain_null"\n  },\n  "excluded_shared_features": [\n    "dst_port"\n  ],\n  "duplicate_policy": "diagnostic",\n  "expected_full": {\n    "benign": 13484708,\n    "attack": 2748235,\n    "nan": 59721,\n    "infinity": 131799,\n    "invalid_negative": 106,\n    "timestamp_outlier": 14,\n    "duplicate_excess": 410707\n  },\n  "spark": {\n    "master": "local[4]",\n    "driver_memory": "8g",\n    "input_partition_bytes": 67108864,\n    "shuffle_partitions": 48\n  },\n  "target_source_bytes_per_part": 134217728\n}\n', 'configs/ml_experiments.yaml': '{\n  "version": "phase4-v1",\n  "seed": 23013,\n  "group_sample_permyriad": 625,\n  "score_bins": 2048,\n  "threshold": 0.5,\n  "rare_family_minimum": 100,\n  "numerical_features": [\n    "flow_duration", "tot_fwd_pkts", "tot_bwd_pkts", "totlen_fwd_pkts", "totlen_bwd_pkts",\n    "pkt_len_mean", "pkt_len_std", "flow_iat_mean", "flow_iat_std", "fwd_iat_mean",\n    "bwd_iat_mean", "flow_pkts_s", "flow_byts_s", "down_up_ratio",\n    "syn_flag_cnt", "ack_flag_cnt", "rst_flag_cnt", "fin_flag_cnt"\n  ],\n  "high_missingness_excluded": ["init_fwd_win_byts", "init_bwd_win_byts"],\n  "optional_categorical": ["protocol"],\n  "random_permyriad": {"train": 7000, "validation": 1500, "test": 1500},\n  "protocols": {\n    "A": {"name": "predictor-group random"},\n    "B": {\n      "name": "chronological capture-day holdout",\n      "train": ["2018-02-14", "2018-02-15", "2018-02-16", "2018-02-20", "2018-02-21", "2018-02-22"],\n      "validation": ["2018-02-23", "2018-02-28"],\n      "test": ["2018-03-01", "2018-03-02"]\n    },\n    "C": {\n      "name": "web-attack capture holdout",\n      "train": ["2018-02-14", "2018-02-15", "2018-02-16", "2018-02-20", "2018-02-21", "2018-03-01", "2018-03-02"],\n      "validation": ["2018-02-28"],\n      "test": ["2018-02-22", "2018-02-23"]\n    }\n  },\n  "trials": {\n    "logistic_regression": [\n      {"maxIter": 12, "regParam": 0.005, "elasticNetParam": 0.0},\n      {"maxIter": 12, "regParam": 0.05, "elasticNetParam": 0.0}\n    ],\n    "random_forest": [\n      {"numTrees": 12, "maxDepth": 5, "minInstancesPerNode": 30},\n      {"numTrees": 20, "maxDepth": 7, "minInstancesPerNode": 30}\n    ],\n    "gradient_boosted_trees": [\n      {"maxIter": 6, "maxDepth": 3, "stepSize": 0.1},\n      {"maxIter": 10, "maxDepth": 4, "stepSize": 0.15}\n    ]\n  },\n  "spark": {"master": "local[4]", "driver_memory": "8g", "shuffle_partitions": 48, "input_partition_bytes": 67108864}\n}\n', 'manifests/parquet_layers.json': '{\n  "status": "validated",\n  "dataset": {\n    "owner": "saisathwikregula",\n    "slug": "mmds-dataset-23013",\n    "version": 1\n  },\n  "policy_version": "phase2-v1",\n  "layers": {\n    "raw_aligned": {\n      "path": "/kaggle/working/mmds_phase2/phase2-v1/raw_aligned",\n      "partition_column": "source_day",\n      "partitions": 10,\n      "format": "parquet",\n      "compression": "SNAPPY",\n      "records": 16232943,\n      "bytes": 2712371594,\n      "files": 55,\n      "write_seconds": 240.044\n    },\n    "clean": {\n      "path": "/kaggle/working/mmds_phase2/phase2-v1/clean",\n      "partition_column": "source_day",\n      "partitions": 10,\n      "format": "parquet",\n      "compression": "SNAPPY",\n      "records": 16232943,\n      "bytes": 2708993526,\n      "files": 55,\n      "write_seconds": 297.806\n    },\n    "feature_ready": {\n      "path": "/kaggle/working/mmds_phase2/phase2-v1/feature_ready",\n      "partition_column": "source_day",\n      "partitions": 10,\n      "format": "parquet",\n      "compression": "SNAPPY",\n      "records": 16232943,\n      "bytes": 3003003600,\n      "files": 53,\n      "write_seconds": 314.859\n    }\n  },\n  "duplicate_policy": "diagnostic",\n  "duplicate_diagnostics": {\n    "records_removed_primary": 0,\n    "exact": {\n      "groups": 107670,\n      "excess": 410707\n    },\n    "cleaned_feature_conflicts": {\n      "groups": 21763,\n      "participating_rows": 710014\n    },\n    "diagnostics_path": "/kaggle/working/mmds_phase2/phase2-v1/diagnostics",\n    "seconds": 72.76\n  },\n  "producer": {\n    "url": "https://www.kaggle.com/code/ramcharan2345/mmds-phase-2-spark-data-engineering",\n    "kernel_version": 1,\n    "status": "COMPLETE"\n  },\n  "resources": {\n    "spark_version": "4.3.0",\n    "master": "local[4]",\n    "driver_memory": "8g",\n    "input_partition_bytes": 67108864,\n    "shuffle_partitions": 48,\n    "default_parallelism": 4,\n    "cpu_affinity": 4,\n    "scratch_output_free_bytes_start": 20940369920,\n    "memory.max": "32212254720",\n    "cpu.max": "400000 100000",\n    "driver_peak_rss_bytes": 5784305664\n  },\n  "totals": {\n    "physical_lines": 16233012,\n    "parsed": 16232943,\n    "headers": 69,\n    "blank": 0,\n    "bad_width_or_quotes": 0,\n    "spark_corrupt": 0,\n    "repeated_headers": 59,\n    "canonical_rows": 16232943,\n    "benign": 13484708,\n    "attack": 2748235,\n    "unknown_labels": 0,\n    "timestamp_outlier": 14,\n    "timestamp_unparseable": 0,\n    "timestamp_missing": 0,\n    "temporal_eligible": 16232929,\n    "quality_affected_rows": 8257114,\n    "quality_affected_cells": 12879783,\n    "structural_missing_rows": 8284195,\n    "structural_missing_cells": 33136780,\n    "nan_rows": 59721,\n    "nan_cells": 59721,\n    "positive_infinity_rows": 95760,\n    "positive_infinity_cells": 131799,\n    "negative_infinity_rows": 0,\n    "negative_infinity_cells": 0,\n    "invalid_negative_rows": 15,\n    "invalid_negative_cells": 106,\n    "sentinel_rows": 8255549,\n    "sentinel_cells": 12688157,\n    "cast_failure_rows": 0,\n    "cast_failure_cells": 0,\n    "source_null_rows": 0,\n    "source_null_cells": 0\n  },\n  "independent_mount": {\n    "layers": {\n      "raw_aligned": {\n        "records": 16232943,\n        "bytes": 2712371594,\n        "benign": 13484708,\n        "attack": 2748235,\n        "files": 55,\n        "partitions": 10\n      },\n      "clean": {\n        "records": 16232943,\n        "bytes": 2708993526,\n        "benign": 13484708,\n        "attack": 2748235,\n        "files": 55,\n        "partitions": 10\n      },\n      "feature_ready": {\n        "records": 16232943,\n        "bytes": 3003003600,\n        "benign": 13484708,\n        "attack": 2748235,\n        "files": 53,\n        "partitions": 10\n      }\n    },\n    "clean_cells_verified_null": 12879783,\n    "clean_quality_affected_rows": 8257114,\n    "clean_quality_reasons": {\n      "sentinel": 12688157,\n      "nan": 59721,\n      "positive_infinity": 131799,\n      "invalid_negative": 106\n    },\n    "clean_timestamp_outliers": 14,\n    "summary_sha256": "1fa00cd2b9c853ee33eaacee8ca7632dbb41f739573326280d6ffbfb2ccff25a",\n    "producer_mount": "/kaggle/input/notebooks/ramcharan2345/mmds-phase-2-spark-data-engineering/mmds_phase2",\n    "elapsed_seconds": 101.019,\n    "url": "https://www.kaggle.com/code/ramcharan2345/mmds-phase-2-parquet-reuse-check",\n    "kernel_version": 1,\n    "status": "COMPLETE"\n  },\n  "feature_columns": [\n    "protocol",\n    "flow_duration",\n    "tot_fwd_pkts",\n    "tot_bwd_pkts",\n    "totlen_fwd_pkts",\n    "totlen_bwd_pkts",\n    "fwd_pkt_len_max",\n    "fwd_pkt_len_min",\n    "fwd_pkt_len_mean",\n    "fwd_pkt_len_std",\n    "bwd_pkt_len_max",\n    "bwd_pkt_len_min",\n    "bwd_pkt_len_mean",\n    "bwd_pkt_len_std",\n    "flow_byts_s",\n    "flow_pkts_s",\n    "flow_iat_mean",\n    "flow_iat_std",\n    "flow_iat_max",\n    "flow_iat_min",\n    "fwd_iat_tot",\n    "fwd_iat_mean",\n    "fwd_iat_std",\n    "fwd_iat_max",\n    "fwd_iat_min",\n    "bwd_iat_tot",\n    "bwd_iat_mean",\n    "bwd_iat_std",\n    "bwd_iat_max",\n    "bwd_iat_min",\n    "fwd_psh_flags",\n    "bwd_psh_flags",\n    "fwd_urg_flags",\n    "bwd_urg_flags",\n    "fwd_header_len",\n    "bwd_header_len",\n    "fwd_pkts_s",\n    "bwd_pkts_s",\n    "pkt_len_min",\n    "pkt_len_max",\n    "pkt_len_mean",\n    "pkt_len_std",\n    "pkt_len_var",\n    "fin_flag_cnt",\n    "syn_flag_cnt",\n    "rst_flag_cnt",\n    "psh_flag_cnt",\n    "ack_flag_cnt",\n    "urg_flag_cnt",\n    "cwe_flag_count",\n    "ece_flag_cnt",\n    "down_up_ratio",\n    "pkt_size_avg",\n    "fwd_seg_size_avg",\n    "bwd_seg_size_avg",\n    "fwd_byts_b_avg",\n    "fwd_pkts_b_avg",\n    "fwd_blk_rate_avg",\n    "bwd_byts_b_avg",\n    "bwd_pkts_b_avg",\n    "bwd_blk_rate_avg",\n    "subflow_fwd_pkts",\n    "subflow_fwd_byts",\n    "subflow_bwd_pkts",\n    "subflow_bwd_byts",\n    "init_fwd_win_byts",\n    "init_bwd_win_byts",\n    "fwd_act_data_pkts",\n    "fwd_seg_size_min",\n    "active_mean",\n    "active_std",\n    "active_max",\n    "active_min",\n    "idle_mean",\n    "idle_std",\n    "idle_max",\n    "idle_min"\n  ]\n}\n', 'src/ml/train_clustering.py': '"""Run a preregistered clustering version against the verified mounted Phase 2 data."""\n\nfrom collections import Counter\nfrom hashlib import sha256\nimport json\nfrom pathlib import Path\nimport time\n\nfrom pyspark import StorageLevel\nfrom pyspark.ml import PipelineModel\nfrom pyspark.ml.functions import vector_to_array\nfrom pyspark.sql import functions as F\n\nfrom src.ingest.dataset_audit import atomic_json, spark_session\nfrom src.ingest.parquet_pipeline import driver_peak_bytes, parquet_footers, resource_limits\nfrom src.ml.cluster_diagnostics import centroid_dominance, choose_k, compare_assignments, fit_kmeans, silhouette\nfrom src.ml.cluster_evolution import occupancy\nfrom src.ml.clustering_features import clustering_registry, fit_transform, verify_finite_vectors\nfrom src.ml.feature_registry import META, finite\nfrom src.ml.novelty_scoring import calibrate, novelty_metrics, squared_distance\nfrom src.ml.split_protocols import assign_population, crossing_groups, group_summary, mark_retention\nfrom src.ml.train_supervised import verified_input\nfrom src.sql.traffic_analytics import compact, locate_producer, require, write_rows\nfrom src.visualization.clustering_charts import draw_phase5\n\n\ndef read(path):\n    return json.loads(path.read_text())\n\n\ndef cohorts(clean, names, phase4, experiment, days, extra_metadata=()):\n    # Exactly the Phase 4 predictor digest, with clean timestamps carried only as metadata.\n    projected = clean.select(*META, "parsed_event_time", "record_fingerprint", *extra_metadata,\n                             *[finite(x).alias(x) for x in names])\n    projected = projected.withColumn("predictor_key", F.unhex(F.sha2(\n        F.concat(F.lit(phase4["version"] + "\\u0000"), F.to_json(F.array(*names))), 256)))\n    if experiment == "G":\n        return projected.withColumn("split_rank", F.lit(0))\n    return assign_population(projected, experiment, phase4, days)\n\n\ndef pick_reference(frame, config):\n    value = F.pmod(F.xxhash64("predictor_key", F.lit(config["seed"]), F.lit("phase5-reference")), F.lit(10000))\n    ranked = frame.where(F.col("split_rank") == 0).withColumn("sample_bucket", value)\n    boundary = config["training_permyriad"]\n    training = ranked.where(F.col("sample_bucket") < boundary)\n    probe = ranked.where((F.col("sample_bucket") >= boundary) & (F.col("sample_bucket") < boundary + config["internal_permyriad"]))\n    return training, probe\n\n\ndef split_counts(data):\n    return compact(data.groupBy("split_rank", "source_day", "binary_label", "attack_family")\n                   .count().orderBy("split_rank", "source_day", "binary_label", "attack_family"), 1000)\n\n\ndef verify_phase4_match(rows, experiment, phase4_receipt):\n    if experiment == "G":\n        return None\n    expected = phase4_receipt["protocols"][experiment]\n    ranks = ("train", "validation", "test")\n    for i, split in enumerate(ranks):\n        families, labels = Counter(), Counter()\n        for row in rows:\n            if row["split_rank"] == i:\n                families[row["attack_family"]] += row["count"]\n                labels[str(row["binary_label"])] += row["count"]\n        support = expected["split_support"][split]\n        require(dict(families) == support["families"] and dict(labels) == support["labels"], "Phase 4 retained cohort differs")\n    return {"protocol": experiment, "kaggle_version": expected["version"], "checked": "matched split rules and family/binary totals",\n            "purged_rows": expected["overlap"]["purged_rows"]}\n\n\ndef profiles(scores, centers, config):\n    return compact(scores.groupBy("cluster").agg(\n        F.count("*").alias("records"), F.count(F.when(F.col("binary_label") == 1, 1)).alias("attacks"),\n        F.avg("score").alias("mean_score"),\n        F.percentile_approx("score", 0.95, config["quantile_accuracy"]).alias("score_p95"),\n        F.avg("flow_duration").alias("mean_duration_us"), F.avg("tot_fwd_pkts").alias("mean_forward_packets"),\n        F.avg("tot_bwd_pkts").alias("mean_backward_packets"), F.avg("totlen_fwd_pkts").alias("mean_forward_bytes"),\n        F.avg("totlen_bwd_pkts").alias("mean_backward_bytes"), F.avg("flow_pkts_s").alias("mean_rate")\n    ).orderBy("cluster"), centers)\n\n\ndef saved_pipeline(preprocessing, model, probe, target):\n    pipeline = PipelineModel(stages=[*preprocessing.stages, model])\n    pipeline.write().save(str(target))\n    recovered = PipelineModel.load(str(target))\n    small = probe.limit(32).cache()\n    try:\n        expected = sorted(row[0] for row in pipeline.transform(small).select("cluster").collect())\n        actual = sorted(row[0] for row in recovered.transform(small).select("cluster").collect())\n        require(expected and actual == expected, "Saved pipeline reload did not reproduce assignments")\n    finally:\n        small.unpersist()\n    files = [p for p in target.rglob("*") if p.is_file() and not p.name.endswith(".crc")]\n    return {"path": str(target), "reload_rows": len(actual),\n            "sha256": {p.relative_to(target).as_posix(): sha256(p.read_bytes()).hexdigest() for p in sorted(files)},\n            "bytes": sum(p.stat().st_size for p in files)}\n\n\ndef run(project, input_root, output, scratch, experiment):\n    config = read(project / "configs/clustering_experiments.yaml")\n    phase4 = read(project / "configs/ml_experiments.yaml")\n    phase4_receipt = read(project / "manifests/phase4_models.json")\n    manifest = read(project / "manifests/parquet_layers.json")\n    policy = read(project / "configs/cleaning.yaml")\n    names = clustering_registry(phase4, manifest["feature_columns"])\n    output.mkdir(parents=True, exist_ok=True)\n    scratch.mkdir(parents=True, exist_ok=True)\n    spark = spark_session(config, scratch)\n    spark.conf.set("spark.sql.sources.partitionColumnTypeInference.enabled", "false")\n    spark.sparkContext.setLogLevel("WARN")\n    started = time.monotonic()\n    cached = []\n\n    def disk(data):\n        result = data.persist(StorageLevel.DISK_ONLY)\n        cached.append(result)\n        return result\n\n    summary = {"experiment": experiment, "config": config, "status": "running",\n               "resources": resource_limits(spark, config, output), "features": names}\n    path = output / "phase5_summary.json"\n    atomic_json(path, summary)\n\n    def emit(name, rows):\n        write_rows(output / f"phase5_{name}.csv", rows)\n        print(f"PHASE5 {experiment} {name}: {len(rows)} rows", flush=True)\n\n    try:\n        receipt = locate_producer(input_root, manifest["independent_mount"]["summary_sha256"])\n        _, summary["input"] = verified_input(spark, receipt, manifest, policy)\n        clean_path = receipt.parent.parent / manifest["policy_version"] / "clean"\n        require(all(manifest["layers"]["clean"][key] == value for key, value in parquet_footers(clean_path).items()),\n                "Clean Parquet metadata differs")\n        clean = spark.read.parquet(str(clean_path))\n        require(dict(clean.dtypes).get("parsed_event_time") == "timestamp_ntz" and set(names) <= set(clean.columns), "Clean schema changed")\n        base = disk(cohorts(clean, names, phase4, experiment, summary["input"]["days"]))\n        require(base.count() == summary["input"]["rows"], "Input split lost records")\n        observed = split_counts(base)\n        label_total = Counter()\n        for row in observed:\n            label_total[row["binary_label"]] += row["count"]\n        require(dict(label_total) == {0: 13484708, 1: 2748235}, "Clean layer binary counts changed")\n        expected_families = read(receipt)["completed_files"]\n        # The feature-ready layer verifies exact original-label mappings before this check.\n        require(len(expected_families) == len({r["source_day"] for r in observed}), "Clean capture-day inventory changed")\n        require(base.where("record_fingerprint IS NULL OR temporal_eligibility IS NULL").limit(1).count() == 0,\n                "Required record provenance is missing")\n        if experiment == "G":\n            retained = base\n            summary["purge"] = {"groups_before": 0, "groups_after": 0, "rows": 0}\n        else:\n            crossings = disk(crossing_groups(base, "predictor_key"))\n            counts = group_summary(crossings)\n            retained = disk(mark_retention(base, crossings, counts["groups"]).where("retained").drop("retained"))\n            retained_count = retained.count()\n            remaining = group_summary(crossing_groups(retained, "predictor_key"))\n            require(remaining["groups"] == 0, "Overlap remains between reference and later cohorts")\n            summary["purge"] = {"groups_before": counts["groups"], "groups_after": remaining["groups"],\n                                "rows": summary["input"]["rows"] - retained_count}\n            base.unpersist(blocking=True)\n            crossings.unpersist(blocking=True)\n        split = split_counts(retained)\n        summary["matched_phase4"] = verify_phase4_match(split, experiment, phase4_receipt)\n        if summary["matched_phase4"]:\n            require(summary["matched_phase4"]["purged_rows"] == summary["purge"]["rows"], "Phase 4 purge size changed")\n        else:\n            classified = Counter()\n            for row in split:\n                classified[row["attack_family"]] += row["count"]\n            canonical = phase4_receipt["protocols"]["A"]["split_support"]\n            aggregate = Counter()\n            for entry in canonical.values():\n                aggregate.update(entry["families"])\n            require(classified == aggregate, "Clean family frequencies differ")\n        emit("cohorts", split)\n        reference_rows = sum(row["count"] for row in split if row["split_rank"] == 0)\n        train, probe = pick_reference(retained, config)\n        train_features, probe_features = disk(train.select(*names)), disk(probe.select(*names))\n        train_rows, probe_rows = train_features.count(), probe_features.count()\n        require(1 < probe_rows <= config["max_internal_rows"] and train_rows > probe_rows, "Bounded reference cohort differs")\n        prep, diagnostics = fit_transform(train_features, names)\n        summary["preprocessing"] = diagnostics\n        summary["training"] = {"fit_rows": train_rows, "internal_rows": probe_rows, "reference_rows": reference_rows,\n                               "share_of_reference": train_rows / reference_rows}\n        fit = disk(prep.transform(train_features).select("features"))\n        internal = disk(prep.transform(probe_features).select("features"))\n        verify_finite_vectors(fit, len(diagnostics["active"]))\n        verify_finite_vectors(internal, len(diagnostics["active"]))\n        trials, models = [], {}\n        for k in config["candidate_k"]:\n            model, info = fit_kmeans(fit, k, config, config["seed"])\n            require(info["training_rows"] == train_rows, "K-Means training sizes do not reconcile")\n            info["silhouette"] = silhouette(model, internal)\n            info["silhouette_rows"] = probe_rows\n            trials.append(info)\n            models[k] = model\n            atomic_json(path, summary | {"trials": trials})\n            print(f"PHASE5 {experiment} k={k} trained {train_rows} rows, silhouette {info[\'silhouette\']}", flush=True)\n        selected = choose_k(trials)\n        best = models[selected["k"]]\n        summary["selected_k"] = selected["k"]\n        summary["trials"] = trials\n        emit("model_selection", [row | {"selected": row["k"] == selected["k"]} for row in trials])\n\n        # Evolve both sensitivity models exclusively on the same fit and internal reference rows.\n        alternate, alt_seed_info = fit_kmeans(fit, selected["k"], config, config["seed"] + 1)\n        seed_ari, _ = compare_assignments(best, alternate, internal.withColumn("alternate_features", F.col("features")))\n        linear, linear_prep = fit_transform(train_features, names, transform="linear")\n        linear_fit = disk(linear.transform(train_features).select("features"))\n        verify_finite_vectors(linear_fit, len(linear_prep["active"]))\n        linear_model, linear_info = fit_kmeans(linear_fit, selected["k"], config, config["seed"])\n        paired = linear.transform(prep.transform(probe_features).withColumnRenamed("features", "primary_features")\n                                  .select(*names, "primary_features"))             .select(F.col("primary_features").alias("features"), F.col("features").alias("alternate_features"))\n        paired = disk(paired)\n        scale_ari, count = compare_assignments(best, linear_model, paired)\n        summary["robustness"] = {"paired_rows": count, "seed_ari": seed_ari, "scale_ari": scale_ari,\n                                 "alternate_seed": alt_seed_info, "linear": linear_info,\n                                 "linear_silhouette": silhouette(linear_model, paired.select(F.col("alternate_features").alias("features"))),\n                                 "dominance": centroid_dominance(best.clusterCenters(), selected["cluster_sizes"], diagnostics["active"])}\n        # Distances for threshold calibration come from the disjoint internal training-day sample.\n        calibration = calibrate(squared_distance(best, internal).select(F.lit(0).alias("split_rank"), "score"),\n                                config["reference_percentiles"], config["quantile_accuracy"])\n        summary["calibration"] = calibration\n        require(calibration["rows"] == probe_rows, "Thresholds were calibrated on an unexpected cohort")\n        for frame in (fit, internal, linear_fit, paired, train_features):\n            frame.unpersist(blocking=True)\n        assembled = disk(prep.transform(retained).select(\n            "source_day", "split_rank", "temporal_eligibility", "parsed_event_time", "binary_label", "attack_family",\n            "original_label", "record_fingerprint", "features", "flow_duration", "tot_fwd_pkts", "tot_bwd_pkts",\n            "totlen_fwd_pkts", "totlen_bwd_pkts", "flow_pkts_s",\n        ))\n        verify_finite_vectors(assembled, len(diagnostics["active"]))\n        total = sum(row["count"] for row in split)\n        scored = disk(squared_distance(best, assembled).drop("features"))\n        require(scored.count() == total, "Scoring changed retained count")\n        assembled.unpersist(blocking=True)\n        require(scored.where("score IS NULL OR isnan(score) OR score < 0 OR abs(score) = double(\'Infinity\')").limit(1).count() == 0,\n                "Centroid distance is invalid")\n        # Diagnose invalid and missing inputs in the retained train/evaluation populations.\n        null_counts = compact(retained.groupBy("split_rank").agg(F.count("*").alias("rows"), *[\n            F.count(F.when(F.col(name).isNull(), 1)).alias(name) for name in names\n        ]).orderBy("split_rank"), 3)\n        emit("transform_diagnostics", [{"split_rank": row["split_rank"], "rows": row["rows"], "feature": name,\n                                         "nullified_or_missing": row[name]} for row in null_counts for name in names])\n        selected_profiles = profiles(scored, selected["k"], config)\n        require(sum(row["records"] for row in selected_profiles) == total, "Cluster profiles lost rows")\n        emit("cluster_profiles", [row | {"attack_share": row["attacks"] / row["records"],\n                                        "binary_purity": max(row["attacks"], row["records"] - row["attacks"]) / row["records"]}\n                                 for row in selected_profiles])\n        emit("centroids", [{"cluster": i, "feature": name, "center": float(center[j])}\n                          for i, center in enumerate(best.clusterCenters()) for j, name in enumerate(diagnostics["active"])])\n        composition = compact(scored.groupBy("source_day", "split_rank", "cluster", "binary_label", "attack_family", "original_label")\n                              .count().orderBy("source_day", "cluster", "original_label"), 3500)\n        emit("composition", composition)\n        require(sum(row["count"] for row in composition) == total, "Composition changed retained count")\n        windows = compact(scored.where("temporal_eligibility").withColumn(\n            "window_hour", (F.floor(F.hour("parsed_event_time") / config["window_hours"]) * config["window_hours"]).cast("int"))\n            .groupBy("source_day", "split_rank", "window_hour", "cluster", "attack_family").count()\n            .orderBy("source_day", "window_hour", "cluster", "attack_family"), 6500)\n        emit("window_family", windows)\n        primary = calibration["cutoffs"][str(config["primary_percentile"])]\n        populated, shifts, reference = occupancy(scored, selected["k"], config, primary)\n        require(sum(row["records"] for row in populated if row["window_hour"] == -1) == total, "Daily occupancy changed retained count")\n        require(sum(row["records"] for row in populated if row["window_hour"] >= 0) == sum(row["count"] for row in windows),\n                "Eligible event-time tables differ")\n        emit("cluster_occupancy", populated)\n        emit("temporal_shift", shifts)\n        summary["reference_occupancy"] = reference\n        catalog = sorted({entry["family"] for entry in policy["label_mapping"].values() if entry["binary"]})\n        emit("novelty_metrics", novelty_metrics(scored, calibration["cutoffs"], catalog, config["rare_support"]))\n        histogram = compact(scored.withColumn("score_bin", F.floor(F.log1p("score") * 4).cast("int"))\n                            .groupBy("split_rank", "binary_label", "score_bin").count()\n                            .orderBy("split_rank", "binary_label", "score_bin"), 1000)\n        emit("score_histogram", histogram)\n        require(sum(row["count"] for row in histogram) == total, "Distance histogram changed retained count")\n        if experiment == "G":\n            duplicates = compact(scored.groupBy("record_fingerprint", "cluster").count().groupBy("cluster").agg(\n                F.sum("count").alias("records"), F.count("*").alias("exact_digest_representatives"),\n                F.sum(F.col("count") - 1).alias("excess")\n            ).orderBy("cluster"), selected["k"])\n            require(sum(row["excess"] for row in duplicates) == 410707, "Exact duplicate sensitivity differs from Phase 2")\n            emit("duplicate_sensitivity", duplicates)\n        summary["retained_rows"] = total\n        summary["model"] = saved_pipeline(prep, best, probe_features, output / "model")\n        summary.update(status="validated", elapsed_seconds=round(time.monotonic() - started, 3))\n        summary["resources"]["driver_peak_rss_bytes"] = driver_peak_bytes(spark)\n        atomic_json(path, summary)\n        summary["figures"] = draw_phase5(output, experiment, project / "reports/phase4_metrics.csv")\n        atomic_json(path, summary)\n        print(f"PHASE5 {experiment} complete k={selected[\'k\']} trained {train_rows}, scored {total} in {summary[\'elapsed_seconds\']}s", flush=True)\n    except Exception as exc:\n        summary.update(status="partial", error=f"{type(exc).__name__}: {str(exc)[:500]}")\n        atomic_json(path, summary)\n        raise\n    finally:\n        for frame in reversed(cached):\n            frame.unpersist()\n        spark.stop()\n', 'src/ml/clustering_features.py': '"""Label-free Spark feature preparation for distance-based experiments."""\n\nfrom pyspark.ml import Pipeline, PipelineModel\nfrom pyspark.ml.feature import Imputer, SQLTransformer, StandardScaler, VectorAssembler\nfrom pyspark.ml.functions import vector_to_array\nfrom pyspark.sql import functions as F\n\nfrom src.ml.feature_registry import FORBIDDEN, registry\nfrom src.sql.traffic_analytics import identifier, require\n\n\ndef clustering_registry(phase4, upstream):\n    return registry(phase4, upstream)\n\n\ndef fit_transform(training_values, names, transform="signed_log1p"):\n    if set(training_values.columns) != set(names) or set(names) & FORBIDDEN:\n        raise ValueError("Reference fit takes approved predictor columns only")\n    if transform not in {"signed_log1p", "linear"}:\n        raise ValueError("Unknown clustering transform")\n    clean = []\n    for name in names:\n        x = identifier(name)\n        value = (f"CASE WHEN {x} IS NOT NULL AND NOT isnan({x}) "\n                 f"AND abs({x}) < CAST(\'Infinity\' AS DOUBLE) THEN {x} END")\n        numerical = f"signum({value}) * log1p(abs({value}))" if transform == "signed_log1p" else value\n        clean.append(f"{numerical} AS {x}_valid")\n    sanitizer = SQLTransformer(statement="SELECT *, " + ", ".join(clean) + " FROM __THIS__")\n    finite = sanitizer.transform(training_values)\n    extrema = finite.agg(*[\n        expr.alias(f"{name}_{part}") for name in names\n        for part, expr in (("min", F.min(f"{name}_valid")), ("max", F.max(f"{name}_valid")),\n                           ("null", F.count(F.when(F.col(f"{name}_valid").isNull(), 1))))\n    ]).first().asDict()\n    active = [x for x in names if extrema[x + "_min"] is not None and extrema[x + "_min"] != extrema[x + "_max"]]\n    if not active:\n        raise ValueError("Reference population has no varying predictors")\n    assembled = Pipeline(stages=[\n        sanitizer, Imputer(inputCols=[x + "_valid" for x in active], outputCols=[x + "_filled" for x in active],\n                           strategy="median", relativeError=0.001),\n        VectorAssembler(inputCols=[x + "_filled" for x in active], outputCol="unscaled", handleInvalid="error"),\n    ]).fit(training_values)\n    scaler = StandardScaler(inputCol="unscaled", outputCol="features", withStd=True, withMean=True).fit(\n        assembled.transform(training_values))\n    stats = {name: {"invalid_or_null": extrema[name + "_null"], "available": name in active} for name in names}\n    for i, name in enumerate(active):\n        stats[name]["reference_std"] = float(scaler.std[i])\n        stats[name]["near_constant"] = float(scaler.std[i]) < 1e-8\n    return PipelineModel(stages=[*assembled.stages, scaler]), {\n        "transform": transform, "fit_columns": list(names), "active": active,\n        "dropped": [name for name in names if name not in active], "reference_stats": stats,\n        "imputation_surrogates": assembled.stages[1].surrogateDF.first().asDict(),\n        "reference_means": [float(x) for x in scaler.mean], "reference_scales": [float(x) for x in scaler.std],\n    }\n\n\ndef verify_finite_vectors(data, size):\n    values = vector_to_array("features")\n    bad = (F.size(values) != size) | F.exists(values, lambda x: F.isnan(x) | (F.abs(x) == float("inf")) | x.isNull())\n    require(data.where(bad).limit(1).count() == 0, "Nonfinite clustering vector")\n', 'src/ml/novelty_scoring.py': '"""Nearest-centroid squared distances and reference-only score calibration."""\n\nfrom pyspark.ml.functions import vector_to_array\nfrom pyspark.sql import functions as F\n\n\ndef squared_distance(model, data):\n    centers = F.array(*[F.array(*[F.lit(float(x)) for x in center]) for center in model.clusterCenters()])\n    prediction = model.transform(data)\n    assigned = F.element_at(centers, (F.col("cluster") + 1).cast("int"))\n    deltas = F.zip_with(vector_to_array("features"), assigned, lambda x, y: (x - y) * (x - y))\n    return prediction.withColumn("score", F.aggregate(deltas, F.lit(0.0), lambda acc, value: acc + value))\n\n\ndef calibrate(reference_scores, percentiles, accuracy):\n    if set(reference_scores.columns) != {"split_rank", "score"} or reference_scores.where("split_rank <> 0 OR split_rank IS NULL").limit(1).count():\n        raise ValueError("Threshold calibration requires reference score rows only")\n    row = reference_scores.agg(F.count("*").alias("rows"),\n                               F.percentile_approx("score", percentiles, accuracy).alias("cutoffs")).first()\n    if not row.rows or row.cutoffs is None:\n        raise ValueError("Empty novelty calibration population")\n    cutoffs = dict(zip(map(str, percentiles), row.cutoffs))\n    exceedance = reference_scores.agg(*[\n        F.count(F.when(F.col("score") > value, 1)).alias(p) for p, value in cutoffs.items()\n    ]).first().asDict()\n    return {"rows": row.rows, "cutoffs": cutoffs, "accuracy": accuracy, "reference_flags": exceedance}\n\n\ndef safe_div(a, b):\n    return a / b if b else None\n\n\ndef novelty_row(positive, negative, tp, fp):\n    count = positive + negative\n    return {"rows": count, "attacks": positive, "benign": negative, "tp": tp, "fp": fp,\n            "flags": tp + fp, "flag_rate": safe_div(tp + fp, count),\n            "benign_fpr": safe_div(fp, negative), "attack_recall": safe_div(tp, positive),\n            "precision": safe_div(tp, tp + fp)}\n\n\ndef novelty_metrics(data, cutoffs, catalog, minimum):\n    rows = []\n    for p, cutoff in cutoffs.items():\n        flag = F.col("score") > cutoff\n        projection = data.withColumn("flag", flag)\n        measured = projection.groupBy("split_rank", "binary_label", "attack_family").agg(\n            F.count("*").alias("records"), F.count(F.when(F.col("flag"), 1)).alias("flagged")).collect()\n        totals, support = {}, {}\n        for row in measured:\n            totals.setdefault(row.split_rank, [0, 0, 0, 0])\n            totals[row.split_rank][0 if row.binary_label else 1] += row.records\n            totals[row.split_rank][2 if row.binary_label else 3] += row.flagged\n            support[row.split_rank, row.attack_family] = (row.records, row.flagged)\n        for rank, counts in sorted(totals.items()):\n            rows.append({"split_rank": rank, "percentile": p, "threshold": cutoff, "scope": "binary", "family": "All",\n                         **novelty_row(*counts), "support_status": "observed"})\n            for family in catalog:\n                s, hit = support.get((rank, family), (0, 0))\n                rows.append({"split_rank": rank, "percentile": p, "threshold": cutoff, "scope": "family", "family": family,\n                             **(novelty_row(s, 0, hit, 0) | {"precision": None}),\n                             "support_status": "absent" if not s else "low" if s < minimum else "observed"})\n    return rows\n', 'src/ml/cluster_diagnostics.py': '"""Training-only model selection and permutation-invariant robustness metrics."""\n\nimport math\nimport time\n\nfrom pyspark.ml.clustering import KMeans\nfrom pyspark.ml.evaluation import ClusteringEvaluator\nfrom pyspark.sql import functions as F\n\nfrom src.ingest.parquet_pipeline import driver_peak_bytes\n\n\ndef fit_kmeans(vectors, k, config, seed):\n    if vectors.columns != ["features"]:\n        raise ValueError("K-Means receives only the fitted feature vector")\n    start = time.monotonic()\n    model = KMeans(k=k, seed=seed, initMode="k-means||", initSteps=config["init_steps"],\n                   maxIter=config["max_iter"], tol=config["tolerance"], featuresCol="features",\n                   predictionCol="cluster", distanceMeasure="euclidean").fit(vectors)\n    elapsed = round(time.monotonic() - start, 3)\n    summary = model.summary\n    sizes = [int(v) for v in summary.clusterSizes]\n    fraction = min(sizes) / sum(sizes) if sizes and sum(sizes) else 0.0\n    return model, {\n        "k": k, "seed": seed, "initialization": "k-means||", "budget": config["max_iter"], "iterations": summary.numIter,\n        "wssse": summary.trainingCost, "cluster_sizes": sizes, "training_rows": sum(sizes),\n        "empty_clusters": k - sum(size > 0 for size in sizes), "smallest_share": fraction, "fit_seconds": elapsed,\n        "jvm_peak_rss_bytes": driver_peak_bytes(vectors.sparkSession),\n    }\n\n\ndef silhouette(model, probe):\n    predictions = model.transform(probe)\n    if predictions.select("cluster").distinct().count() < 2:\n        return None\n    value = ClusteringEvaluator(featuresCol="features", predictionCol="cluster",\n                                metricName="silhouette", distanceMeasure="squaredEuclidean").evaluate(predictions)\n    return value if math.isfinite(value) else None\n\n\ndef choose_k(rows):\n    eligible = [row for row in rows if not row["empty_clusters"] and row["smallest_share"] >= 0.001\n                and row["silhouette"] is not None]\n    if not eligible:\n        raise ValueError("All candidates are degenerate or lack silhouette support")\n    return max(eligible, key=lambda row: (row["silhouette"], -row["k"]))\n\n\ndef adjusted_rand(contingency):\n    n = sum(n for _, _, n in contingency)\n    if n < 2:\n        return None\n    left, right = {}, {}\n    for a, b, count in contingency:\n        left[a] = left.get(a, 0) + count\n        right[b] = right.get(b, 0) + count\n    observed = sum(math.comb(count, 2) for _, _, count in contingency)\n    a, b = (sum(math.comb(count, 2) for count in side.values()) for side in (left, right))\n    expected = a * b / math.comb(n, 2)\n    denominator = (a + b) / 2 - expected\n    return (observed - expected) / denominator if denominator else 1.0\n\n\ndef compare_assignments(standard, alternative, probe):\n    pairs = (alternative.transform(probe.withColumnRenamed("features", "primary_features")\n                                   .withColumnRenamed("alternate_features", "features"))\n             .withColumnRenamed("cluster", "other_cluster").drop("features")\n             .withColumnRenamed("primary_features", "features"))\n    counts = standard.transform(pairs).groupBy("cluster", "other_cluster").count().collect()\n    return adjusted_rand([(r.cluster, r.other_cluster, r["count"]) for r in counts]), sum(row["count"] for row in counts)\n\n\ndef centroid_dominance(centers, sizes, features):\n    normalized = [n / sum(sizes) for n in sizes]\n    sums = []\n    for j, feature in enumerate(features):\n        mean = sum(weight * float(c[j]) for weight, c in zip(normalized, centers))\n        sums.append((feature, sum(weight * (float(c[j]) - mean) ** 2 for weight, c in zip(normalized, centers))))\n    total = sum(v for _, v in sums)\n    return [{"feature": name, "share_between_centroids": share / total if total else 0.0}\n            for name, share in sorted(sums, key=lambda item: (-item[1], item[0]))]\n', 'src/ml/cluster_evolution.py': '"""Capture-day and three-hour occupancy comparisons on a frozen model."""\n\nimport math\n\nfrom pyspark.sql import functions as F\n\nfrom src.sql.traffic_analytics import compact\n\n\ndef divergence(left, right):\n    if sum(left) == 0 or sum(right) == 0 or len(left) != len(right):\n        return None\n    p = [value / sum(left) for value in left]\n    q = [value / sum(right) for value in right]\n    distance = sum(\n        (a * math.log2(2 * a / (a + b)) if a else 0.0)\n        + (b * math.log2(2 * b / (a + b)) if b else 0.0) for a, b in zip(p, q)\n    ) / 2\n    return max(0.0, distance)\n\n\ndef occupancy(data, k, config, cutoff):\n    hours = config["window_hours"]\n    bins = F.floor(F.hour("parsed_event_time") / hours) * hours\n    day = data.withColumn("window_hour", F.lit(-1))\n    window = data.where("temporal_eligibility").withColumn("window_hour", bins.cast("int"))\n    fields = ("source_day", "split_rank", "window_hour", "cluster", "score", "binary_label", "flow_duration", "tot_fwd_pkts", "totlen_fwd_pkts")\n    scoped = day.select(*fields).unionByName(window.select(*fields))\n    raw = compact(scoped.groupBy("source_day", "split_rank", "window_hour", "cluster").agg(\n        F.count("*").alias("records"), F.count(F.when(F.col("binary_label") == 1, 1)).alias("attacks"),\n        F.count(F.when(F.col("score") > cutoff, 1)).alias("flags"),\n        F.avg("score").alias("mean_score"), F.percentile_approx("score", 0.95, config["quantile_accuracy"]).alias("score_p95"),\n        F.avg("flow_duration").alias("mean_duration_us"), F.avg("tot_fwd_pkts").alias("mean_forward_packets"),\n        F.avg("totlen_fwd_pkts").alias("mean_forward_bytes")\n    ).orderBy("source_day", "window_hour", "cluster"), 2000)\n    reference = [0] * k\n    for row in raw:\n        if row["split_rank"] == 0 and row["window_hour"] == -1:\n            reference[row["cluster"]] += row["records"]\n    cohorts = {}\n    for row in raw:\n        key = row["source_day"], row["split_rank"], row["window_hour"]\n        cohorts.setdefault(key, [0] * k)[row["cluster"]] += row["records"]\n    rows = [row | {"occupancy": row["records"] / sum(cohorts[row["source_day"], row["split_rank"], row["window_hour"]])} for row in raw]\n    shifts = [{"source_day": day, "split_rank": rank, "window_hour": hour, "rows": sum(counts),\n               "occupied_clusters": sum(n > 0 for n in counts), "js_divergence_bits": divergence(reference, counts),\n               "reference_rows": sum(reference)} for (day, rank, hour), counts in sorted(cohorts.items())]\n    return rows, shifts, reference\n', 'src/visualization/clustering_charts.py': '"""Generate compact Phase 5 figures from reconciled aggregate tables."""\n\nfrom collections import Counter\nimport json\nfrom pathlib import Path\n\nimport matplotlib\n\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nimport numpy as np\n\n\n\ndef draw_phase5(root, experiment, supervised_csv):\n    from csv import DictReader\n\n    def read(name):\n        with (root / f"phase5_{name}.csv").open(newline="", encoding="utf-8") as stream:\n            return list(DictReader(stream))\n\n    summary = json.loads((root / "phase5_summary.json").read_text())\n    profiles, composition = read("cluster_profiles"), read("composition")\n    occupancy, shifts, centroids = read("cluster_occupancy"), read("temporal_shift"), read("centroids")\n    novelty, histogram = read("novelty_metrics"), read("score_histogram")\n    total = sum(int(item["records"]) for item in profiles)\n    if total != summary["retained_rows"] or sum(int(item["count"]) for item in composition) != total:\n        raise ValueError("Chart totals do not match the retained population")\n    folder = root / "figures"\n    folder.mkdir(exist_ok=True)\n    prefix = f"phase5_{experiment.lower()}"\n    k = summary["selected_k"]\n    ids = list(range(k))\n    figures = []\n    colors = plt.get_cmap("tab20")(np.linspace(0, 1, k))\n\n    def show(fig, name):\n        target = folder / f"{prefix}_{name}.png"\n        fig.savefig(target, dpi=120, bbox_inches="tight", metadata={"Software": "MMDS Phase 5"})\n        plt.close(fig)\n        figures.append(target.name)\n\n    def bar(name, items, label, title):\n        fig, ax = plt.subplots(figsize=(6.8, 3.4), layout="constrained")\n        ax.bar([str(a) for a, _ in items], [b for _, b in items], color="#087e8b")\n        ax.set_ylabel(label)\n        ax.set_title(title, loc="left")\n        ax.grid(axis="y", alpha=.2)\n        show(fig, name)\n\n    bar("cluster_sizes", [(r["cluster"], int(r["records"])) for r in profiles], "Records",\n        f"{experiment} · cluster size across retained captures")\n    bar("cluster_attack_share", [(r["cluster"], 100 * float(r["attack_share"])) for r in profiles],\n        "Observed attack share (%)", f"{experiment} · external label composition")\n    families = sorted({r["attack_family"] for r in composition})\n    matrix = Counter()\n    for row in composition:\n        matrix[int(row["cluster"]), row["attack_family"]] += int(row["count"])\n    counts = [[matrix[i, family] for family in families] for i in ids]\n    fig, ax = plt.subplots(figsize=(7.8, 3.9), layout="constrained")\n    view = ax.imshow(np.log10(1 + np.array(counts)), aspect="auto", cmap="cividis")\n    ax.set_xticks(range(len(families)), families, rotation=25, ha="right")\n    ax.set_yticks(ids, ids)\n    ax.set_ylabel("Cluster")\n    ax.set_title(f"{experiment} · cluster by observed family")\n    fig.colorbar(view, ax=ax, label="log10(1 + records)")\n    show(fig, "family_heatmap")\n\n    daily = [r for r in occupancy if r["window_hour"] == "-1"]\n    days = sorted({r["source_day"] for r in daily})\n    posts = {(r["source_day"], int(r["cluster"])): float(r["occupancy"]) for r in daily}\n    if sum(int(r["records"]) for r in daily) != total or any(\n        abs(sum(posts.get((day, i), 0) for i in ids) - 1) > 1e-8 for day in days\n    ):\n        raise ValueError("Chart occupancy partitions do not reconcile")\n    fig, ax = plt.subplots(figsize=(8.1, 3.8), layout="constrained")\n    bottom = np.zeros(len(days))\n    for i in ids:\n        y = [100 * posts.get((day, i), 0) for day in days]\n        ax.bar([d[5:] for d in days], y, bottom=bottom, color=colors[i], label=str(i))\n        bottom += y\n    ax.set_ylabel("Cluster occupancy (%)")\n    ax.set_title(f"{experiment} · occupancy by capture day")\n    ax.legend(ncols=min(k, 6), title="Cluster", fontsize=7)\n    show(fig, "daily_occupancy")\n    bar("temporal_shift", [(r["source_day"][5:], float(r["js_divergence_bits"]))\n                           for r in shifts if r["window_hour"] == "-1"], "Jensen–Shannon divergence (bits)",\n        f"{experiment} · daily occupancy versus reference")\n\n    names = summary["preprocessing"]["active"]\n    positions = {(int(r["cluster"]), r["feature"]): float(r["center"]) for r in centroids}\n    fig, ax = plt.subplots(figsize=(9.5, 4.6), layout="constrained")\n    view = ax.imshow([[positions[i, key] for key in names] for i in ids], aspect="auto", cmap="coolwarm")\n    ax.set_xticks(range(len(names)), names, rotation=60, ha="right", fontsize=7)\n    ax.set_yticks(ids, ids)\n    ax.set_ylabel("Cluster")\n    ax.set_title(f"{experiment} · centered and scaled signed-log centroids")\n    fig.colorbar(view, ax=ax, label="Reference standard deviations")\n    show(fig, "centroid_profiles")\n\n    rank = "0" if experiment == "G" else "2"\n    fig, ax = plt.subplots(figsize=(7, 3.6), layout="constrained")\n    for label, title, color in (("0", "Benign", "#087e8b"), ("1", "Attack", "#b44a31")):\n        rows = sorted((r for r in histogram if r["split_rank"] == rank and r["binary_label"] == label),\n                      key=lambda row: int(row["score_bin"]))\n        if rows:\n            denominator = sum(int(r["count"]) for r in rows)\n            ax.plot([int(r["score_bin"]) / 4 for r in rows], [int(r["count"]) / denominator for r in rows],\n                    marker=".", label=title, color=color)\n    ax.set_xlabel("log1p(squared distance), 0.25-width bins")\n    ax.set_ylabel("Within-label fraction per bin")\n    ax.set_title(f"{experiment} · {\'all retained\' if rank == \'0\' else \'test\'} distance distribution")\n    ax.legend()\n    show(fig, "distance_histogram")\n\n    day_counts = Counter()\n    day_flags = Counter()\n    for row in daily:\n        day_counts[row["source_day"]] += int(row["records"])\n        day_flags[row["source_day"]] += int(row["flags"])\n    bar("daily_novelty", [(day[5:], 100 * day_flags[day] / day_counts[day]) for day in days],\n        "Rows above reference P99 (%)", f"{experiment} · novelty flags by capture day")\n\n    primary = next(r for r in novelty if r["split_rank"] == rank and r["percentile"] == "0.99" and r["scope"] == "binary")\n    bar("binary_novelty", [("Benign", 100 * float(primary["benign_fpr"])), ("Attack", 100 * float(primary["attack_recall"]))],\n        "Within-label flag rate (%)", f"{experiment} · {\'all retained\' if rank == \'0\' else \'test\'} at reference P99")\n    if experiment != "G":\n        with Path(supervised_csv).open(newline="", encoding="utf-8") as stream:\n            rf = next(row for row in DictReader(stream) if row["protocol"] == experiment and row["split"] == "test"\n                      and row["model"] == "random_forest" and row["training_cohort"] == "group_sample")\n        if int(rf["tp"]) + int(rf["fn"]) != int(primary["attacks"]) or int(rf["tn"]) + int(rf["fp"]) != int(primary["benign"]):\n            raise ValueError("Comparison chart test denominators differ")\n        fig, ax = plt.subplots(figsize=(7.0, 3.8), layout="constrained")\n        x = np.arange(2)\n        ax.bar(x - .16, [100 * (1 - float(rf["specificity"])), 100 * float(rf["recall"])], width=.32, label="Phase 4 RF, 0.5")\n        ax.bar(x + .16, [100 * float(primary["benign_fpr"]), 100 * float(primary["attack_recall"])], width=.32,\n               label="Centroid distance, reference P99")\n        ax.set_xticks(x, ["Benign false positives", "Attack recall"])\n        ax.set_ylabel("Rate (%)")\n        ax.set_title(f"{experiment} · cohort-matched aggregate signals")\n        ax.legend()\n        show(fig, "supervised_comparison")\n    return figures\n', 'configs/clustering_experiments.yaml': '{\n  "version": "phase5-v1",\n  "seed": 23013,\n  "candidate_k": [3, 5, 8, 12],\n  "max_iter": 16,\n  "init_steps": 2,\n  "tolerance": 0.0001,\n  "training_permyriad": 250,\n  "internal_permyriad": 12,\n  "reference_percentiles": [0.95, 0.99, 0.999],\n  "primary_percentile": 0.99,\n  "quantile_accuracy": 8192,\n  "max_internal_rows": 40000,\n  "rare_support": 100,\n  "window_hours": 3,\n  "lag_reference": "full retained phase4 train-day population",\n  "spark": {"master": "local[4]", "driver_memory": "8g", "shuffle_partitions": 48, "input_partition_bytes": 67108864}\n}\n', 'manifests/phase4_models.json': '{\n  "status": "validated",\n  "kernel_url": "https://www.kaggle.com/code/ramcharan2345/mmds-phase-4-supervised-attack-detection",\n  "config_version": "phase4-v1",\n  "seed": 23013,\n  "metric_policy": {\n    "positive_label": 1,\n    "threshold": 0.5,\n    "score_bins": 2048,\n    "pr_auc": "step-integrated binned average precision",\n    "roc_auc": "trapezoidal binned ROC"\n  },\n  "protocols": {\n    "A": {\n      "version": "phase4-v1",\n      "status": "validated",\n      "log_events": 952,\n      "protocol": "A",\n      "design": {\n        "name": "predictor-group random"\n      },\n      "resources": {\n        "spark_version": "4.3.0",\n        "master": "local[4]",\n        "driver_memory": "8g",\n        "shuffle_partitions": 48,\n        "input_partition_bytes": 67108864,\n        "default_parallelism": 4,\n        "cpu_affinity": 4,\n        "scratch_output_free_bytes_start": 20940279808,\n        "memory.max": "32212254720",\n        "cpu.max": "400000 100000",\n        "driver_peak_rss_bytes": 7474003968\n      },\n      "models": [\n        {\n          "path": "/kaggle/working/mmds_phase4/models/logistic_regression_group_sample",\n          "model": "logistic_regression",\n          "reload_probe_rows": 695307,\n          "model_bytes": 16239,\n          "model_files": 16,\n          "sha256": {\n            "metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "metadata/part-00000-d22084df-5d33-404a-9fda-26a9776312cd-c000.txt": "f005026679ac611c02e066dd3303ee750f90f51995f41ff60b7e0eb1fba64375",\n            "stages/0_Imputer_abfd4cd929fc/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/0_Imputer_abfd4cd929fc/data/part-00000-a26e794d-22f5-4b92-8881-621f6128fc27-c000.snappy.parquet": "6782881f601f55774f6a3a328f36b8e67f1a4c597e029385bf75967133da6c93",\n            "stages/0_Imputer_abfd4cd929fc/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/0_Imputer_abfd4cd929fc/metadata/part-00000-63fcca9b-f1ab-4f0b-a432-1d2b90bbd47a-c000.txt": "7c47f4e28af6ded54377e01998e0a20f73361071b79171107cdea893e3ee1a5a",\n            "stages/1_VectorAssembler_74fb923cd0b6/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/1_VectorAssembler_74fb923cd0b6/metadata/part-00000-cdec3ca8-3b2e-44f5-8df1-f4d37510420d-c000.txt": "93504030dc87b4b4f7024d52d575af05938a50e7b0cc192b2f0175f8b654b899",\n            "stages/2_StandardScaler_7ec078203426/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/2_StandardScaler_7ec078203426/data/part-00000-f00ed6b1-97a9-4b2f-abc6-ae310b7bb5f8-c000.snappy.parquet": "9c258408f8b723d8defb95ac34ec00ba12ac69d56608731095db3bba6f26fc56",\n            "stages/2_StandardScaler_7ec078203426/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/2_StandardScaler_7ec078203426/metadata/part-00000-34893155-4a3a-42c0-896c-30c3482ed6b9-c000.txt": "0ca8ef31bd4cae35781d3a71af933f5e35073b8c4eaadfd5b1f278517f76e529",\n            "stages/3_LogisticRegression_adbfc1c28bd8/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/3_LogisticRegression_adbfc1c28bd8/data/part-00000-cb77979f-85c3-40d3-9084-547f9be98e01-c000.snappy.parquet": "64026013080a9f13f3ed3bec051db9d6acac0bd6f0706a7a17c8ea3975e49f23",\n            "stages/3_LogisticRegression_adbfc1c28bd8/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/3_LogisticRegression_adbfc1c28bd8/metadata/part-00000-14a1eb4f-b7b0-4932-bfec-8795703e1cc2-c000.txt": "1acb85d84306f1006da5833a4bcf9dd02a1ee659c5975642a01bf5611c791447"\n          },\n          "training_cohort": "group_sample",\n          "training_seconds": 6.295,\n          "parameters": {\n            "maxIter": 12,\n            "regParam": 0.05,\n            "elasticNetParam": 0.0\n          },\n          "explanation": [\n            {\n              "feature": "ack_flag_cnt",\n              "method": "scaled_lr_coefficient",\n              "value": 0.6055004815071301\n            },\n            {\n              "feature": "rst_flag_cnt",\n              "method": "scaled_lr_coefficient",\n              "value": 0.41801635461493525\n            },\n            {\n              "feature": "pkt_len_mean",\n              "method": "scaled_lr_coefficient",\n              "value": -0.33783932660100696\n            },\n            {\n              "feature": "syn_flag_cnt",\n              "method": "scaled_lr_coefficient",\n              "value": -0.28812792814204863\n            },\n            {\n              "feature": "flow_duration",\n              "method": "scaled_lr_coefficient",\n              "value": -0.21893573945720465\n            },\n            {\n              "feature": "flow_pkts_s",\n              "method": "scaled_lr_coefficient",\n              "value": -0.1911204322503146\n            },\n            {\n              "feature": "flow_byts_s",\n              "method": "scaled_lr_coefficient",\n              "value": -0.10934142934216814\n            },\n            {\n              "feature": "flow_iat_std",\n              "method": "scaled_lr_coefficient",\n              "value": -0.09230271255991071\n            },\n            {\n              "feature": "down_up_ratio",\n              "method": "scaled_lr_coefficient",\n              "value": -0.06399728305368631\n            },\n            {\n              "feature": "fin_flag_cnt",\n              "method": "scaled_lr_coefficient",\n              "value": -0.05260695834107492\n            },\n            {\n              "feature": "fwd_iat_mean",\n              "method": "scaled_lr_coefficient",\n              "value": -0.04171377005068665\n            },\n            {\n              "feature": "flow_iat_mean",\n              "method": "scaled_lr_coefficient",\n              "value": -0.03784969621292684\n            },\n            {\n              "feature": "tot_fwd_pkts",\n              "method": "scaled_lr_coefficient",\n              "value": 0.033926155607644295\n            },\n            {\n              "feature": "totlen_fwd_pkts",\n              "method": "scaled_lr_coefficient",\n              "value": 0.028382859854910233\n            },\n            {\n              "feature": "bwd_iat_mean",\n              "method": "scaled_lr_coefficient",\n              "value": 0.016827809341042027\n            },\n            {\n              "feature": "pkt_len_std",\n              "method": "scaled_lr_coefficient",\n              "value": -0.006682537282281606\n            },\n            {\n              "feature": "tot_bwd_pkts",\n              "method": "scaled_lr_coefficient",\n              "value": -0.002908947666697675\n            },\n            {\n              "feature": "totlen_bwd_pkts",\n              "method": "scaled_lr_coefficient",\n              "value": -0.0010682323226017172\n            }\n          ]\n        },\n        {\n          "path": "/kaggle/working/mmds_phase4/models/random_forest_group_sample",\n          "model": "random_forest",\n          "reload_probe_rows": 695307,\n          "model_bytes": 64114,\n          "model_files": 14,\n          "sha256": {\n            "metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "metadata/part-00000-03f8af52-37ff-4e51-b578-9c9a31cb044a-c000.txt": "c472c97b7d4f76325a5c2bff4801f373f84771ce67a5754dd62adc5556344050",\n            "stages/0_Imputer_abfd4cd929fc/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/0_Imputer_abfd4cd929fc/data/part-00000-1afea790-1b40-4f38-88b9-4900cc34bcdc-c000.snappy.parquet": "6782881f601f55774f6a3a328f36b8e67f1a4c597e029385bf75967133da6c93",\n            "stages/0_Imputer_abfd4cd929fc/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/0_Imputer_abfd4cd929fc/metadata/part-00000-46157776-e1d4-41bc-9c67-47ef8c332784-c000.txt": "dfb6194e70d6199e21d5264bf1b8ed6b623287d2d889fde1e04ffabab2d9630c",\n            "stages/1_VectorAssembler_74fb923cd0b6/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/1_VectorAssembler_74fb923cd0b6/metadata/part-00000-f92d70a1-c40f-4703-b90f-45ad52828272-c000.txt": "87be86c491ff69af50fcde183eeca14f496880cf07c4e51c58f54d526bf41e0d",\n            "stages/2_RandomForestClassifier_98377622b0a4/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/2_RandomForestClassifier_98377622b0a4/data/part-00000-54695e2b-40a2-4d19-8359-87a4f76599e5-c000.snappy.parquet": "49f34e3c812c007226aeabf69eca1c4c8ea94eb49ecbcdeac5030cc2ba50365d",\n            "stages/2_RandomForestClassifier_98377622b0a4/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/2_RandomForestClassifier_98377622b0a4/metadata/part-00000-4f7eabc6-63f5-4786-94bf-0de646ae4dac-c000.txt": "89a32402b2207e53d6aa590d23e5f6cfae9a78254603f5788f7380dc1d603e5f",\n            "stages/2_RandomForestClassifier_98377622b0a4/treesMetadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/2_RandomForestClassifier_98377622b0a4/treesMetadata/part-00000-e6ba7265-32a3-4c98-a108-f38ba02146c2-c000.snappy.parquet": "f059e6874d66606dbfb53557ade88e8d088050bde1890daf40a73204ead7c197"\n          },\n          "training_cohort": "group_sample",\n          "training_seconds": 11.737,\n          "parameters": {\n            "numTrees": 20,\n            "maxDepth": 7,\n            "minInstancesPerNode": 30\n          },\n          "explanation": [\n            {\n              "feature": "fwd_iat_mean",\n              "method": "impurity_split_importance",\n              "value": 0.1485828303115913\n            },\n            {\n              "feature": "flow_pkts_s",\n              "method": "impurity_split_importance",\n              "value": 0.12844340395588738\n            },\n            {\n              "feature": "totlen_fwd_pkts",\n              "method": "impurity_split_importance",\n              "value": 0.1280578502643492\n            },\n            {\n              "feature": "flow_iat_mean",\n              "method": "impurity_split_importance",\n              "value": 0.0847239780596514\n            },\n            {\n              "feature": "flow_duration",\n              "method": "impurity_split_importance",\n              "value": 0.08105967665769179\n            },\n            {\n              "feature": "tot_bwd_pkts",\n              "method": "impurity_split_importance",\n              "value": 0.07936652024408121\n            },\n            {\n              "feature": "pkt_len_mean",\n              "method": "impurity_split_importance",\n              "value": 0.0696051952123036\n            },\n            {\n              "feature": "flow_byts_s",\n              "method": "impurity_split_importance",\n              "value": 0.05286345774391453\n            },\n            {\n              "feature": "tot_fwd_pkts",\n              "method": "impurity_split_importance",\n              "value": 0.05121029536467747\n            },\n            {\n              "feature": "totlen_bwd_pkts",\n              "method": "impurity_split_importance",\n              "value": 0.0465217396021467\n            },\n            {\n              "feature": "pkt_len_std",\n              "method": "impurity_split_importance",\n              "value": 0.03721990062869542\n            },\n            {\n              "feature": "ack_flag_cnt",\n              "method": "impurity_split_importance",\n              "value": 0.035757636894877015\n            },\n            {\n              "feature": "flow_iat_std",\n              "method": "impurity_split_importance",\n              "value": 0.02376530816764866\n            },\n            {\n              "feature": "rst_flag_cnt",\n              "method": "impurity_split_importance",\n              "value": 0.0162936756054958\n            },\n            {\n              "feature": "down_up_ratio",\n              "method": "impurity_split_importance",\n              "value": 0.007973356728062384\n            },\n            {\n              "feature": "bwd_iat_mean",\n              "method": "impurity_split_importance",\n              "value": 0.007940022192018519\n            },\n            {\n              "feature": "syn_flag_cnt",\n              "method": "impurity_split_importance",\n              "value": 0.0003883854687734745\n            },\n            {\n              "feature": "fin_flag_cnt",\n              "method": "impurity_split_importance",\n              "value": 0.0002267668981342781\n            }\n          ]\n        },\n        {\n          "path": "/kaggle/working/mmds_phase4/models/gradient_boosted_trees_group_sample",\n          "model": "gradient_boosted_trees",\n          "reload_probe_rows": 695307,\n          "model_bytes": 31043,\n          "model_files": 14,\n          "sha256": {\n            "metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "metadata/part-00000-9997ea4b-620c-4725-ab56-fb60547bb9ca-c000.txt": "3a8db28bac890f120b68ac7c817836f4e782d8d0ea0fa29d475b76e13fc0b340",\n            "stages/0_Imputer_abfd4cd929fc/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/0_Imputer_abfd4cd929fc/data/part-00000-9f0438c1-df56-44c6-80be-c7c3e8053633-c000.snappy.parquet": "6782881f601f55774f6a3a328f36b8e67f1a4c597e029385bf75967133da6c93",\n            "stages/0_Imputer_abfd4cd929fc/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/0_Imputer_abfd4cd929fc/metadata/part-00000-230b1995-5c8d-46e5-be65-f2d1c5a6ad5f-c000.txt": "a74ab818a4d5303c5ac0f500ea9e51e7953458a8a8bb5f660892f7fc63718798",\n            "stages/1_VectorAssembler_74fb923cd0b6/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/1_VectorAssembler_74fb923cd0b6/metadata/part-00000-e9eb1da3-7ab7-4f5e-a707-9986e33ad612-c000.txt": "20fe83ecfa8949ef2e674ee7234f02eddf8b15c57a5165c400fffcef8fc82e03",\n            "stages/2_GBTClassifier_a572097ba766/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/2_GBTClassifier_a572097ba766/data/part-00000-b233c444-18d1-49f3-962e-2ea467d359cb-c000.snappy.parquet": "ecb29f883aa85e5cb7840f65d423e6052074e6e2bd9654b4e713601239fedd69",\n            "stages/2_GBTClassifier_a572097ba766/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/2_GBTClassifier_a572097ba766/metadata/part-00000-7d715cac-4f06-4492-8119-14dc1ecd8a25-c000.txt": "d5585381f22369d61863594ec2961d7fb2a2ec9ffda17c8386a848f8858138f7",\n            "stages/2_GBTClassifier_a572097ba766/treesMetadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/2_GBTClassifier_a572097ba766/treesMetadata/part-00000-1ab94296-2336-49c8-987f-c633b0b25e06-c000.snappy.parquet": "f4b6fd6dc04571ec9886d8999a5f355a09938bede8c9acff28490820d4a51bc5"\n          },\n          "training_cohort": "group_sample",\n          "training_seconds": 19.557,\n          "parameters": {\n            "maxIter": 10,\n            "maxDepth": 4,\n            "stepSize": 0.15\n          },\n          "explanation": [\n            {\n              "feature": "fwd_iat_mean",\n              "method": "impurity_split_importance",\n              "value": 0.2943983652590758\n            },\n            {\n              "feature": "totlen_fwd_pkts",\n              "method": "impurity_split_importance",\n              "value": 0.19264049041299314\n            },\n            {\n              "feature": "tot_fwd_pkts",\n              "method": "impurity_split_importance",\n              "value": 0.09968025403414996\n            },\n            {\n              "feature": "flow_iat_mean",\n              "method": "impurity_split_importance",\n              "value": 0.08468494650972742\n            },\n            {\n              "feature": "flow_iat_std",\n              "method": "impurity_split_importance",\n              "value": 0.06280931085634042\n            },\n            {\n              "feature": "tot_bwd_pkts",\n              "method": "impurity_split_importance",\n              "value": 0.05522076067527951\n            },\n            {\n              "feature": "rst_flag_cnt",\n              "method": "impurity_split_importance",\n              "value": 0.04857612370895901\n            },\n            {\n              "feature": "flow_pkts_s",\n              "method": "impurity_split_importance",\n              "value": 0.029906156494087812\n            },\n            {\n              "feature": "ack_flag_cnt",\n              "method": "impurity_split_importance",\n              "value": 0.02867864603479734\n            },\n            {\n              "feature": "flow_duration",\n              "method": "impurity_split_importance",\n              "value": 0.02703419735576855\n            },\n            {\n              "feature": "flow_byts_s",\n              "method": "impurity_split_importance",\n              "value": 0.02153572869042957\n            },\n            {\n              "feature": "pkt_len_mean",\n              "method": "impurity_split_importance",\n              "value": 0.019431320174945952\n            },\n            {\n              "feature": "totlen_bwd_pkts",\n              "method": "impurity_split_importance",\n              "value": 0.014504644201957304\n            },\n            {\n              "feature": "pkt_len_std",\n              "method": "impurity_split_importance",\n              "value": 0.011030620287699642\n            },\n            {\n              "feature": "bwd_iat_mean",\n              "method": "impurity_split_importance",\n              "value": 0.0036449596053326314\n            },\n            {\n              "feature": "down_up_ratio",\n              "method": "impurity_split_importance",\n              "value": 0.003488000014053678\n            },\n            {\n              "feature": "syn_flag_cnt",\n              "method": "impurity_split_importance",\n              "value": 0.0023378444989426003\n            },\n            {\n              "feature": "fin_flag_cnt",\n              "method": "impurity_split_importance",\n              "value": 0.00039763118545956694\n            }\n          ]\n        },\n        {\n          "path": "/kaggle/working/mmds_phase4/models/logistic_regression_full_train_refit",\n          "model": "logistic_regression",\n          "reload_probe_rows": 695307,\n          "model_bytes": 16239,\n          "model_files": 16,\n          "sha256": {\n            "metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "metadata/part-00000-af34b176-c9ed-4a14-bc17-c4528a9db97c-c000.txt": "5f9c7fcfe76d6793e40b30320a2c6965cad9fbaf6947cfc7b4376502e2e1cbbd",\n            "stages/0_Imputer_abfd4cd929fc/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/0_Imputer_abfd4cd929fc/data/part-00000-6ece0894-f8b7-447e-b38f-0a3a6f7be9e3-c000.snappy.parquet": "6782881f601f55774f6a3a328f36b8e67f1a4c597e029385bf75967133da6c93",\n            "stages/0_Imputer_abfd4cd929fc/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/0_Imputer_abfd4cd929fc/metadata/part-00000-f5cb6293-72c8-4eda-a90b-7772f83cd552-c000.txt": "0dfe3449b12edc0e7e2265ed506be64328550439fa9f3ff05a748dfa63a8e13e",\n            "stages/1_VectorAssembler_74fb923cd0b6/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/1_VectorAssembler_74fb923cd0b6/metadata/part-00000-55025e42-4898-4e6c-84cd-cbf88d885849-c000.txt": "5f8c9591510f6db8ba2d716013d0abc429e08d51ee782afa0a23d03d198648f9",\n            "stages/2_StandardScaler_7ec078203426/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/2_StandardScaler_7ec078203426/data/part-00000-11ef9831-1439-48d8-a72d-5c655aa5981a-c000.snappy.parquet": "9c258408f8b723d8defb95ac34ec00ba12ac69d56608731095db3bba6f26fc56",\n            "stages/2_StandardScaler_7ec078203426/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/2_StandardScaler_7ec078203426/metadata/part-00000-11f18883-902c-4946-bd20-6f4256506f0f-c000.txt": "a08f37a7cf9420b3b9c0ab7eab3199adcd4adce78d809339286cd543c741de1e",\n            "stages/3_LogisticRegression_1d57b334e6b3/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/3_LogisticRegression_1d57b334e6b3/data/part-00000-4b28ecbb-85e7-4326-9f97-2e17c9903b37-c000.snappy.parquet": "a00af7e5250ccffc4308c60bf19f7ff14e6b7bdc51700c556b80ca919a2187a8",\n            "stages/3_LogisticRegression_1d57b334e6b3/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/3_LogisticRegression_1d57b334e6b3/metadata/part-00000-d5ee61ca-ea00-4da9-b0e0-16a68b4a113b-c000.txt": "96d230188f28f3dc821df7a0551e4c67015f676638a31104f8501b9626f2f008"\n          },\n          "training_cohort": "full_train_refit",\n          "training_seconds": 25.26,\n          "parameters": {\n            "maxIter": 12,\n            "regParam": 0.05,\n            "elasticNetParam": 0.0\n          },\n          "explanation": [\n            {\n              "feature": "ack_flag_cnt",\n              "method": "scaled_lr_coefficient",\n              "value": 0.47266925598336645\n            },\n            {\n              "feature": "flow_pkts_s",\n              "method": "scaled_lr_coefficient",\n              "value": 0.46701478324029866\n            },\n            {\n              "feature": "pkt_len_mean",\n              "method": "scaled_lr_coefficient",\n              "value": -0.40680489960034694\n            },\n            {\n              "feature": "rst_flag_cnt",\n              "method": "scaled_lr_coefficient",\n              "value": 0.3730107928819338\n            },\n            {\n              "feature": "syn_flag_cnt",\n              "method": "scaled_lr_coefficient",\n              "value": -0.29921406286165514\n            },\n            {\n              "feature": "flow_duration",\n              "method": "scaled_lr_coefficient",\n              "value": -0.24159368747030682\n            },\n            {\n              "feature": "flow_byts_s",\n              "method": "scaled_lr_coefficient",\n              "value": -0.23808594939360386\n            },\n            {\n              "feature": "fin_flag_cnt",\n              "method": "scaled_lr_coefficient",\n              "value": -0.10447304907465696\n            },\n            {\n              "feature": "fwd_iat_mean",\n              "method": "scaled_lr_coefficient",\n              "value": -0.040445744628794016\n            },\n            {\n              "feature": "pkt_len_std",\n              "method": "scaled_lr_coefficient",\n              "value": -0.03633762510402606\n            },\n            {\n              "feature": "tot_fwd_pkts",\n              "method": "scaled_lr_coefficient",\n              "value": 0.03439898174550569\n            },\n            {\n              "feature": "flow_iat_mean",\n              "method": "scaled_lr_coefficient",\n              "value": -0.032870148850709296\n            },\n            {\n              "feature": "down_up_ratio",\n              "method": "scaled_lr_coefficient",\n              "value": 0.027685986407391772\n            },\n            {\n              "feature": "totlen_fwd_pkts",\n              "method": "scaled_lr_coefficient",\n              "value": 0.023698087067449074\n            },\n            {\n              "feature": "bwd_iat_mean",\n              "method": "scaled_lr_coefficient",\n              "value": -0.018089822328427284\n            },\n            {\n              "feature": "tot_bwd_pkts",\n              "method": "scaled_lr_coefficient",\n              "value": -0.007690750197213877\n            },\n            {\n              "feature": "totlen_bwd_pkts",\n              "method": "scaled_lr_coefficient",\n              "value": 0.0006219430260135988\n            },\n            {\n              "feature": "flow_iat_std",\n              "method": "scaled_lr_coefficient",\n              "value": -0.0002691260185619522\n            }\n          ]\n        }\n      ],\n      "trial_selections": {\n        "logistic_regression": {\n          "trial": 2,\n          "parameters": {\n            "maxIter": 12,\n            "regParam": 0.05,\n            "elasticNetParam": 0.0\n          }\n        },\n        "random_forest": {\n          "trial": 2,\n          "parameters": {\n            "numTrees": 20,\n            "maxDepth": 7,\n            "minInstancesPerNode": 30\n          }\n        },\n        "gradient_boosted_trees": {\n          "trial": 2,\n          "parameters": {\n            "maxIter": 10,\n            "maxDepth": 4,\n            "stepSize": 0.15\n          }\n        }\n      },\n      "input": {\n        "path": "/kaggle/input/notebooks/ramcharan2345/mmds-phase-2-spark-data-engineering/mmds_phase2/phase2-v1/feature_ready",\n        "days": [\n          "2018-02-14",\n          "2018-02-15",\n          "2018-02-16",\n          "2018-02-20",\n          "2018-02-21",\n          "2018-02-22",\n          "2018-02-23",\n          "2018-02-28",\n          "2018-03-01",\n          "2018-03-02"\n        ],\n        "totals": {\n          "0": 13484708,\n          "1": 2748235\n        },\n        "rows": 16232943,\n        "temporally_flagged": 14,\n        "policy_version": "phase2-v1",\n        "receipt_sha256": "1fa00cd2b9c853ee33eaacee8ca7632dbb41f739573326280d6ffbfb2ccff25a",\n        "producer": {\n          "url": "https://www.kaggle.com/code/ramcharan2345/mmds-phase-2-spark-data-engineering",\n          "kernel_version": 1,\n          "status": "COMPLETE"\n        }\n      },\n      "registered_predictors": [\n        "flow_duration",\n        "tot_fwd_pkts",\n        "tot_bwd_pkts",\n        "totlen_fwd_pkts",\n        "totlen_bwd_pkts",\n        "pkt_len_mean",\n        "pkt_len_std",\n        "flow_iat_mean",\n        "flow_iat_std",\n        "fwd_iat_mean",\n        "bwd_iat_mean",\n        "flow_pkts_s",\n        "flow_byts_s",\n        "down_up_ratio",\n        "syn_flag_cnt",\n        "ack_flag_cnt",\n        "rst_flag_cnt",\n        "fin_flag_cnt"\n      ],\n      "overlap": {\n        "predictor_before": {\n          "groups": 0,\n          "rows": 0\n        },\n        "after": {\n          "predictor_key": {\n            "groups": 0,\n            "rows": 0\n          },\n          "record_fingerprint": {\n            "groups": 0,\n            "rows": 0\n          },\n          "split_feature_key": {\n            "groups": 0,\n            "rows": 0\n          }\n        },\n        "purged_rows": 0\n      },\n      "retained_predictor_conflicts": {\n        "groups": 25011,\n        "rows": 1792767\n      },\n      "split_support": {\n        "train": {\n          "labels": {\n            "0": 9426379,\n            "1": 2029546\n          },\n          "families": {\n            "Benign": 9426379,\n            "Credential brute force": 341040,\n            "DoS": 479941,\n            "DDoS": 885772,\n            "Web attack": 634,\n            "Infiltration": 116242,\n            "Bot": 205917\n          },\n          "original_labels": {\n            "Benign": 9426379,\n            "FTP-BruteForce": 183538,\n            "SSH-Bruteforce": 157502,\n            "DoS attacks-GoldenEye": 29251,\n            "DoS attacks-Slowloris": 7475,\n            "DoS attacks-Hulk": 323298,\n            "DoS attacks-SlowHTTPTest": 119917,\n            "DDoS attacks-LOIC-HTTP": 403700,\n            "DDOS attack-HOIC": 480857,\n            "DDOS attack-LOIC-UDP": 1215,\n            "Brute Force -Web": 411,\n            "Brute Force -XSS": 163,\n            "SQL Injection": 60,\n            "Infilteration": 116242,\n            "Bot": 205917\n          },\n          "rows": 11455925\n        },\n        "validation": {\n          "labels": {\n            "0": 1990157,\n            "1": 354569\n          },\n          "families": {\n            "Benign": 1990157,\n            "Credential brute force": 17801,\n            "DoS": 85648,\n            "DDoS": 189228,\n            "Web attack": 140,\n            "Infiltration": 21285,\n            "Bot": 40467\n          },\n          "original_labels": {\n            "Benign": 1990157,\n            "FTP-BruteForce": 2697,\n            "SSH-Bruteforce": 15104,\n            "DoS attacks-GoldenEye": 6145,\n            "DoS attacks-Slowloris": 1933,\n            "DoS attacks-Hulk": 70165,\n            "DoS attacks-SlowHTTPTest": 7405,\n            "DDoS attacks-LOIC-HTTP": 86510,\n            "DDOS attack-HOIC": 102441,\n            "DDOS attack-LOIC-UDP": 277,\n            "Brute Force -Web": 107,\n            "Brute Force -XSS": 25,\n            "SQL Injection": 8,\n            "Infilteration": 21285,\n            "Bot": 40467\n          },\n          "rows": 2344726\n        },\n        "test": {\n          "labels": {\n            "0": 2068172,\n            "1": 364120\n          },\n          "families": {\n            "Benign": 2068172,\n            "Credential brute force": 22108,\n            "DoS": 88711,\n            "DDoS": 188933,\n            "Web attack": 154,\n            "Infiltration": 24407,\n            "Bot": 39807\n          },\n          "original_labels": {\n            "Benign": 2068172,\n            "FTP-BruteForce": 7125,\n            "SSH-Bruteforce": 14983,\n            "DoS attacks-GoldenEye": 6112,\n            "DoS attacks-Slowloris": 1582,\n            "DoS attacks-Hulk": 68449,\n            "DoS attacks-SlowHTTPTest": 12568,\n            "DDoS attacks-LOIC-HTTP": 85981,\n            "DDOS attack-HOIC": 102714,\n            "DDOS attack-LOIC-UDP": 238,\n            "Brute Force -Web": 93,\n            "Brute Force -XSS": 42,\n            "SQL Injection": 19,\n            "Infilteration": 24407,\n            "Bot": 39807\n          },\n          "rows": 2432292\n        }\n      },\n      "train_sample_support": {\n        "labels": {\n          "0": 591111,\n          "1": 104196\n        },\n        "families": {\n          "Benign": 591111,\n          "Credential brute force": 4989,\n          "DoS": 22380,\n          "DDoS": 56636,\n          "Web attack": 44,\n          "Infiltration": 6207,\n          "Bot": 13940\n        },\n        "original_labels": {\n          "Benign": 591111,\n          "FTP-BruteForce": 806,\n          "SSH-Bruteforce": 4183,\n          "DoS attacks-GoldenEye": 1878,\n          "DoS attacks-Slowloris": 472,\n          "DoS attacks-Hulk": 19884,\n          "DoS attacks-SlowHTTPTest": 146,\n          "DDoS attacks-LOIC-HTTP": 25296,\n          "DDOS attack-HOIC": 31263,\n          "DDOS attack-LOIC-UDP": 77,\n          "Brute Force -Web": 27,\n          "Brute Force -XSS": 12,\n          "SQL Injection": 5,\n          "Infilteration": 6207,\n          "Bot": 13940\n        },\n        "rows": 695307\n      },\n      "baselines": {\n        "majority": {\n          "attack_prediction": 0\n        },\n        "duration_rule": {\n          "benign_training_median_us": 34807.0\n        }\n      },\n      "preprocessing": {\n        "active": [\n          "flow_duration",\n          "tot_fwd_pkts",\n          "tot_bwd_pkts",\n          "totlen_fwd_pkts",\n          "totlen_bwd_pkts",\n          "pkt_len_mean",\n          "pkt_len_std",\n          "flow_iat_mean",\n          "flow_iat_std",\n          "fwd_iat_mean",\n          "bwd_iat_mean",\n          "flow_pkts_s",\n          "flow_byts_s",\n          "down_up_ratio",\n          "syn_flag_cnt",\n          "ack_flag_cnt",\n          "rst_flag_cnt",\n          "fin_flag_cnt"\n        ],\n        "dropped_train_constant_or_empty": [],\n        "training_medians": {\n          "flow_duration": 24873.0,\n          "tot_fwd_pkts": 2.0,\n          "tot_bwd_pkts": 1.0,\n          "totlen_fwd_pkts": 44.0,\n          "totlen_bwd_pkts": 104.0,\n          "pkt_len_mean": 53.6666666667,\n          "pkt_len_std": 33.9587985653203,\n          "flow_iat_mean": 12681.0,\n          "flow_iat_std": 90.86785276799858,\n          "fwd_iat_mean": 4816.0,\n          "bwd_iat_mean": 0.0,\n          "flow_pkts_s": 126.1438957822,\n          "flow_byts_s": 979.496885506,\n          "down_up_ratio": 0.0,\n          "syn_flag_cnt": 0.0,\n          "ack_flag_cnt": 0.0,\n          "rst_flag_cnt": 0.0,\n          "fin_flag_cnt": 0.0\n        }\n      },\n      "preprocess_seconds": 14.6,\n      "post_imputation_overlap": {\n        "groups": 0,\n        "rows": 0\n      },\n      "selection_frozen": true,\n      "elapsed_seconds": 517.592,\n      "downloaded_reload": [\n        {\n          "artifact_directory": "logistic_regression_group_sample",\n          "finite_probe_predictions": 3\n        },\n        {\n          "artifact_directory": "random_forest_group_sample",\n          "finite_probe_predictions": 3\n        },\n        {\n          "artifact_directory": "gradient_boosted_trees_group_sample",\n          "finite_probe_predictions": 3\n        },\n        {\n          "artifact_directory": "logistic_regression_full_train_refit",\n          "finite_probe_predictions": 3\n        }\n      ]\n    },\n    "B": {\n      "version": "phase4-v1",\n      "status": "validated",\n      "log_events": 1091,\n      "protocol": "B",\n      "design": {\n        "name": "chronological capture-day holdout",\n        "train": [\n          "2018-02-14",\n          "2018-02-15",\n          "2018-02-16",\n          "2018-02-20",\n          "2018-02-21",\n          "2018-02-22"\n        ],\n        "validation": [\n          "2018-02-23",\n          "2018-02-28"\n        ],\n        "test": [\n          "2018-03-01",\n          "2018-03-02"\n        ]\n      },\n      "resources": {\n        "spark_version": "4.3.0",\n        "master": "local[4]",\n        "driver_memory": "8g",\n        "shuffle_partitions": 48,\n        "input_partition_bytes": 67108864,\n        "default_parallelism": 4,\n        "cpu_affinity": 4,\n        "scratch_output_free_bytes_start": 20940279808,\n        "memory.max": "32212254720",\n        "cpu.max": "400000 100000",\n        "driver_peak_rss_bytes": 6735474688\n      },\n      "models": [\n        {\n          "path": "/kaggle/working/mmds_phase4/models/logistic_regression_group_sample",\n          "model": "logistic_regression",\n          "reload_probe_rows": 795270,\n          "model_bytes": 16241,\n          "model_files": 16,\n          "sha256": {\n            "metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "metadata/part-00000-d09704e3-8b08-46c6-a97c-9d2b4c0bd829-c000.txt": "b222cf21e85042226cad2bab56a4e300e57ec59719dee464ed21214056369120",\n            "stages/0_Imputer_7052b64974ed/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/0_Imputer_7052b64974ed/data/part-00000-36cc507f-5d3c-46f9-8d2f-5363bc28c598-c000.snappy.parquet": "22f8f563ed7937c121a2add4a56ab773e344bff06b77e54a2a1c916067da552e",\n            "stages/0_Imputer_7052b64974ed/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/0_Imputer_7052b64974ed/metadata/part-00000-381bd52c-4e68-47b0-9d90-698c937d24d6-c000.txt": "8920fa2bcbe8905239fce1612d2adc7edd9350ba3b5b175cf5f3808938a16c56",\n            "stages/1_VectorAssembler_1d71e7b472ad/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/1_VectorAssembler_1d71e7b472ad/metadata/part-00000-d045abaa-13f9-420c-b4ed-be77d4ebaec3-c000.txt": "315c19d9becbaa11fd72a95a79320cf57a5d7b17a42117f2aa6847388d741e3a",\n            "stages/2_StandardScaler_aa30bbdb44b9/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/2_StandardScaler_aa30bbdb44b9/data/part-00000-2a7c52a9-2af2-43a3-bb68-26ffa3b26989-c000.snappy.parquet": "1e06bec0086ac57a353dca45e6703337e8d0c598cce0aa0d505a2846821723b2",\n            "stages/2_StandardScaler_aa30bbdb44b9/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/2_StandardScaler_aa30bbdb44b9/metadata/part-00000-31c7267e-71ab-4d7d-ab32-c5618d2dbac5-c000.txt": "cbd9564b9c94ddc4f2e0452c84b149d59ece9b84805a33780d0b9b1cfa8e6f22",\n            "stages/3_LogisticRegression_93ba6b170c64/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/3_LogisticRegression_93ba6b170c64/data/part-00000-eca9e110-62c5-469b-a24c-54d7e36f72fb-c000.snappy.parquet": "b2cb257a862d1dc1c25a1cc6a117f63f3501cae8bc3d20d6a6978b5e706f1940",\n            "stages/3_LogisticRegression_93ba6b170c64/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/3_LogisticRegression_93ba6b170c64/metadata/part-00000-ad94d93b-661f-418a-9b58-e7e85b634947-c000.txt": "6e46ad499a82c57e9c52715d65c9c14379d68068a77dcfe5187bd8445d95daed"\n          },\n          "training_cohort": "group_sample",\n          "training_seconds": 13.5,\n          "parameters": {\n            "maxIter": 12,\n            "regParam": 0.005,\n            "elasticNetParam": 0.0\n          },\n          "explanation": [\n            {\n              "feature": "pkt_len_mean",\n              "method": "scaled_lr_coefficient",\n              "value": -1.780591224500976\n            },\n            {\n              "feature": "flow_byts_s",\n              "method": "scaled_lr_coefficient",\n              "value": -1.112350263625203\n            },\n            {\n              "feature": "pkt_len_std",\n              "method": "scaled_lr_coefficient",\n              "value": 0.923384841756616\n            },\n            {\n              "feature": "flow_duration",\n              "method": "scaled_lr_coefficient",\n              "value": -0.8852168829523376\n            },\n            {\n              "feature": "ack_flag_cnt",\n              "method": "scaled_lr_coefficient",\n              "value": 0.7615032677887672\n            },\n            {\n              "feature": "syn_flag_cnt",\n              "method": "scaled_lr_coefficient",\n              "value": -0.7260335062130642\n            },\n            {\n              "feature": "flow_pkts_s",\n              "method": "scaled_lr_coefficient",\n              "value": 0.6303179128463633\n            },\n            {\n              "feature": "rst_flag_cnt",\n              "method": "scaled_lr_coefficient",\n              "value": 0.5938143723566249\n            },\n            {\n              "feature": "bwd_iat_mean",\n              "method": "scaled_lr_coefficient",\n              "value": 0.42173284043500003\n            },\n            {\n              "feature": "flow_iat_std",\n              "method": "scaled_lr_coefficient",\n              "value": -0.38031640864136457\n            },\n            {\n              "feature": "fin_flag_cnt",\n              "method": "scaled_lr_coefficient",\n              "value": -0.11848661645584345\n            },\n            {\n              "feature": "tot_fwd_pkts",\n              "method": "scaled_lr_coefficient",\n              "value": 0.10991900173640909\n            },\n            {\n              "feature": "fwd_iat_mean",\n              "method": "scaled_lr_coefficient",\n              "value": 0.09485773269568075\n            },\n            {\n              "feature": "flow_iat_mean",\n              "method": "scaled_lr_coefficient",\n              "value": 0.09328298989681266\n            },\n            {\n              "feature": "totlen_fwd_pkts",\n              "method": "scaled_lr_coefficient",\n              "value": 0.06828791267348731\n            },\n            {\n              "feature": "down_up_ratio",\n              "method": "scaled_lr_coefficient",\n              "value": 0.035025609080342794\n            },\n            {\n              "feature": "tot_bwd_pkts",\n              "method": "scaled_lr_coefficient",\n              "value": -0.01209622751847114\n            },\n            {\n              "feature": "totlen_bwd_pkts",\n              "method": "scaled_lr_coefficient",\n              "value": 0.0020208612240548065\n            }\n          ]\n        },\n        {\n          "path": "/kaggle/working/mmds_phase4/models/random_forest_group_sample",\n          "model": "random_forest",\n          "reload_probe_rows": 795270,\n          "model_bytes": 62923,\n          "model_files": 14,\n          "sha256": {\n            "metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "metadata/part-00000-e921fc7c-54c9-4406-b75c-5897681c5ac6-c000.txt": "b781e9c5b3ddcc480bfd4bf9375ded0c7a68f2f62be6752befc81a1c60b93f14",\n            "stages/0_Imputer_7052b64974ed/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/0_Imputer_7052b64974ed/data/part-00000-58e83166-5de1-41a4-91fb-df039444c268-c000.snappy.parquet": "22f8f563ed7937c121a2add4a56ab773e344bff06b77e54a2a1c916067da552e",\n            "stages/0_Imputer_7052b64974ed/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/0_Imputer_7052b64974ed/metadata/part-00000-5ef38d51-4df2-4030-a68a-7e20ea6dfebc-c000.txt": "3dcf615dc6ead2f20284250a3d292977eae18d866bacaf1aa147293fe3302494",\n            "stages/1_VectorAssembler_1d71e7b472ad/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/1_VectorAssembler_1d71e7b472ad/metadata/part-00000-a1a7f9ec-5f45-4366-8c33-6e3acb58b507-c000.txt": "08aa06a7f9a6c6a72fabbba50f8dd3d9edf824fde87adc41811e9d533fe03976",\n            "stages/2_RandomForestClassifier_e65672fa318a/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/2_RandomForestClassifier_e65672fa318a/data/part-00000-8cf4c00d-8eeb-4f4e-8415-17f0daa8614f-c000.snappy.parquet": "f2446f6db6876001a19d9a3030d77d7218d82d9d8c8678fab71feb696ebb2988",\n            "stages/2_RandomForestClassifier_e65672fa318a/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/2_RandomForestClassifier_e65672fa318a/metadata/part-00000-22903d16-7742-4297-a8b6-f94998acfbd2-c000.txt": "9a9f25c21aae0ba3d2959cd946f89a3c58b3ce88543f0fe7627036498609d7dd",\n            "stages/2_RandomForestClassifier_e65672fa318a/treesMetadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/2_RandomForestClassifier_e65672fa318a/treesMetadata/part-00000-5d0f9c54-e9e2-4e49-9430-8fee7f1ab794-c000.snappy.parquet": "111a1b821b5a16165f68182cc327be978b75e07b03aaa27ca64f146054838b01"\n          },\n          "training_cohort": "group_sample",\n          "training_seconds": 19.239,\n          "parameters": {\n            "numTrees": 20,\n            "maxDepth": 7,\n            "minInstancesPerNode": 30\n          },\n          "explanation": [\n            {\n              "feature": "totlen_fwd_pkts",\n              "method": "impurity_split_importance",\n              "value": 0.15005228174630172\n            },\n            {\n              "feature": "ack_flag_cnt",\n              "method": "impurity_split_importance",\n              "value": 0.09894914887463935\n            },\n            {\n              "feature": "fwd_iat_mean",\n              "method": "impurity_split_importance",\n              "value": 0.09744757969988256\n            },\n            {\n              "feature": "flow_pkts_s",\n              "method": "impurity_split_importance",\n              "value": 0.08211103799790603\n            },\n            {\n              "feature": "tot_bwd_pkts",\n              "method": "impurity_split_importance",\n              "value": 0.08190782732459835\n            },\n            {\n              "feature": "flow_duration",\n              "method": "impurity_split_importance",\n              "value": 0.07810701388066098\n            },\n            {\n              "feature": "flow_iat_mean",\n              "method": "impurity_split_importance",\n              "value": 0.0662482749594117\n            },\n            {\n              "feature": "totlen_bwd_pkts",\n              "method": "impurity_split_importance",\n              "value": 0.06442200035495507\n            },\n            {\n              "feature": "tot_fwd_pkts",\n              "method": "impurity_split_importance",\n              "value": 0.06201574725367744\n            },\n            {\n              "feature": "flow_byts_s",\n              "method": "impurity_split_importance",\n              "value": 0.061839212527291465\n            },\n            {\n              "feature": "pkt_len_mean",\n              "method": "impurity_split_importance",\n              "value": 0.06049114563830391\n            },\n            {\n              "feature": "pkt_len_std",\n              "method": "impurity_split_importance",\n              "value": 0.059348765425908626\n            },\n            {\n              "feature": "flow_iat_std",\n              "method": "impurity_split_importance",\n              "value": 0.011960422418420463\n            },\n            {\n              "feature": "down_up_ratio",\n              "method": "impurity_split_importance",\n              "value": 0.011442614078587212\n            },\n            {\n              "feature": "rst_flag_cnt",\n              "method": "impurity_split_importance",\n              "value": 0.00955825948925279\n            },\n            {\n              "feature": "bwd_iat_mean",\n              "method": "impurity_split_importance",\n              "value": 0.0028809813214295205\n            },\n            {\n              "feature": "fin_flag_cnt",\n              "method": "impurity_split_importance",\n              "value": 0.0006229749712901726\n            },\n            {\n              "feature": "syn_flag_cnt",\n              "method": "impurity_split_importance",\n              "value": 0.0005947120374825553\n            }\n          ]\n        },\n        {\n          "path": "/kaggle/working/mmds_phase4/models/gradient_boosted_trees_group_sample",\n          "model": "gradient_boosted_trees",\n          "reload_probe_rows": 795270,\n          "model_bytes": 20054,\n          "model_files": 14,\n          "sha256": {\n            "metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "metadata/part-00000-efbdecf1-4803-45db-ae68-abe98da0e199-c000.txt": "4123305ce71538ad80e064db257b9f735fcec77b5eabc50cc023cf1fde3f1b80",\n            "stages/0_Imputer_7052b64974ed/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/0_Imputer_7052b64974ed/data/part-00000-8adbc5fc-0c93-42fa-b346-dd9476f83a90-c000.snappy.parquet": "22f8f563ed7937c121a2add4a56ab773e344bff06b77e54a2a1c916067da552e",\n            "stages/0_Imputer_7052b64974ed/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/0_Imputer_7052b64974ed/metadata/part-00000-b6cab653-d0fd-4c77-aca7-aab3cfc2b275-c000.txt": "d03fd702a9c4695d1d13ca1b594cb8fa037f5ae5e9ff6f730a4d411bdb2f05f6",\n            "stages/1_VectorAssembler_1d71e7b472ad/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/1_VectorAssembler_1d71e7b472ad/metadata/part-00000-32eec798-d9d3-45b1-9045-7d5ebc915551-c000.txt": "ede986a40e1120df8143417f1d79f5f295da93aef163afaec6eb614392470d48",\n            "stages/2_GBTClassifier_ed0a682fa045/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/2_GBTClassifier_ed0a682fa045/data/part-00000-d216835b-7635-4173-8e6e-c09c9b700350-c000.snappy.parquet": "481f7cf4fa2ee3a69a170fe2752ecf03f2f2ba20a86f2cde92d0540d2b30e60e",\n            "stages/2_GBTClassifier_ed0a682fa045/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/2_GBTClassifier_ed0a682fa045/metadata/part-00000-140fbea1-593b-4019-9aeb-0b01b674de20-c000.txt": "3049f6423197b1b3a40e8355ba44fc77c826bc2426ab0c2ede9eeb537c64eb98",\n            "stages/2_GBTClassifier_ed0a682fa045/treesMetadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/2_GBTClassifier_ed0a682fa045/treesMetadata/part-00000-2fee5052-d4dd-4dbf-9da0-ba29400facaa-c000.snappy.parquet": "7603b2afadd15d52949931d502b65ea74d4637aa058f3bcd4616ade79fc78234"\n          },\n          "training_cohort": "group_sample",\n          "training_seconds": 17.415,\n          "parameters": {\n            "maxIter": 6,\n            "maxDepth": 3,\n            "stepSize": 0.1\n          },\n          "explanation": [\n            {\n              "feature": "tot_bwd_pkts",\n              "method": "impurity_split_importance",\n              "value": 0.2339302773742976\n            },\n            {\n              "feature": "ack_flag_cnt",\n              "method": "impurity_split_importance",\n              "value": 0.21187176671645613\n            },\n            {\n              "feature": "totlen_fwd_pkts",\n              "method": "impurity_split_importance",\n              "value": 0.15919712437755826\n            },\n            {\n              "feature": "fwd_iat_mean",\n              "method": "impurity_split_importance",\n              "value": 0.11143762140075865\n            },\n            {\n              "feature": "flow_duration",\n              "method": "impurity_split_importance",\n              "value": 0.10278018282023525\n            },\n            {\n              "feature": "flow_iat_mean",\n              "method": "impurity_split_importance",\n              "value": 0.08532863814951952\n            },\n            {\n              "feature": "flow_iat_std",\n              "method": "impurity_split_importance",\n              "value": 0.03942685523415636\n            },\n            {\n              "feature": "totlen_bwd_pkts",\n              "method": "impurity_split_importance",\n              "value": 0.027452293578463044\n            },\n            {\n              "feature": "pkt_len_mean",\n              "method": "impurity_split_importance",\n              "value": 0.018163734698595824\n            },\n            {\n              "feature": "pkt_len_std",\n              "method": "impurity_split_importance",\n              "value": 0.0053939156650885615\n            },\n            {\n              "feature": "tot_fwd_pkts",\n              "method": "impurity_split_importance",\n              "value": 0.005017589984870909\n            },\n            {\n              "feature": "bwd_iat_mean",\n              "method": "impurity_split_importance",\n              "value": 0.0\n            },\n            {\n              "feature": "down_up_ratio",\n              "method": "impurity_split_importance",\n              "value": 0.0\n            },\n            {\n              "feature": "fin_flag_cnt",\n              "method": "impurity_split_importance",\n              "value": 0.0\n            },\n            {\n              "feature": "flow_byts_s",\n              "method": "impurity_split_importance",\n              "value": 0.0\n            },\n            {\n              "feature": "flow_pkts_s",\n              "method": "impurity_split_importance",\n              "value": 0.0\n            },\n            {\n              "feature": "rst_flag_cnt",\n              "method": "impurity_split_importance",\n              "value": 0.0\n            },\n            {\n              "feature": "syn_flag_cnt",\n              "method": "impurity_split_importance",\n              "value": 0.0\n            }\n          ]\n        },\n        {\n          "path": "/kaggle/working/mmds_phase4/models/logistic_regression_full_train_refit",\n          "model": "logistic_regression",\n          "reload_probe_rows": 795270,\n          "model_bytes": 16241,\n          "model_files": 16,\n          "sha256": {\n            "metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "metadata/part-00000-0e78daa9-def3-46bd-8d5f-5607608f4ecb-c000.txt": "c82e755e8e0ae81bb51041d109e4101692b6a30bb7821a0463317d0e038e2a3c",\n            "stages/0_Imputer_7052b64974ed/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/0_Imputer_7052b64974ed/data/part-00000-4c23c8bb-1bf5-44f5-87d0-b52d0d40765c-c000.snappy.parquet": "22f8f563ed7937c121a2add4a56ab773e344bff06b77e54a2a1c916067da552e",\n            "stages/0_Imputer_7052b64974ed/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/0_Imputer_7052b64974ed/metadata/part-00000-8746ef4d-a351-4a4f-a0a0-59d4f4ad63cf-c000.txt": "aa498b839593c79905d79b3cdc9ad123863e275870570fb39e73f4cae40e2cd2",\n            "stages/1_VectorAssembler_1d71e7b472ad/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/1_VectorAssembler_1d71e7b472ad/metadata/part-00000-80ecd58f-b8a5-4270-9518-9b73b5085a67-c000.txt": "e2633b6193a8cc69fb646fde56d7fa6b4e3d5331111230c1953ebd5cadff28d6",\n            "stages/2_StandardScaler_aa30bbdb44b9/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/2_StandardScaler_aa30bbdb44b9/data/part-00000-e073eb75-12e6-4a47-a5a7-38a1b3e8cd41-c000.snappy.parquet": "1e06bec0086ac57a353dca45e6703337e8d0c598cce0aa0d505a2846821723b2",\n            "stages/2_StandardScaler_aa30bbdb44b9/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/2_StandardScaler_aa30bbdb44b9/metadata/part-00000-de360e70-a176-4c7a-a035-1f85b59a51d7-c000.txt": "16e55e02ebe10c0612e1ce1db5d2d4084b26f79c0aa6089f930a1cebf5b0af8c",\n            "stages/3_LogisticRegression_80d3fafce516/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/3_LogisticRegression_80d3fafce516/data/part-00000-a186bbfc-7c71-40fe-b226-a0e11ceddac0-c000.snappy.parquet": "3e97852a5efcad0aff181529fb2fad7b578e7d596277a0086b866f3b404b95d9",\n            "stages/3_LogisticRegression_80d3fafce516/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/3_LogisticRegression_80d3fafce516/metadata/part-00000-c7566b37-a7d4-44ba-85a0-d8d8df05e671-c000.txt": "6e3e4a6e7c88ebe2d671ac4f72e452177d73d214fecbbef5a04556c90e9822e5"\n          },\n          "training_cohort": "full_train_refit",\n          "training_seconds": 44.995,\n          "parameters": {\n            "maxIter": 12,\n            "regParam": 0.005,\n            "elasticNetParam": 0.0\n          },\n          "explanation": [\n            {\n              "feature": "pkt_len_mean",\n              "method": "scaled_lr_coefficient",\n              "value": -1.8670591581079088\n            },\n            {\n              "feature": "pkt_len_std",\n              "method": "scaled_lr_coefficient",\n              "value": 0.9680905133125925\n            },\n            {\n              "feature": "flow_duration",\n              "method": "scaled_lr_coefficient",\n              "value": -0.9397318701779404\n            },\n            {\n              "feature": "flow_byts_s",\n              "method": "scaled_lr_coefficient",\n              "value": -0.6402688432634688\n            },\n            {\n              "feature": "ack_flag_cnt",\n              "method": "scaled_lr_coefficient",\n              "value": 0.6284176725086255\n            },\n            {\n              "feature": "syn_flag_cnt",\n              "method": "scaled_lr_coefficient",\n              "value": -0.6224668326042223\n            },\n            {\n              "feature": "rst_flag_cnt",\n              "method": "scaled_lr_coefficient",\n              "value": 0.5455725718029051\n            },\n            {\n              "feature": "flow_pkts_s",\n              "method": "scaled_lr_coefficient",\n              "value": 0.3899168903099706\n            },\n            {\n              "feature": "down_up_ratio",\n              "method": "scaled_lr_coefficient",\n              "value": 0.18137120862326153\n            },\n            {\n              "feature": "bwd_iat_mean",\n              "method": "scaled_lr_coefficient",\n              "value": 0.177248781830544\n            },\n            {\n              "feature": "fin_flag_cnt",\n              "method": "scaled_lr_coefficient",\n              "value": -0.1533909502840441\n            },\n            {\n              "feature": "flow_iat_mean",\n              "method": "scaled_lr_coefficient",\n              "value": 0.13777249981586884\n            },\n            {\n              "feature": "fwd_iat_mean",\n              "method": "scaled_lr_coefficient",\n              "value": 0.10781704655154939\n            },\n            {\n              "feature": "tot_fwd_pkts",\n              "method": "scaled_lr_coefficient",\n              "value": 0.10341793377036967\n            },\n            {\n              "feature": "totlen_fwd_pkts",\n              "method": "scaled_lr_coefficient",\n              "value": 0.061376137349828615\n            },\n            {\n              "feature": "tot_bwd_pkts",\n              "method": "scaled_lr_coefficient",\n              "value": -0.022734416750892123\n            },\n            {\n              "feature": "totlen_bwd_pkts",\n              "method": "scaled_lr_coefficient",\n              "value": 0.00469054131037264\n            },\n            {\n              "feature": "flow_iat_std",\n              "method": "scaled_lr_coefficient",\n              "value": -0.0001483209333920456\n            }\n          ]\n        }\n      ],\n      "trial_selections": {\n        "logistic_regression": {\n          "trial": 1,\n          "parameters": {\n            "maxIter": 12,\n            "regParam": 0.005,\n            "elasticNetParam": 0.0\n          }\n        },\n        "random_forest": {\n          "trial": 2,\n          "parameters": {\n            "numTrees": 20,\n            "maxDepth": 7,\n            "minInstancesPerNode": 30\n          }\n        },\n        "gradient_boosted_trees": {\n          "trial": 1,\n          "parameters": {\n            "maxIter": 6,\n            "maxDepth": 3,\n            "stepSize": 0.1\n          }\n        }\n      },\n      "input": {\n        "path": "/kaggle/input/notebooks/ramcharan2345/mmds-phase-2-spark-data-engineering/mmds_phase2/phase2-v1/feature_ready",\n        "days": [\n          "2018-02-14",\n          "2018-02-15",\n          "2018-02-16",\n          "2018-02-20",\n          "2018-02-21",\n          "2018-02-22",\n          "2018-02-23",\n          "2018-02-28",\n          "2018-03-01",\n          "2018-03-02"\n        ],\n        "totals": {\n          "0": 13484708,\n          "1": 2748235\n        },\n        "rows": 16232943,\n        "temporally_flagged": 14,\n        "policy_version": "phase2-v1",\n        "receipt_sha256": "1fa00cd2b9c853ee33eaacee8ca7632dbb41f739573326280d6ffbfb2ccff25a",\n        "producer": {\n          "url": "https://www.kaggle.com/code/ramcharan2345/mmds-phase-2-spark-data-engineering",\n          "kernel_version": 1,\n          "status": "COMPLETE"\n        }\n      },\n      "registered_predictors": [\n        "flow_duration",\n        "tot_fwd_pkts",\n        "tot_bwd_pkts",\n        "totlen_fwd_pkts",\n        "totlen_bwd_pkts",\n        "pkt_len_mean",\n        "pkt_len_std",\n        "flow_iat_mean",\n        "flow_iat_std",\n        "fwd_iat_mean",\n        "bwd_iat_mean",\n        "flow_pkts_s",\n        "flow_byts_s",\n        "down_up_ratio",\n        "syn_flag_cnt",\n        "ack_flag_cnt",\n        "rst_flag_cnt",\n        "fin_flag_cnt"\n      ],\n      "overlap": {\n        "predictor_before": {\n          "groups": 136985,\n          "rows": 2773438\n        },\n        "after": {\n          "predictor_key": {\n            "groups": 0,\n            "rows": 0\n          },\n          "record_fingerprint": {\n            "groups": 0,\n            "rows": 0\n          },\n          "split_feature_key": {\n            "groups": 0,\n            "rows": 0\n          }\n        },\n        "purged_rows": 909956\n      },\n      "retained_predictor_conflicts": {\n        "groups": 12312,\n        "rows": 784773\n      },\n      "split_support": {\n        "train": {\n          "labels": {\n            "0": 10892078,\n            "1": 2299544\n          },\n          "families": {\n            "Benign": 10892078,\n            "Credential brute force": 380949,\n            "DoS": 654300,\n            "DDoS": 1263933,\n            "Web attack": 362\n          },\n          "original_labels": {\n            "Benign": 10892078,\n            "FTP-BruteForce": 193360,\n            "SSH-Bruteforce": 187589,\n            "DoS attacks-GoldenEye": 41508,\n            "DoS attacks-Slowloris": 10990,\n            "DoS attacks-Hulk": 461912,\n            "DoS attacks-SlowHTTPTest": 139890,\n            "DDoS attacks-LOIC-HTTP": 576191,\n            "DDOS attack-HOIC": 686012,\n            "DDOS attack-LOIC-UDP": 1730,\n            "Brute Force -Web": 249,\n            "Brute Force -XSS": 79,\n            "SQL Injection": 34\n          },\n          "rows": 13191622\n        },\n        "validation": {\n          "labels": {\n            "0": 1094200,\n            "1": 57056\n          },\n          "families": {\n            "Benign": 1094200,\n            "Web attack": 423,\n            "Infiltration": 56633\n          },\n          "original_labels": {\n            "Benign": 1094200,\n            "Brute Force -Web": 276,\n            "Brute Force -XSS": 111,\n            "SQL Injection": 36,\n            "Infilteration": 56633\n          },\n          "rows": 1151256\n        },\n        "test": {\n          "labels": {\n            "0": 771584,\n            "1": 208525\n          },\n          "families": {\n            "Benign": 771584,\n            "Infiltration": 65251,\n            "Bot": 143274\n          },\n          "original_labels": {\n            "Benign": 771584,\n            "Infilteration": 65251,\n            "Bot": 143274\n          },\n          "rows": 980109\n        }\n      },\n      "train_sample_support": {\n        "labels": {\n          "0": 664936,\n          "1": 130334\n        },\n        "families": {\n          "Benign": 664936,\n          "Credential brute force": 12384,\n          "DoS": 37445,\n          "DDoS": 80477,\n          "Web attack": 28\n        },\n        "original_labels": {\n          "Benign": 664936,\n          "FTP-BruteForce": 6425,\n          "SSH-Bruteforce": 5959,\n          "DoS attacks-GoldenEye": 2663,\n          "DoS attacks-Slowloris": 684,\n          "DoS attacks-Hulk": 28198,\n          "DoS attacks-SlowHTTPTest": 5900,\n          "DDoS attacks-LOIC-HTTP": 36066,\n          "DDOS attack-HOIC": 44289,\n          "DDOS attack-LOIC-UDP": 122,\n          "Brute Force -Web": 20,\n          "Brute Force -XSS": 5,\n          "SQL Injection": 3\n        },\n        "rows": 795270\n      },\n      "baselines": {\n        "majority": {\n          "attack_prediction": 0\n        },\n        "duration_rule": {\n          "benign_training_median_us": 30201.0\n        }\n      },\n      "preprocessing": {\n        "active": [\n          "flow_duration",\n          "tot_fwd_pkts",\n          "tot_bwd_pkts",\n          "totlen_fwd_pkts",\n          "totlen_bwd_pkts",\n          "pkt_len_mean",\n          "pkt_len_std",\n          "flow_iat_mean",\n          "flow_iat_std",\n          "fwd_iat_mean",\n          "bwd_iat_mean",\n          "flow_pkts_s",\n          "flow_byts_s",\n          "down_up_ratio",\n          "syn_flag_cnt",\n          "ack_flag_cnt",\n          "rst_flag_cnt",\n          "fin_flag_cnt"\n        ],\n        "dropped_train_constant_or_empty": [],\n        "training_medians": {\n          "flow_duration": 29185.0,\n          "tot_fwd_pkts": 2.0,\n          "tot_bwd_pkts": 1.0,\n          "totlen_fwd_pkts": 44.0,\n          "totlen_bwd_pkts": 110.0,\n          "pkt_len_mean": 54.5,\n          "pkt_len_std": 35.21836642056717,\n          "flow_iat_mean": 14048.0,\n          "flow_iat_std": 106.4700270812,\n          "fwd_iat_mean": 6398.0,\n          "bwd_iat_mean": 0.0,\n          "flow_pkts_s": 112.1361945053,\n          "flow_byts_s": 948.1289043378442,\n          "down_up_ratio": 0.0,\n          "syn_flag_cnt": 0.0,\n          "ack_flag_cnt": 0.0,\n          "rst_flag_cnt": 0.0,\n          "fin_flag_cnt": 0.0\n        }\n      },\n      "preprocess_seconds": 21.572,\n      "post_imputation_overlap": {\n        "groups": 0,\n        "rows": 0\n      },\n      "selection_frozen": true,\n      "elapsed_seconds": 745.004,\n      "downloaded_reload": [\n        {\n          "artifact_directory": "logistic_regression_group_sample",\n          "finite_probe_predictions": 3\n        },\n        {\n          "artifact_directory": "random_forest_group_sample",\n          "finite_probe_predictions": 3\n        },\n        {\n          "artifact_directory": "gradient_boosted_trees_group_sample",\n          "finite_probe_predictions": 3\n        },\n        {\n          "artifact_directory": "logistic_regression_full_train_refit",\n          "finite_probe_predictions": 3\n        }\n      ]\n    },\n    "C": {\n      "version": "phase4-v1",\n      "status": "validated",\n      "log_events": 836,\n      "protocol": "C",\n      "design": {\n        "name": "web-attack capture holdout",\n        "train": [\n          "2018-02-14",\n          "2018-02-15",\n          "2018-02-16",\n          "2018-02-20",\n          "2018-02-21",\n          "2018-03-01",\n          "2018-03-02"\n        ],\n        "validation": [\n          "2018-02-28"\n        ],\n        "test": [\n          "2018-02-22",\n          "2018-02-23"\n        ]\n      },\n      "resources": {\n        "spark_version": "4.3.0",\n        "master": "local[4]",\n        "driver_memory": "8g",\n        "shuffle_partitions": 48,\n        "input_partition_bytes": 67108864,\n        "default_parallelism": 4,\n        "cpu_affinity": 4,\n        "scratch_output_free_bytes_start": 20940279808,\n        "memory.max": "32212254720",\n        "cpu.max": "400000 100000",\n        "driver_peak_rss_bytes": 7445041152\n      },\n      "models": [\n        {\n          "path": "/kaggle/working/mmds_phase4/models/logistic_regression_group_sample",\n          "model": "logistic_regression",\n          "reload_probe_rows": 812520,\n          "model_bytes": 16241,\n          "model_files": 16,\n          "sha256": {\n            "metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "metadata/part-00000-5f3aed8d-e6c0-4c41-8408-d4ebc7759bff-c000.txt": "6f5a888c299e3b95f5e67ac04ceb3c8e09256811c154ece318a45a788d0ac76e",\n            "stages/0_Imputer_75e58e12f8f2/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/0_Imputer_75e58e12f8f2/data/part-00000-568211c1-d62a-4813-87d4-721e07961390-c000.snappy.parquet": "b95f973e5c9d08382e52dde2888bfb19b4371d8eb2000e48801a085ca373c01a",\n            "stages/0_Imputer_75e58e12f8f2/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/0_Imputer_75e58e12f8f2/metadata/part-00000-a5350c72-13b1-4671-8b21-405b164fcf5f-c000.txt": "7d40b5f74f6e0dc9cafa6a2511a24c70e55e6e8a88d06433463fc5f8f37511eb",\n            "stages/1_VectorAssembler_bc29ebd64ced/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/1_VectorAssembler_bc29ebd64ced/metadata/part-00000-b23307b0-99ea-4929-ac94-98ec16ab3b92-c000.txt": "c6c93ab74095ae50bc62d2157fa3fed7842b6fb4e5b7d4b7ee30e8d6a0e7f2c1",\n            "stages/2_StandardScaler_611ae33371ff/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/2_StandardScaler_611ae33371ff/data/part-00000-7df6f915-7e7b-4704-b777-ba432f18668b-c000.snappy.parquet": "e5ca48abe1b71ba77b74e5909088ec9af7ba5e9fbe4af16dbd0c3c2b4c22031d",\n            "stages/2_StandardScaler_611ae33371ff/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/2_StandardScaler_611ae33371ff/metadata/part-00000-82921780-3bde-4b7b-8af1-38cd239d38bb-c000.txt": "8b9d547dfefa311dc6af3b9f6801b9928e7c151326509dce13baeea0ba0ed736",\n            "stages/3_LogisticRegression_acfcb9fe427d/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/3_LogisticRegression_acfcb9fe427d/data/part-00000-9d7f0f44-8be6-4a81-890c-5495f53dd9f2-c000.snappy.parquet": "21faf6cd326a4950c2d07b386fd8a8c96b1bb7f9a9d916394f09d6c03f14019b",\n            "stages/3_LogisticRegression_acfcb9fe427d/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/3_LogisticRegression_acfcb9fe427d/metadata/part-00000-e761b683-a3b3-4662-b732-5d1a3d903a77-c000.txt": "402637179e659a8a028b928e74f9499db623abf40d4256bffb6fbef1566d5f74"\n          },\n          "training_cohort": "group_sample",\n          "training_seconds": 10.976,\n          "parameters": {\n            "maxIter": 12,\n            "regParam": 0.005,\n            "elasticNetParam": 0.0\n          },\n          "explanation": [\n            {\n              "feature": "pkt_len_mean",\n              "method": "scaled_lr_coefficient",\n              "value": -1.7785150131300438\n            },\n            {\n              "feature": "flow_byts_s",\n              "method": "scaled_lr_coefficient",\n              "value": -1.0410201765929772\n            },\n            {\n              "feature": "flow_duration",\n              "method": "scaled_lr_coefficient",\n              "value": -0.8030793330807489\n            },\n            {\n              "feature": "rst_flag_cnt",\n              "method": "scaled_lr_coefficient",\n              "value": 0.7534496577589579\n            },\n            {\n              "feature": "ack_flag_cnt",\n              "method": "scaled_lr_coefficient",\n              "value": 0.7343297701813489\n            },\n            {\n              "feature": "pkt_len_std",\n              "method": "scaled_lr_coefficient",\n              "value": 0.7049722561445563\n            },\n            {\n              "feature": "syn_flag_cnt",\n              "method": "scaled_lr_coefficient",\n              "value": -0.6645100754422375\n            },\n            {\n              "feature": "flow_pkts_s",\n              "method": "scaled_lr_coefficient",\n              "value": 0.5080633310751165\n            },\n            {\n              "feature": "bwd_iat_mean",\n              "method": "scaled_lr_coefficient",\n              "value": 0.3738130260259236\n            },\n            {\n              "feature": "flow_iat_std",\n              "method": "scaled_lr_coefficient",\n              "value": -0.3134998622713633\n            },\n            {\n              "feature": "fin_flag_cnt",\n              "method": "scaled_lr_coefficient",\n              "value": -0.1173579512487568\n            },\n            {\n              "feature": "tot_fwd_pkts",\n              "method": "scaled_lr_coefficient",\n              "value": 0.10171587944469686\n            },\n            {\n              "feature": "fwd_iat_mean",\n              "method": "scaled_lr_coefficient",\n              "value": 0.07664484544466971\n            },\n            {\n              "feature": "totlen_fwd_pkts",\n              "method": "scaled_lr_coefficient",\n              "value": 0.07118048655568075\n            },\n            {\n              "feature": "flow_iat_mean",\n              "method": "scaled_lr_coefficient",\n              "value": 0.06730844086707058\n            },\n            {\n              "feature": "down_up_ratio",\n              "method": "scaled_lr_coefficient",\n              "value": 0.01772022458770494\n            },\n            {\n              "feature": "tot_bwd_pkts",\n              "method": "scaled_lr_coefficient",\n              "value": -0.01632524693025906\n            },\n            {\n              "feature": "totlen_bwd_pkts",\n              "method": "scaled_lr_coefficient",\n              "value": 0.0025806242947113618\n            }\n          ]\n        },\n        {\n          "path": "/kaggle/working/mmds_phase4/models/random_forest_group_sample",\n          "model": "random_forest",\n          "reload_probe_rows": 812520,\n          "model_bytes": 30720,\n          "model_files": 14,\n          "sha256": {\n            "metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "metadata/part-00000-e87e419d-477f-4831-880c-c3b5b4236ba1-c000.txt": "50b7f1b39629d827f55bac65a9ccf7011fe4a7085ac18e7d18978bea619d9aa8",\n            "stages/0_Imputer_75e58e12f8f2/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/0_Imputer_75e58e12f8f2/data/part-00000-088020e3-6b6d-40af-9c21-cf635b36a09a-c000.snappy.parquet": "b95f973e5c9d08382e52dde2888bfb19b4371d8eb2000e48801a085ca373c01a",\n            "stages/0_Imputer_75e58e12f8f2/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/0_Imputer_75e58e12f8f2/metadata/part-00000-4cf65d3b-775b-440f-9dad-69454a3372b6-c000.txt": "4a967ba5fcd67d0945183b3015ef2f47ccd2eb1b9702a15bed744e9ee4189111",\n            "stages/1_VectorAssembler_bc29ebd64ced/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/1_VectorAssembler_bc29ebd64ced/metadata/part-00000-2acd1782-0a6d-4bcb-a1b9-3d82e1874db5-c000.txt": "52fe7777948866b88f884557ec2800ffd48ae8b714713fed745ceb6829fc63f6",\n            "stages/2_RandomForestClassifier_2155d2c82f7d/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/2_RandomForestClassifier_2155d2c82f7d/data/part-00000-0b8e5abd-1473-449e-a5ce-3d82b78c1984-c000.snappy.parquet": "ae6f9f9fe8a564716b870ef1ab34055ad594077b76abf554719aca28bf3e8a25",\n            "stages/2_RandomForestClassifier_2155d2c82f7d/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/2_RandomForestClassifier_2155d2c82f7d/metadata/part-00000-f53cf08d-2505-478a-99b7-3a2dbad69d7c-c000.txt": "9cb36682d6caa92e2c51d874a10abe1132cd4e212856c8f0f6a3e5a95032e3bc",\n            "stages/2_RandomForestClassifier_2155d2c82f7d/treesMetadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/2_RandomForestClassifier_2155d2c82f7d/treesMetadata/part-00000-0dde3d69-b538-4de2-ab4c-d4164f038eb4-c000.snappy.parquet": "77dae29a6fe030f4e3acc0a91e88a57402111f5f94f40fb35048f6ec7973ab0d"\n          },\n          "training_cohort": "group_sample",\n          "training_seconds": 9.08,\n          "parameters": {\n            "numTrees": 12,\n            "maxDepth": 5,\n            "minInstancesPerNode": 30\n          },\n          "explanation": [\n            {\n              "feature": "totlen_fwd_pkts",\n              "method": "impurity_split_importance",\n              "value": 0.1834688886651975\n            },\n            {\n              "feature": "fwd_iat_mean",\n              "method": "impurity_split_importance",\n              "value": 0.17999052420586634\n            },\n            {\n              "feature": "flow_iat_mean",\n              "method": "impurity_split_importance",\n              "value": 0.09223460132340447\n            },\n            {\n              "feature": "tot_bwd_pkts",\n              "method": "impurity_split_importance",\n              "value": 0.07188866158158025\n            },\n            {\n              "feature": "flow_duration",\n              "method": "impurity_split_importance",\n              "value": 0.06977032009191769\n            },\n            {\n              "feature": "flow_pkts_s",\n              "method": "impurity_split_importance",\n              "value": 0.06713696862479576\n            },\n            {\n              "feature": "tot_fwd_pkts",\n              "method": "impurity_split_importance",\n              "value": 0.06233368101232681\n            },\n            {\n              "feature": "pkt_len_mean",\n              "method": "impurity_split_importance",\n              "value": 0.05861413688890729\n            },\n            {\n              "feature": "ack_flag_cnt",\n              "method": "impurity_split_importance",\n              "value": 0.05150918768659484\n            },\n            {\n              "feature": "pkt_len_std",\n              "method": "impurity_split_importance",\n              "value": 0.04881579751315323\n            },\n            {\n              "feature": "flow_byts_s",\n              "method": "impurity_split_importance",\n              "value": 0.0472943636653173\n            },\n            {\n              "feature": "totlen_bwd_pkts",\n              "method": "impurity_split_importance",\n              "value": 0.03154728010101541\n            },\n            {\n              "feature": "rst_flag_cnt",\n              "method": "impurity_split_importance",\n              "value": 0.012938145915254228\n            },\n            {\n              "feature": "flow_iat_std",\n              "method": "impurity_split_importance",\n              "value": 0.012289366606927317\n            },\n            {\n              "feature": "down_up_ratio",\n              "method": "impurity_split_importance",\n              "value": 0.00648405002904396\n            },\n            {\n              "feature": "bwd_iat_mean",\n              "method": "impurity_split_importance",\n              "value": 0.002626307110376393\n            },\n            {\n              "feature": "fin_flag_cnt",\n              "method": "impurity_split_importance",\n              "value": 0.0010577189783211242\n            },\n            {\n              "feature": "syn_flag_cnt",\n              "method": "impurity_split_importance",\n              "value": 0.0\n            }\n          ]\n        },\n        {\n          "path": "/kaggle/working/mmds_phase4/models/gradient_boosted_trees_group_sample",\n          "model": "gradient_boosted_trees",\n          "reload_probe_rows": 812520,\n          "model_bytes": 20175,\n          "model_files": 14,\n          "sha256": {\n            "metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "metadata/part-00000-28d7f8b4-8ae7-46cf-9ed3-61ab76345462-c000.txt": "9be6d1901d29d8df31aa5f6c5987491a0bbc62f1d026e1227b99be02647e24a6",\n            "stages/0_Imputer_75e58e12f8f2/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/0_Imputer_75e58e12f8f2/data/part-00000-5578f0ca-09f6-4d1f-8a1b-63219920519f-c000.snappy.parquet": "b95f973e5c9d08382e52dde2888bfb19b4371d8eb2000e48801a085ca373c01a",\n            "stages/0_Imputer_75e58e12f8f2/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/0_Imputer_75e58e12f8f2/metadata/part-00000-017f6684-c8b5-46ad-8526-a51bbf358713-c000.txt": "2a42bfb2b7bbb8602d3fa621a90eeea9d2e92abb20204d98eadfbfdad2c164ab",\n            "stages/1_VectorAssembler_bc29ebd64ced/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/1_VectorAssembler_bc29ebd64ced/metadata/part-00000-d8c453d2-e5e3-45fa-8e5d-8d2d1ea513a0-c000.txt": "4017fe5436b7188ce9285079cdc097d8ed69f5472e6744ce75353c801caba4ad",\n            "stages/2_GBTClassifier_3fc58e0cc2fc/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/2_GBTClassifier_3fc58e0cc2fc/data/part-00000-b101510e-4864-4f65-86c8-dd9ac3e9792d-c000.snappy.parquet": "e4920fc7039ec7c457169130f20824828a691b620326d2f170a660511c173f18",\n            "stages/2_GBTClassifier_3fc58e0cc2fc/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/2_GBTClassifier_3fc58e0cc2fc/metadata/part-00000-f7d506b0-6dcc-409c-a756-af0ae33184eb-c000.txt": "1bec11f595f0766317c60ad02e660259f468cb7eeda98a2bae2b894136dd4e23",\n            "stages/2_GBTClassifier_3fc58e0cc2fc/treesMetadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/2_GBTClassifier_3fc58e0cc2fc/treesMetadata/part-00000-42c3cf4a-3edf-4e98-978e-1d47e8d34860-c000.snappy.parquet": "2bc9334240b0b57bec706ca40e6f46b17cf643d7d16b9ff3393e16dd3ea8f7a9"\n          },\n          "training_cohort": "group_sample",\n          "training_seconds": 11.686,\n          "parameters": {\n            "maxIter": 6,\n            "maxDepth": 3,\n            "stepSize": 0.1\n          },\n          "explanation": [\n            {\n              "feature": "totlen_fwd_pkts",\n              "method": "impurity_split_importance",\n              "value": 0.2517482769853455\n            },\n            {\n              "feature": "fwd_iat_mean",\n              "method": "impurity_split_importance",\n              "value": 0.17895648984132703\n            },\n            {\n              "feature": "tot_fwd_pkts",\n              "method": "impurity_split_importance",\n              "value": 0.14091877663723415\n            },\n            {\n              "feature": "ack_flag_cnt",\n              "method": "impurity_split_importance",\n              "value": 0.11721792049786338\n            },\n            {\n              "feature": "tot_bwd_pkts",\n              "method": "impurity_split_importance",\n              "value": 0.09355878219713805\n            },\n            {\n              "feature": "flow_iat_mean",\n              "method": "impurity_split_importance",\n              "value": 0.08489831401785873\n            },\n            {\n              "feature": "flow_iat_std",\n              "method": "impurity_split_importance",\n              "value": 0.05290235565805834\n            },\n            {\n              "feature": "flow_byts_s",\n              "method": "impurity_split_importance",\n              "value": 0.029652021330934645\n            },\n            {\n              "feature": "rst_flag_cnt",\n              "method": "impurity_split_importance",\n              "value": 0.01869923511136819\n            },\n            {\n              "feature": "flow_duration",\n              "method": "impurity_split_importance",\n              "value": 0.015432340067010655\n            },\n            {\n              "feature": "flow_pkts_s",\n              "method": "impurity_split_importance",\n              "value": 0.011706679680500173\n            },\n            {\n              "feature": "syn_flag_cnt",\n              "method": "impurity_split_importance",\n              "value": 0.0026407667398478285\n            },\n            {\n              "feature": "totlen_bwd_pkts",\n              "method": "impurity_split_importance",\n              "value": 0.00166804123551327\n            },\n            {\n              "feature": "bwd_iat_mean",\n              "method": "impurity_split_importance",\n              "value": 0.0\n            },\n            {\n              "feature": "down_up_ratio",\n              "method": "impurity_split_importance",\n              "value": 0.0\n            },\n            {\n              "feature": "fin_flag_cnt",\n              "method": "impurity_split_importance",\n              "value": 0.0\n            },\n            {\n              "feature": "pkt_len_mean",\n              "method": "impurity_split_importance",\n              "value": 0.0\n            },\n            {\n              "feature": "pkt_len_std",\n              "method": "impurity_split_importance",\n              "value": 0.0\n            }\n          ]\n        },\n        {\n          "path": "/kaggle/working/mmds_phase4/models/logistic_regression_full_train_refit",\n          "model": "logistic_regression",\n          "reload_probe_rows": 812520,\n          "model_bytes": 16241,\n          "model_files": 16,\n          "sha256": {\n            "metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "metadata/part-00000-91526216-e836-468f-a884-814d345baba3-c000.txt": "29bf7ba3ab603bead56262a35a798ff3ec1d6f7697221f490bd80de514f4e5be",\n            "stages/0_Imputer_75e58e12f8f2/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/0_Imputer_75e58e12f8f2/data/part-00000-e6012794-1f89-4dcc-8191-05dbe5ff6bbc-c000.snappy.parquet": "b95f973e5c9d08382e52dde2888bfb19b4371d8eb2000e48801a085ca373c01a",\n            "stages/0_Imputer_75e58e12f8f2/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/0_Imputer_75e58e12f8f2/metadata/part-00000-f83819d3-79c9-472d-9d42-8a8987472b8b-c000.txt": "901e89faa78cfdea969ceb4c5f8ea24435f1b1764c31236a31275a68b12493e7",\n            "stages/1_VectorAssembler_bc29ebd64ced/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/1_VectorAssembler_bc29ebd64ced/metadata/part-00000-c7f54243-2ca4-4ec3-87ad-ace77130e722-c000.txt": "77459e19b797b9485568cbb4fd5f07a03b7dcaed763ba36b2f4f9ac3e24c7f17",\n            "stages/2_StandardScaler_611ae33371ff/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/2_StandardScaler_611ae33371ff/data/part-00000-602906a3-3234-4aea-8937-b0fbcfb7f42d-c000.snappy.parquet": "e5ca48abe1b71ba77b74e5909088ec9af7ba5e9fbe4af16dbd0c3c2b4c22031d",\n            "stages/2_StandardScaler_611ae33371ff/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/2_StandardScaler_611ae33371ff/metadata/part-00000-1b5979f1-b2ea-4d79-97a1-e0aaa667769d-c000.txt": "3905fc148dc2e1f77c392ddb6326e117bd7b2bb980687811cb4971638bdbdc50",\n            "stages/3_LogisticRegression_396b6b8873c9/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/3_LogisticRegression_396b6b8873c9/data/part-00000-555c029a-021a-42cc-8388-93bd79754a90-c000.snappy.parquet": "52054a1a1c413a2129d5edab7413589d0659a49c8d1fbdc5141004d963766df4",\n            "stages/3_LogisticRegression_396b6b8873c9/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n            "stages/3_LogisticRegression_396b6b8873c9/metadata/part-00000-a3b741ca-c5cf-4d56-97a8-698539fda113-c000.txt": "650d50212587c62d9b4649d679dd9ff3cde5662431a564c1b8794976517b8084"\n          },\n          "training_cohort": "full_train_refit",\n          "training_seconds": 29.211,\n          "parameters": {\n            "maxIter": 12,\n            "regParam": 0.005,\n            "elasticNetParam": 0.0\n          },\n          "explanation": [\n            {\n              "feature": "pkt_len_mean",\n              "method": "scaled_lr_coefficient",\n              "value": -1.9062686969568232\n            },\n            {\n              "feature": "flow_duration",\n              "method": "scaled_lr_coefficient",\n              "value": -0.8674368222976909\n            },\n            {\n              "feature": "pkt_len_std",\n              "method": "scaled_lr_coefficient",\n              "value": 0.8222475553209422\n            },\n            {\n              "feature": "rst_flag_cnt",\n              "method": "scaled_lr_coefficient",\n              "value": 0.6883605432941551\n            },\n            {\n              "feature": "flow_byts_s",\n              "method": "scaled_lr_coefficient",\n              "value": -0.609026279602007\n            },\n            {\n              "feature": "ack_flag_cnt",\n              "method": "scaled_lr_coefficient",\n              "value": 0.6034332888339793\n            },\n            {\n              "feature": "syn_flag_cnt",\n              "method": "scaled_lr_coefficient",\n              "value": -0.5696343352283386\n            },\n            {\n              "feature": "flow_pkts_s",\n              "method": "scaled_lr_coefficient",\n              "value": 0.3474925176329858\n            },\n            {\n              "feature": "down_up_ratio",\n              "method": "scaled_lr_coefficient",\n              "value": 0.23560894032017093\n            },\n            {\n              "feature": "bwd_iat_mean",\n              "method": "scaled_lr_coefficient",\n              "value": 0.15253831532400683\n            },\n            {\n              "feature": "fin_flag_cnt",\n              "method": "scaled_lr_coefficient",\n              "value": -0.1473693534529459\n            },\n            {\n              "feature": "flow_iat_mean",\n              "method": "scaled_lr_coefficient",\n              "value": 0.11848347565720055\n            },\n            {\n              "feature": "fwd_iat_mean",\n              "method": "scaled_lr_coefficient",\n              "value": 0.10038513374101134\n            },\n            {\n              "feature": "tot_fwd_pkts",\n              "method": "scaled_lr_coefficient",\n              "value": 0.08887497151124718\n            },\n            {\n              "feature": "totlen_fwd_pkts",\n              "method": "scaled_lr_coefficient",\n              "value": 0.05895817586666829\n            },\n            {\n              "feature": "totlen_bwd_pkts",\n              "method": "scaled_lr_coefficient",\n              "value": 0.011839444588369944\n            },\n            {\n              "feature": "tot_bwd_pkts",\n              "method": "scaled_lr_coefficient",\n              "value": -0.0031745524107773537\n            },\n            {\n              "feature": "flow_iat_std",\n              "method": "scaled_lr_coefficient",\n              "value": -0.0004389912442234386\n            }\n          ]\n        }\n      ],\n      "trial_selections": {\n        "logistic_regression": {\n          "trial": 1,\n          "parameters": {\n            "maxIter": 12,\n            "regParam": 0.005,\n            "elasticNetParam": 0.0\n          }\n        },\n        "random_forest": {\n          "trial": 1,\n          "parameters": {\n            "numTrees": 12,\n            "maxDepth": 5,\n            "minInstancesPerNode": 30\n          }\n        },\n        "gradient_boosted_trees": {\n          "trial": 1,\n          "parameters": {\n            "maxIter": 6,\n            "maxDepth": 3,\n            "stepSize": 0.1\n          }\n        }\n      },\n      "input": {\n        "path": "/kaggle/input/notebooks/ramcharan2345/mmds-phase-2-spark-data-engineering/mmds_phase2/phase2-v1/feature_ready",\n        "days": [\n          "2018-02-14",\n          "2018-02-15",\n          "2018-02-16",\n          "2018-02-20",\n          "2018-02-21",\n          "2018-02-22",\n          "2018-02-23",\n          "2018-02-28",\n          "2018-03-01",\n          "2018-03-02"\n        ],\n        "totals": {\n          "0": 13484708,\n          "1": 2748235\n        },\n        "rows": 16232943,\n        "temporally_flagged": 14,\n        "policy_version": "phase2-v1",\n        "receipt_sha256": "1fa00cd2b9c853ee33eaacee8ca7632dbb41f739573326280d6ffbfb2ccff25a",\n        "producer": {\n          "url": "https://www.kaggle.com/code/ramcharan2345/mmds-phase-2-spark-data-engineering",\n          "kernel_version": 1,\n          "status": "COMPLETE"\n        }\n      },\n      "registered_predictors": [\n        "flow_duration",\n        "tot_fwd_pkts",\n        "tot_bwd_pkts",\n        "totlen_fwd_pkts",\n        "totlen_bwd_pkts",\n        "pkt_len_mean",\n        "pkt_len_std",\n        "flow_iat_mean",\n        "flow_iat_std",\n        "fwd_iat_mean",\n        "bwd_iat_mean",\n        "flow_pkts_s",\n        "flow_byts_s",\n        "down_up_ratio",\n        "syn_flag_cnt",\n        "ack_flag_cnt",\n        "rst_flag_cnt",\n        "fin_flag_cnt"\n      ],\n      "overlap": {\n        "predictor_before": {\n          "groups": 142016,\n          "rows": 2547596\n        },\n        "after": {\n          "predictor_key": {\n            "groups": 0,\n            "rows": 0\n          },\n          "record_fingerprint": {\n            "groups": 0,\n            "rows": 0\n          },\n          "split_feature_key": {\n            "groups": 0,\n            "rows": 0\n          }\n        },\n        "purged_rows": 859101\n      },\n      "retained_predictor_conflicts": {\n        "groups": 17434,\n        "rows": 1363306\n      },\n      "split_support": {\n        "train": {\n          "labels": {\n            "0": 10844286,\n            "1": 2678436\n          },\n          "families": {\n            "Benign": 10844286,\n            "Credential brute force": 380949,\n            "DoS": 654300,\n            "DDoS": 1263933,\n            "Infiltration": 93063,\n            "Bot": 286191\n          },\n          "original_labels": {\n            "Benign": 10844286,\n            "FTP-BruteForce": 193360,\n            "SSH-Bruteforce": 187589,\n            "DoS attacks-GoldenEye": 41508,\n            "DoS attacks-Slowloris": 10990,\n            "DoS attacks-Hulk": 461912,\n            "DoS attacks-SlowHTTPTest": 139890,\n            "DDoS attacks-LOIC-HTTP": 576191,\n            "DDOS attack-HOIC": 686012,\n            "DDOS attack-LOIC-UDP": 1730,\n            "Infilteration": 93063,\n            "Bot": 286191\n          },\n          "rows": 13522722\n        },\n        "validation": {\n          "labels": {\n            "0": 351060,\n            "1": 41842\n          },\n          "families": {\n            "Benign": 351060,\n            "Infiltration": 41842\n          },\n          "original_labels": {\n            "Benign": 351060,\n            "Infilteration": 41842\n          },\n          "rows": 392902\n        },\n        "test": {\n          "labels": {\n            "0": 1457560,\n            "1": 658\n          },\n          "families": {\n            "Benign": 1457560,\n            "Web attack": 658\n          },\n          "original_labels": {\n            "Benign": 1457560,\n            "Brute Force -Web": 423,\n            "Brute Force -XSS": 176,\n            "SQL Injection": 59\n          },\n          "rows": 1458218\n        }\n      },\n      "train_sample_support": {\n        "labels": {\n          "0": 659296,\n          "1": 153224\n        },\n        "families": {\n          "Benign": 659296,\n          "Credential brute force": 12384,\n          "DoS": 37445,\n          "DDoS": 80477,\n          "Infiltration": 5516,\n          "Bot": 17402\n        },\n        "original_labels": {\n          "Benign": 659296,\n          "FTP-BruteForce": 6425,\n          "SSH-Bruteforce": 5959,\n          "DoS attacks-GoldenEye": 2663,\n          "DoS attacks-Slowloris": 684,\n          "DoS attacks-Hulk": 28198,\n          "DoS attacks-SlowHTTPTest": 5900,\n          "DDoS attacks-LOIC-HTTP": 36066,\n          "DDOS attack-HOIC": 44289,\n          "DDOS attack-LOIC-UDP": 122,\n          "Infilteration": 5516,\n          "Bot": 17402\n        },\n        "rows": 812520\n      },\n      "baselines": {\n        "majority": {\n          "attack_prediction": 0\n        },\n        "duration_rule": {\n          "benign_training_median_us": 30920.0\n        }\n      },\n      "preprocessing": {\n        "active": [\n          "flow_duration",\n          "tot_fwd_pkts",\n          "tot_bwd_pkts",\n          "totlen_fwd_pkts",\n          "totlen_bwd_pkts",\n          "pkt_len_mean",\n          "pkt_len_std",\n          "flow_iat_mean",\n          "flow_iat_std",\n          "fwd_iat_mean",\n          "bwd_iat_mean",\n          "flow_pkts_s",\n          "flow_byts_s",\n          "down_up_ratio",\n          "syn_flag_cnt",\n          "ack_flag_cnt",\n          "rst_flag_cnt",\n          "fin_flag_cnt"\n        ],\n        "dropped_train_constant_or_empty": [],\n        "training_medians": {\n          "flow_duration": 25413.0,\n          "tot_fwd_pkts": 2.0,\n          "tot_bwd_pkts": 1.0,\n          "totlen_fwd_pkts": 45.0,\n          "totlen_bwd_pkts": 112.0,\n          "pkt_len_mean": 55.0,\n          "pkt_len_std": 35.21836642056717,\n          "flow_iat_mean": 12840.6666666667,\n          "flow_iat_std": 123.03657992645927,\n          "fwd_iat_mean": 5123.0,\n          "bwd_iat_mean": 0.0,\n          "flow_pkts_s": 122.92563,\n          "flow_byts_s": 1002.68197,\n          "down_up_ratio": 0.0,\n          "syn_flag_cnt": 0.0,\n          "ack_flag_cnt": 0.0,\n          "rst_flag_cnt": 0.0,\n          "fin_flag_cnt": 0.0\n        }\n      },\n      "preprocess_seconds": 14.438,\n      "post_imputation_overlap": {\n        "groups": 0,\n        "rows": 0\n      },\n      "selection_frozen": true,\n      "elapsed_seconds": 491.052,\n      "downloaded_reload": [\n        {\n          "artifact_directory": "logistic_regression_group_sample",\n          "finite_probe_predictions": 3\n        },\n        {\n          "artifact_directory": "random_forest_group_sample",\n          "finite_probe_predictions": 3\n        },\n        {\n          "artifact_directory": "gradient_boosted_trees_group_sample",\n          "finite_probe_predictions": 3\n        },\n        {\n          "artifact_directory": "logistic_regression_full_train_refit",\n          "finite_probe_predictions": 3\n        }\n      ]\n    }\n  }\n}\n', 'reports/phase4_metrics.csv': 'protocol,model,training_cohort,split,training_rows,training_seconds,tp,tn,fp,fn,rows,accuracy,precision,recall,specificity,f1,balanced_accuracy,roc_auc,pr_auc,prediction_seconds\nA,majority,full_train_baseline,validation,11455925,0.0,0,1990157,0,354569,2344726,0.8487801986244875,,0.0,1.0,0.0,0.5,0.5,0.15121980137551252,2.885\nA,duration_rule,full_train_baseline,validation,11455925,0.0,215748,1010854,979303,138821,2344726,0.5231323404099242,0.18053455459223078,0.6084795907143602,0.5079267615570028,0.2784527819723545,0.5582031761356815,0.5582031761356815,0.1690572304147137,2.882\nA,majority,full_train_baseline,test,11455925,0.0,0,2068172,0,364120,2432292,0.8502975794024731,,0.0,1.0,0.0,0.5,0.5,0.14970242059752695,2.509\nA,duration_rule,full_train_baseline,test,11455925,0.0,225592,1010484,1057688,138528,2432292,0.5081939175066151,0.17579327972071568,0.6195539931890586,0.48858798978034707,0.27387641131479906,0.5540709914847028,0.5540709914847028,0.1658671165530317,2.586\nA,logistic_regression,group_sample,validation,695307,6.295,364,1990154,3,354205,2344726,0.8489341611770416,0.9918256130790191,0.0010265984899977155,0.9999984925812385,0.002051073996438795,0.5005125455356181,0.8203354549081758,0.6100444118279686,3.753\nA,logistic_regression,group_sample,test,695307,6.295,324,2068167,5,363796,2432292,0.8504287314187606,0.9848024316109423,0.0008898165439964847,0.9999975824061055,0.0017780265551558653,0.500443699475051,0.8063869233854803,0.5930784428992267,4.051\nA,random_forest,group_sample,validation,695307,11.737,302628,1967648,22509,51941,2344726,0.9682478890923716,0.9307707212651898,0.8535094720632656,0.9886898370329577,0.8904673491185895,0.9210996545481116,0.9676168582168041,0.9220911936832712,5.861\nA,random_forest,group_sample,test,695307,11.737,301553,2045915,22257,62567,2432292,0.9651258977129391,0.931265248139341,0.8281692848511479,0.9892383225379707,0.8766967569374791,0.9087038036945594,0.9656252738284283,0.9165679954807866,5.802\nA,gradient_boosted_trees,group_sample,validation,695307,19.557,304070,1968486,21671,50499,2344726,0.9692202841611344,0.933471684559205,0.8575763814659488,0.9891109093403184,0.8939160088783055,0.9233436454031336,0.9609982675249529,0.8959337445321933,4.47\nA,gradient_boosted_trees,group_sample,test,695307,19.557,309359,2044256,23916,54761,2432292,0.9676531436192694,0.9282394419023329,0.849607272327804,0.9884361648837717,0.8871844507058411,0.9190217186057879,0.9576786382978011,0.889155941145078,4.237\nA,logistic_regression,full_train_refit,validation,11455925,25.26,460,1989107,1050,354109,2344726,0.8485285700759918,0.304635761589404,0.0012973497401070032,0.999472403433498,0.0025836963145818765,0.5003848765868025,0.7910496253085418,0.336277301248326,3.908\nA,logistic_regression,full_train_refit,test,11455925,25.26,1392,1983162,85010,362728,2432292,0.8159193057412515,0.016110738177357007,0.0038229155223552676,0.9588960686055125,0.006179498448466446,0.48135949206393386,0.752074094446787,0.2833215198288908,4.026\nB,majority,full_train_baseline,validation,13191622,0.0,0,1094200,0,57056,1151256,0.9504402148609866,,0.0,1.0,0.0,0.5,0.5,0.04955978513901339,3.731\nB,duration_rule,full_train_baseline,validation,13191622,0.0,27586,795014,299186,29470,1151256,0.7145239633930247,0.08441971772367278,0.48348990465507574,0.7265710107841346,0.1437414675323322,0.6050304577196052,0.6050304577196052,0.06641421062065272,3.456\nB,majority,full_train_baseline,test,13191622,0.0,0,771584,0,208525,980109,0.7872430515381452,,0.0,1.0,0.0,0.5,0.5,0.21275694846185475,2.908\nB,duration_rule,full_train_baseline,test,13191622,0.0,172571,559001,212583,35954,980109,0.7464190207415706,0.4480571407800516,0.827579426927227,0.7244849556237558,0.5813613080469412,0.7760321912754914,0.7760321912754914,0.4074865467764144,2.704\nB,logistic_regression,group_sample,validation,795270,13.5,600,1092598,1602,56456,1151256,0.9495698610908434,0.2724795640326976,0.010515984296130118,0.9985359166514348,0.020250430321644336,0.5045259504737825,0.44900943161671913,0.054790438911638734,3.6\nB,logistic_regression,group_sample,test,795270,13.5,177,771533,51,208348,980109,0.7873716086680155,0.7763157894736842,0.0008488190864404748,0.9999339022063702,0.0016957840126848476,0.5003913606464053,0.8224500524391902,0.6231745883963631,3.829\nB,random_forest,group_sample,validation,795270,19.239,586,1084706,9494,56470,1151256,0.9427025787487753,0.05813492063492064,0.010270611329220415,0.9913233412538841,0.017457102001906578,0.5007969762915523,0.3919743896266559,0.04074654677392458,5.548\nB,random_forest,group_sample,test,795270,19.239,417,766721,4863,208108,980109,0.7827068213841522,0.07897727272727273,0.0019997602205970508,0.99369738097213,0.0039007506840345175,0.4978485705963635,0.7985884732334407,0.6035922752334859,5.206\nB,gradient_boosted_trees,group_sample,validation,795270,17.415,470,1080372,13828,56586,1151256,0.9388372351588179,0.03287173031193174,0.008237521031968592,0.987362456589289,0.013173753398548085,0.4977999888106288,0.4240689529095466,0.044368054370374464,4.048\nB,gradient_boosted_trees,group_sample,test,795270,17.415,1218,764563,7021,207307,980109,0.7813222815013432,0.14783347493627869,0.005841026255844623,0.9909005370769741,0.011238028454909487,0.49837078166640936,0.429676530486362,0.21884530784082926,3.894\nB,logistic_regression,full_train_refit,validation,13191622,44.995,72,1093939,261,56984,1151256,0.9502760463354806,0.21621621621621623,0.0012619181155356142,0.9997614695668068,0.0025091916569377408,0.5005116938411712,0.45398883551558955,0.05589106607229337,4.134\nB,logistic_regression,full_train_refit,test,13191622,44.995,787,771493,91,207738,980109,0.7879531766364761,0.8963553530751709,0.0037741278024217718,0.9998820607996018,0.007516606734382984,0.5018280943010118,0.8276010573455483,0.6496362370253705,3.95\nC,majority,full_train_baseline,validation,13522722,0.0,0,351060,0,41842,392902,0.8935052506731959,,0.0,1.0,0.0,0.5,0.5,0.10649474932680414,2.363\nC,duration_rule,full_train_baseline,validation,13522722,0.0,13096,232873,118187,28746,392902,0.6260314276842571,0.09975396662172559,0.31298695091056833,0.6633424485842876,0.15128953068592058,0.48816469974742793,0.48816469974742793,0.10438497230119091,2.078\nC,majority,full_train_baseline,test,13522722,0.0,0,1457560,0,658,1458218,0.9995487643137034,,0.0,1.0,0.0,0.5,0.5,0.000451235686296562,2.034\nC,duration_rule,full_train_baseline,test,13522722,0.0,71,1131774,325786,587,1458218,0.7761836707543043,0.00021788698723673267,0.10790273556231003,0.7764853590932792,0.00043489579345512455,0.4421940473277946,0.44219404732779466,0.0004260567233281001,2.285\nC,logistic_regression,group_sample,validation,812520,10.976,641,347525,3535,41201,392902,0.8861395462481738,0.15349616858237547,0.015319535395057598,0.9899304962114738,0.02785866400104307,0.5026250158032657,0.4838858048415502,0.10611844996187014,2.27\nC,logistic_regression,group_sample,test,812520,10.976,1,1455269,2291,657,1458218,0.9979783544024282,0.0004363001745200698,0.001519756838905775,0.9984281950657263,0.0006779661016949153,0.4999739759523161,0.3605909537912005,0.0012080643742875576,3.129\nC,random_forest,group_sample,validation,812520,9.08,743,327394,23666,41099,392902,0.8351624578139077,0.030439591953787537,0.01775727737679843,0.9325870221614538,0.022429850115469954,0.47517214976912614,0.4652998271123345,0.09593180138975133,2.729\nC,random_forest,group_sample,test,812520,9.08,16,1445591,11969,642,1458218,0.9913517731916627,0.0013350020859407593,0.0243161094224924,0.991788331183622,0.002531044846950882,0.5080522203030572,0.5062522047297098,0.000545093476709124,3.932\nC,gradient_boosted_trees,group_sample,validation,812520,11.686,1450,322085,28975,40392,392902,0.8234496133895984,0.047658175842235,0.03465417523062951,0.9174642511251638,0.040128966194805375,0.4760592131778967,0.47360582403309415,0.09809767096047926,2.449\nC,gradient_boosted_trees,group_sample,test,812520,11.686,27,1435584,21976,631,1458218,0.9844968310636681,0.001227105394718902,0.041033434650455926,0.9849227476055874,0.002382948678346057,0.5129780911280216,0.4776807109912883,0.0006529601645462665,2.577\nC,logistic_regression,full_train_refit,validation,13522722,29.211,551,347773,3287,41291,392902,0.8865416821497473,0.14356435643564355,0.013168586587639215,0.9906369281604284,0.024124343257443084,0.5019027573740338,0.4837028020579236,0.10551223929699871,2.659\nC,logistic_regression,full_train_refit,test,13522722,29.211,0,1457108,452,658,1458218,0.9992387969425697,0.0,0.0,0.9996898926973846,0.0,0.4998449463486923,0.36451130208365046,0.001251129809765967,3.043\n', 'configs/streaming_experiments.yaml': '{\n  "version": "phase6-v1",\n  "phase2": "ramcharan2345/mmds-phase-2-spark-data-engineering/1",\n  "phase4": "ramcharan2345/mmds-phase-4-supervised-attack-detection/2",\n  "phase5": "ramcharan2345/mmds-phase-5-clustering-pattern-evolution/2",\n  "window_hours": 3,\n  "files_per_trigger": 1,\n  "comparison_files_per_trigger": 2,\n  "comparison_chunks": 4,\n  "max_chunks": 90,\n  "max_staging_bytes": 8589934592,\n  "max_chunk_rows": 4000000,\n  "features": {\n    "flow_duration": [0, 100, 10000, 1000000, 10000000, 60000000],\n    "tot_fwd_pkts": [0, 1, 2, 5, 20, 100],\n    "totlen_fwd_pkts": [0, 64, 128, 512, 1460, 10000],\n    "pkt_len_mean": [0, 32, 64, 128, 512, 1200],\n    "flow_iat_mean": [0, 100, 10000, 100000, 1000000, 10000000],\n    "flow_pkts_s": [0, 1, 10, 100, 1000, 10000]\n  },\n  "score_bins": 2048,\n  "distance_bounds": [0, 1, 2, 4, 8, 12, 16, 20, 32, 64, 128, 256, 512, 1024],\n  "alert_thresholds": {"INPUT_SHIFT": 0.10, "SCORE_SHIFT": 0.10, "CLUSTER_SHIFT": 0.10, "NOVELTY_RATE": 0.015, "PERFORMANCE_WARNING": 0.20},\n  "minimum_window": 10000,\n  "minimum_per_class": 100,\n  "spark": {"master": "local[4]", "driver_memory": "8g", "shuffle_partitions": 48, "input_partition_bytes": 16777216}\n}\n', 'manifests/phase5_clustering_models.json': '{\n  "status": "validated",\n  "kernel_url": "https://www.kaggle.com/code/ramcharan2345/mmds-phase-5-clustering-pattern-evolution",\n  "experiments": {\n    "G": {\n      "experiment": "G",\n      "config": {\n        "version": "phase5-v1",\n        "seed": 23013,\n        "candidate_k": [\n          3,\n          5,\n          8,\n          12\n        ],\n        "max_iter": 16,\n        "init_steps": 2,\n        "tolerance": 0.0001,\n        "training_permyriad": 250,\n        "internal_permyriad": 12,\n        "reference_percentiles": [\n          0.95,\n          0.99,\n          0.999\n        ],\n        "primary_percentile": 0.99,\n        "quantile_accuracy": 8192,\n        "max_internal_rows": 40000,\n        "rare_support": 100,\n        "window_hours": 3,\n        "lag_reference": "full retained phase4 train-day population",\n        "spark": {\n          "master": "local[4]",\n          "driver_memory": "8g",\n          "shuffle_partitions": 48,\n          "input_partition_bytes": 67108864\n        }\n      },\n      "status": "COMPLETE",\n      "resources": {\n        "spark_version": "4.3.0",\n        "master": "local[4]",\n        "driver_memory": "8g",\n        "shuffle_partitions": 48,\n        "input_partition_bytes": 67108864,\n        "default_parallelism": 4,\n        "cpu_affinity": 4,\n        "scratch_output_free_bytes_start": 20939968512,\n        "memory.max": "32212254720",\n        "cpu.max": "400000 100000",\n        "driver_peak_rss_bytes": 4750106624\n      },\n      "features": [\n        "flow_duration",\n        "tot_fwd_pkts",\n        "tot_bwd_pkts",\n        "totlen_fwd_pkts",\n        "totlen_bwd_pkts",\n        "pkt_len_mean",\n        "pkt_len_std",\n        "flow_iat_mean",\n        "flow_iat_std",\n        "fwd_iat_mean",\n        "bwd_iat_mean",\n        "flow_pkts_s",\n        "flow_byts_s",\n        "down_up_ratio",\n        "syn_flag_cnt",\n        "ack_flag_cnt",\n        "rst_flag_cnt",\n        "fin_flag_cnt"\n      ],\n      "input": {\n        "path": "/kaggle/input/notebooks/ramcharan2345/mmds-phase-2-spark-data-engineering/mmds_phase2/phase2-v1/feature_ready",\n        "days": [\n          "2018-02-14",\n          "2018-02-15",\n          "2018-02-16",\n          "2018-02-20",\n          "2018-02-21",\n          "2018-02-22",\n          "2018-02-23",\n          "2018-02-28",\n          "2018-03-01",\n          "2018-03-02"\n        ],\n        "totals": {\n          "0": 13484708,\n          "1": 2748235\n        },\n        "rows": 16232943,\n        "temporally_flagged": 14,\n        "policy_version": "phase2-v1",\n        "receipt_sha256": "1fa00cd2b9c853ee33eaacee8ca7632dbb41f739573326280d6ffbfb2ccff25a",\n        "producer": {\n          "url": "https://www.kaggle.com/code/ramcharan2345/mmds-phase-2-spark-data-engineering",\n          "kernel_version": 1,\n          "status": "COMPLETE"\n        }\n      },\n      "purge": {\n        "groups_before": 0,\n        "groups_after": 0,\n        "rows": 0\n      },\n      "matched_phase4": null,\n      "preprocessing": {\n        "transform": "signed_log1p",\n        "fit_columns": [\n          "flow_duration",\n          "tot_fwd_pkts",\n          "tot_bwd_pkts",\n          "totlen_fwd_pkts",\n          "totlen_bwd_pkts",\n          "pkt_len_mean",\n          "pkt_len_std",\n          "flow_iat_mean",\n          "flow_iat_std",\n          "fwd_iat_mean",\n          "bwd_iat_mean",\n          "flow_pkts_s",\n          "flow_byts_s",\n          "down_up_ratio",\n          "syn_flag_cnt",\n          "ack_flag_cnt",\n          "rst_flag_cnt",\n          "fin_flag_cnt"\n        ],\n        "active": [\n          "flow_duration",\n          "tot_fwd_pkts",\n          "tot_bwd_pkts",\n          "totlen_fwd_pkts",\n          "totlen_bwd_pkts",\n          "pkt_len_mean",\n          "pkt_len_std",\n          "flow_iat_mean",\n          "flow_iat_std",\n          "fwd_iat_mean",\n          "bwd_iat_mean",\n          "flow_pkts_s",\n          "flow_byts_s",\n          "down_up_ratio",\n          "syn_flag_cnt",\n          "ack_flag_cnt",\n          "rst_flag_cnt",\n          "fin_flag_cnt"\n        ],\n        "dropped": [],\n        "reference_stats": {\n          "flow_duration": {\n            "invalid_or_null": 2,\n            "available": true,\n            "reference_std": 4.947151865641074,\n            "near_constant": false\n          },\n          "tot_fwd_pkts": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 0.7530213879187553,\n            "near_constant": false\n          },\n          "tot_bwd_pkts": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 0.9378945538594473,\n            "near_constant": false\n          },\n          "totlen_fwd_pkts": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 2.691333643392363,\n            "near_constant": false\n          },\n          "totlen_bwd_pkts": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 3.173848812379961,\n            "near_constant": false\n          },\n          "pkt_len_mean": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 2.1025627607160313,\n            "near_constant": false\n          },\n          "pkt_len_std": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 2.328633749104362,\n            "near_constant": false\n          },\n          "flow_iat_mean": {\n            "invalid_or_null": 2,\n            "available": true,\n            "reference_std": 4.181293329541656,\n            "near_constant": false\n          },\n          "flow_iat_std": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 6.264059109273554,\n            "near_constant": false\n          },\n          "fwd_iat_mean": {\n            "invalid_or_null": 2,\n            "available": true,\n            "reference_std": 6.0557528119987944,\n            "near_constant": false\n          },\n          "bwd_iat_mean": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 6.163171726202694,\n            "near_constant": false\n          },\n          "flow_pkts_s": {\n            "invalid_or_null": 9,\n            "available": true,\n            "reference_std": 3.8365363784055213,\n            "near_constant": false\n          },\n          "flow_byts_s": {\n            "invalid_or_null": 7,\n            "available": true,\n            "reference_std": 4.9738935827118915,\n            "near_constant": false\n          },\n          "down_up_ratio": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 0.3569073900673366,\n            "near_constant": false\n          },\n          "syn_flag_cnt": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 0.15167067548018145,\n            "near_constant": false\n          },\n          "ack_flag_cnt": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 0.32647293591636467,\n            "near_constant": false\n          },\n          "rst_flag_cnt": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 0.27492864689025487,\n            "near_constant": false\n          },\n          "fin_flag_cnt": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 0.042408098008640864,\n            "near_constant": false\n          }\n        },\n        "imputation_surrogates": {\n          "flow_duration_valid": 10.271839059304192,\n          "tot_fwd_pkts_valid": 1.0986122886681096,\n          "tot_bwd_pkts_valid": 0.6931471805599453,\n          "totlen_fwd_pkts_valid": 3.8501476017100584,\n          "totlen_bwd_pkts_valid": 4.6913478822291435,\n          "pkt_len_mean_valid": 4.013375499688434,\n          "pkt_len_std_valid": 3.589566349836834,\n          "flow_iat_mean_valid": 9.528454651548163,\n          "flow_iat_std_valid": 4.70593411340219,\n          "fwd_iat_mean_valid": 8.70167907103957,\n          "bwd_iat_mean_valid": 0.0,\n          "flow_pkts_s_valid": 4.739346173287946,\n          "flow_byts_s_valid": 6.795194361202203,\n          "down_up_ratio_valid": 0.0,\n          "syn_flag_cnt_valid": 0.0,\n          "ack_flag_cnt_valid": 0.0,\n          "rst_flag_cnt_valid": 0.0,\n          "fin_flag_cnt_valid": 0.0\n        },\n        "reference_means": [\n          10.635960082574252,\n          1.4396548986409057,\n          1.0914135214250371,\n          3.7598334915584624,\n          3.948889165036629,\n          3.1733805946445233,\n          3.2033206649848287,\n          9.470806759791927,\n          6.135387665867699,\n          7.7695968071489805,\n          5.05226565531731,\n          5.137223128237749,\n          6.45527291520749,\n          0.3450593185074743,\n          0.03494990928388112,\n          0.23025958254038817,\n          0.13555774713056207,\n          0.00260438917912373\n        ],\n        "reference_scales": [\n          4.947151865641074,\n          0.7530213879187553,\n          0.9378945538594473,\n          2.691333643392363,\n          3.173848812379961,\n          2.1025627607160313,\n          2.328633749104362,\n          4.181293329541656,\n          6.264059109273554,\n          6.0557528119987944,\n          6.163171726202694,\n          3.8365363784055213,\n          4.9738935827118915,\n          0.3569073900673366,\n          0.15167067548018145,\n          0.32647293591636467,\n          0.27492864689025487,\n          0.042408098008640864\n        ]\n      },\n      "training": {\n        "fit_rows": 390702,\n        "internal_rows": 18099,\n        "reference_rows": 16232943,\n        "share_of_reference": 0.024068463740678447\n      },\n      "selected_k": 5,\n      "trials": [\n        {\n          "k": 3,\n          "seed": 23013,\n          "initialization": "k-means||",\n          "budget": 16,\n          "iterations": 11,\n          "wssse": 3963550.706735454,\n          "cluster_sizes": [\n            161216,\n            212405,\n            17081\n          ],\n          "training_rows": 390702,\n          "empty_clusters": 0,\n          "smallest_share": 0.043718742161545115,\n          "fit_seconds": 17.601,\n          "jvm_peak_rss_bytes": 3835715584,\n          "silhouette": 0.5376128609801271,\n          "silhouette_rows": 18099\n        },\n        {\n          "k": 5,\n          "seed": 23013,\n          "initialization": "k-means||",\n          "budget": 16,\n          "iterations": 16,\n          "wssse": 2224306.089655832,\n          "cluster_sizes": [\n            68133,\n            121466,\n            52459,\n            16947,\n            131697\n          ],\n          "training_rows": 390702,\n          "empty_clusters": 0,\n          "smallest_share": 0.04337576976826328,\n          "fit_seconds": 15.082,\n          "jvm_peak_rss_bytes": 3835715584,\n          "silhouette": 0.6178674460772436,\n          "silhouette_rows": 18099\n        },\n        {\n          "k": 8,\n          "seed": 23013,\n          "initialization": "k-means||",\n          "budget": 16,\n          "iterations": 16,\n          "wssse": 1496496.2336061099,\n          "cluster_sizes": [\n            51871,\n            67081,\n            49778,\n            89091,\n            14903,\n            1468,\n            111836,\n            4674\n          ],\n          "training_rows": 390702,\n          "empty_clusters": 0,\n          "smallest_share": 0.003757339353266684,\n          "fit_seconds": 14.047,\n          "jvm_peak_rss_bytes": 3835715584,\n          "silhouette": 0.5663268851635334,\n          "silhouette_rows": 18099\n        },\n        {\n          "k": 12,\n          "seed": 23013,\n          "initialization": "k-means||",\n          "budget": 16,\n          "iterations": 16,\n          "wssse": 1395303.8753451167,\n          "cluster_sizes": [\n            13662,\n            28818,\n            44556,\n            82850,\n            36502,\n            52696,\n            9326,\n            15733,\n            16239,\n            3845,\n            25260,\n            61215\n          ],\n          "training_rows": 390702,\n          "empty_clusters": 0,\n          "smallest_share": 0.0098412600908109,\n          "fit_seconds": 13.797,\n          "jvm_peak_rss_bytes": 3835715584,\n          "silhouette": 0.609291062515786,\n          "silhouette_rows": 18099\n        }\n      ],\n      "robustness": {\n        "paired_rows": 18099,\n        "seed_ari": 0.8954796677466573,\n        "scale_ari": 0.2899832117108664,\n        "alternate_seed": {\n          "k": 5,\n          "seed": 23014,\n          "initialization": "k-means||",\n          "budget": 16,\n          "iterations": 10,\n          "wssse": 2375059.0026243413,\n          "cluster_sizes": [\n            17019,\n            112559,\n            133596,\n            126060,\n            1468\n          ],\n          "training_rows": 390702,\n          "empty_clusters": 0,\n          "smallest_share": 0.003757339353266684,\n          "fit_seconds": 9.427,\n          "jvm_peak_rss_bytes": 3835715584\n        },\n        "linear": {\n          "k": 5,\n          "seed": 23013,\n          "initialization": "k-means||",\n          "budget": 16,\n          "iterations": 10,\n          "wssse": 4351926.195591406,\n          "cluster_sizes": [\n            16853,\n            16480,\n            113096,\n            4544,\n            239729\n          ],\n          "training_rows": 390702,\n          "empty_clusters": 0,\n          "smallest_share": 0.011630347425915403,\n          "fit_seconds": 9.35,\n          "jvm_peak_rss_bytes": 3835715584\n        },\n        "linear_silhouette": 0.42688826264849933,\n        "dominance": [\n          {\n            "feature": "pkt_len_mean",\n            "share_between_centroids": 0.07455241217018589\n          },\n          {\n            "feature": "totlen_bwd_pkts",\n            "share_between_centroids": 0.07377299369619311\n          },\n          {\n            "feature": "pkt_len_std",\n            "share_between_centroids": 0.07062813550435584\n          },\n          {\n            "feature": "syn_flag_cnt",\n            "share_between_centroids": 0.069520903400549\n          },\n          {\n            "feature": "totlen_fwd_pkts",\n            "share_between_centroids": 0.06892779538897799\n          },\n          {\n            "feature": "bwd_iat_mean",\n            "share_between_centroids": 0.067890089838014\n          },\n          {\n            "feature": "flow_duration",\n            "share_between_centroids": 0.06287261838219879\n          },\n          {\n            "feature": "flow_pkts_s",\n            "share_between_centroids": 0.06149442711609902\n          },\n          {\n            "feature": "flow_byts_s",\n            "share_between_centroids": 0.061485581507726456\n          },\n          {\n            "feature": "tot_bwd_pkts",\n            "share_between_centroids": 0.05867352115926721\n          },\n          {\n            "feature": "flow_iat_mean",\n            "share_between_centroids": 0.05824763952026458\n          },\n          {\n            "feature": "flow_iat_std",\n            "share_between_centroids": 0.057153496203609\n          },\n          {\n            "feature": "fwd_iat_mean",\n            "share_between_centroids": 0.05669788728773676\n          },\n          {\n            "feature": "ack_flag_cnt",\n            "share_between_centroids": 0.053910587465197723\n          },\n          {\n            "feature": "tot_fwd_pkts",\n            "share_between_centroids": 0.051573294462222294\n          },\n          {\n            "feature": "down_up_ratio",\n            "share_between_centroids": 0.03299269798319865\n          },\n          {\n            "feature": "rst_flag_cnt",\n            "share_between_centroids": 0.018241696706969762\n          },\n          {\n            "feature": "fin_flag_cnt",\n            "share_between_centroids": 0.0013642222072339724\n          }\n        ]\n      },\n      "calibration": {\n        "rows": 18099,\n        "cutoffs": {\n          "0.95": 13.37902829503273,\n          "0.99": 31.042924705057366,\n          "0.999": 263.37528662305107\n        },\n        "accuracy": 8192,\n        "reference_flags": {\n          "0.95": 905,\n          "0.99": 181,\n          "0.999": 20\n        }\n      },\n      "reference_occupancy": [\n        3313334,\n        4931789,\n        2115505,\n        602891,\n        5269424\n      ],\n      "retained_rows": 16232943,\n      "model": {\n        "path": "/kaggle/working/mmds_phase5/model",\n        "reload_rows": 32,\n        "sha256": {\n          "metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n          "metadata/part-00000-0c87b28e-f308-4221-be39-19552f15cae3-c000.txt": "30384fd3941c9003c0bbacc7539717ed7e2fec48b80eb398139246fded412610",\n          "stages/0_SQLTransformer_5ca5a59fde15/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n          "stages/0_SQLTransformer_5ca5a59fde15/metadata/part-00000-f99cdd32-8a0c-4326-8417-8d2ca3e2cc68-c000.txt": "a364092902ecb06292cf90c86f13b5c411ea08f9ad8a3f23c1bb998942ff32ff",\n          "stages/1_Imputer_dedda7ea3349/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n          "stages/1_Imputer_dedda7ea3349/data/part-00000-683c9d4f-e991-4b63-b365-8b9b862e5c47-c000.snappy.parquet": "94a3393daa471262d9eef2aaef21eaa4909412c36d022de91248ebe2872b9a95",\n          "stages/1_Imputer_dedda7ea3349/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n          "stages/1_Imputer_dedda7ea3349/metadata/part-00000-5049af53-561a-4641-b5e7-3c612198d0da-c000.txt": "7c60af0aafa39b4fe5733a5d5c5cae4ce489b8501ec29dd38e1bd7af8f6fa351",\n          "stages/2_VectorAssembler_087035b852be/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n          "stages/2_VectorAssembler_087035b852be/metadata/part-00000-6f5448aa-4153-4c62-81e1-9fda6620230c-c000.txt": "4cc48d033c823c24464618d6572157be7a0f8fe59c659a5d4fb7a564e219c774",\n          "stages/3_StandardScaler_0d1d4f35be62/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n          "stages/3_StandardScaler_0d1d4f35be62/data/part-00000-4b1de19e-2265-4edf-b7f2-896c2a90e676-c000.snappy.parquet": "304920b1e927c87dd9c996297666ae6a500243c0dc670a7fa2990feae29cb355",\n          "stages/3_StandardScaler_0d1d4f35be62/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n          "stages/3_StandardScaler_0d1d4f35be62/metadata/part-00000-0f5661c4-36ce-4d1d-aede-76690ea637c3-c000.txt": "594b3b43be877f97a70a89e0565da67ca71e492cbd583d013b08eb6b2350023d",\n          "stages/4_KMeans_aaeaaa224ba4/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n          "stages/4_KMeans_aaeaaa224ba4/data/part-00000-bff25530-0601-4c4c-8719-355553c18780-c000.snappy.parquet": "f2571f1c6417de4aa7554eee7a685e880e9421740bea8544315d1ed56a1a385a",\n          "stages/4_KMeans_aaeaaa224ba4/data/part-00001-bff25530-0601-4c4c-8719-355553c18780-c000.snappy.parquet": "4a52b62afb4efaa4d9e2108ababc2bab8710cd84c98731786d330ecee8212c3e",\n          "stages/4_KMeans_aaeaaa224ba4/data/part-00002-bff25530-0601-4c4c-8719-355553c18780-c000.snappy.parquet": "5c35ee55ec66c7e7e68a0b4237e53e979c756110af7b50864765805bd5e63874",\n          "stages/4_KMeans_aaeaaa224ba4/data/part-00003-bff25530-0601-4c4c-8719-355553c18780-c000.snappy.parquet": "8c2357343b3ca6dca74eeec01b686e9faacee83b77af8b8f6103af121a0011ea",\n          "stages/4_KMeans_aaeaaa224ba4/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n          "stages/4_KMeans_aaeaaa224ba4/metadata/part-00000-7939bfff-34cf-4268-9988-9efb2f086114-c000.txt": "ea809e26eae2de98fbb124fc66a5dc0b2a50003f3ba6493078ea119475de9a12"\n        },\n        "bytes": 26027\n      },\n      "elapsed_seconds": 578.076,\n      "figures": [\n        "phase5_g_cluster_sizes.png",\n        "phase5_g_cluster_attack_share.png",\n        "phase5_g_family_heatmap.png",\n        "phase5_g_daily_occupancy.png",\n        "phase5_g_temporal_shift.png",\n        "phase5_g_centroid_profiles.png",\n        "phase5_g_distance_histogram.png",\n        "phase5_g_daily_novelty.png",\n        "phase5_g_binary_novelty.png"\n      ],\n      "version": 1,\n      "log_events": 702,\n      "downloaded_reload": {\n        "probes": 3,\n        "pipeline_stages": 5,\n        "verified": true\n      }\n    },\n    "B": {\n      "experiment": "B",\n      "config": {\n        "version": "phase5-v1",\n        "seed": 23013,\n        "candidate_k": [\n          3,\n          5,\n          8,\n          12\n        ],\n        "max_iter": 16,\n        "init_steps": 2,\n        "tolerance": 0.0001,\n        "training_permyriad": 250,\n        "internal_permyriad": 12,\n        "reference_percentiles": [\n          0.95,\n          0.99,\n          0.999\n        ],\n        "primary_percentile": 0.99,\n        "quantile_accuracy": 8192,\n        "max_internal_rows": 40000,\n        "rare_support": 100,\n        "window_hours": 3,\n        "lag_reference": "full retained phase4 train-day population",\n        "spark": {\n          "master": "local[4]",\n          "driver_memory": "8g",\n          "shuffle_partitions": 48,\n          "input_partition_bytes": 67108864\n        }\n      },\n      "status": "COMPLETE",\n      "resources": {\n        "spark_version": "4.3.0",\n        "master": "local[4]",\n        "driver_memory": "8g",\n        "shuffle_partitions": 48,\n        "input_partition_bytes": 67108864,\n        "default_parallelism": 4,\n        "cpu_affinity": 4,\n        "scratch_output_free_bytes_start": 20939968512,\n        "memory.max": "32212254720",\n        "cpu.max": "400000 100000",\n        "driver_peak_rss_bytes": 5523873792\n      },\n      "features": [\n        "flow_duration",\n        "tot_fwd_pkts",\n        "tot_bwd_pkts",\n        "totlen_fwd_pkts",\n        "totlen_bwd_pkts",\n        "pkt_len_mean",\n        "pkt_len_std",\n        "flow_iat_mean",\n        "flow_iat_std",\n        "fwd_iat_mean",\n        "bwd_iat_mean",\n        "flow_pkts_s",\n        "flow_byts_s",\n        "down_up_ratio",\n        "syn_flag_cnt",\n        "ack_flag_cnt",\n        "rst_flag_cnt",\n        "fin_flag_cnt"\n      ],\n      "input": {\n        "path": "/kaggle/input/notebooks/ramcharan2345/mmds-phase-2-spark-data-engineering/mmds_phase2/phase2-v1/feature_ready",\n        "days": [\n          "2018-02-14",\n          "2018-02-15",\n          "2018-02-16",\n          "2018-02-20",\n          "2018-02-21",\n          "2018-02-22",\n          "2018-02-23",\n          "2018-02-28",\n          "2018-03-01",\n          "2018-03-02"\n        ],\n        "totals": {\n          "0": 13484708,\n          "1": 2748235\n        },\n        "rows": 16232943,\n        "temporally_flagged": 14,\n        "policy_version": "phase2-v1",\n        "receipt_sha256": "1fa00cd2b9c853ee33eaacee8ca7632dbb41f739573326280d6ffbfb2ccff25a",\n        "producer": {\n          "url": "https://www.kaggle.com/code/ramcharan2345/mmds-phase-2-spark-data-engineering",\n          "kernel_version": 1,\n          "status": "COMPLETE"\n        }\n      },\n      "purge": {\n        "groups_before": 136985,\n        "groups_after": 0,\n        "rows": 909956\n      },\n      "matched_phase4": {\n        "protocol": "B",\n        "kaggle_version": "phase4-v1",\n        "checked": "matched split rules and family/binary totals",\n        "purged_rows": 909956\n      },\n      "preprocessing": {\n        "transform": "signed_log1p",\n        "fit_columns": [\n          "flow_duration",\n          "tot_fwd_pkts",\n          "tot_bwd_pkts",\n          "totlen_fwd_pkts",\n          "totlen_bwd_pkts",\n          "pkt_len_mean",\n          "pkt_len_std",\n          "flow_iat_mean",\n          "flow_iat_std",\n          "fwd_iat_mean",\n          "bwd_iat_mean",\n          "flow_pkts_s",\n          "flow_byts_s",\n          "down_up_ratio",\n          "syn_flag_cnt",\n          "ack_flag_cnt",\n          "rst_flag_cnt",\n          "fin_flag_cnt"\n        ],\n        "active": [\n          "flow_duration",\n          "tot_fwd_pkts",\n          "tot_bwd_pkts",\n          "totlen_fwd_pkts",\n          "totlen_bwd_pkts",\n          "pkt_len_mean",\n          "pkt_len_std",\n          "flow_iat_mean",\n          "flow_iat_std",\n          "fwd_iat_mean",\n          "bwd_iat_mean",\n          "flow_pkts_s",\n          "flow_byts_s",\n          "down_up_ratio",\n          "syn_flag_cnt",\n          "ack_flag_cnt",\n          "rst_flag_cnt",\n          "fin_flag_cnt"\n        ],\n        "dropped": [],\n        "reference_stats": {\n          "flow_duration": {\n            "invalid_or_null": 2,\n            "available": true,\n            "reference_std": 4.922656482085623,\n            "near_constant": false\n          },\n          "tot_fwd_pkts": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 0.7472669479236256,\n            "near_constant": false\n          },\n          "tot_bwd_pkts": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 0.9304382766169819,\n            "near_constant": false\n          },\n          "totlen_fwd_pkts": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 2.6960342991497512,\n            "near_constant": false\n          },\n          "totlen_bwd_pkts": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 3.1685009030872178,\n            "near_constant": false\n          },\n          "pkt_len_mean": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 2.1105019819450357,\n            "near_constant": false\n          },\n          "pkt_len_std": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 2.3422247994945606,\n            "near_constant": false\n          },\n          "flow_iat_mean": {\n            "invalid_or_null": 2,\n            "available": true,\n            "reference_std": 4.160112954121857,\n            "near_constant": false\n          },\n          "flow_iat_std": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 6.278238645416323,\n            "near_constant": false\n          },\n          "fwd_iat_mean": {\n            "invalid_or_null": 2,\n            "available": true,\n            "reference_std": 6.013749976297104,\n            "near_constant": false\n          },\n          "bwd_iat_mean": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 6.151272138517253,\n            "near_constant": false\n          },\n          "flow_pkts_s": {\n            "invalid_or_null": 7,\n            "available": true,\n            "reference_std": 3.840823380586106,\n            "near_constant": false\n          },\n          "flow_byts_s": {\n            "invalid_or_null": 5,\n            "available": true,\n            "reference_std": 4.985637883459185,\n            "near_constant": false\n          },\n          "down_up_ratio": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 0.3555765408371492,\n            "near_constant": false\n          },\n          "syn_flag_cnt": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 0.1506367131351718,\n            "near_constant": false\n          },\n          "ack_flag_cnt": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 0.3294001170930551,\n            "near_constant": false\n          },\n          "rst_flag_cnt": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 0.27314421194577493,\n            "near_constant": false\n          },\n          "fin_flag_cnt": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 0.042873913468058716,\n            "near_constant": false\n          }\n        },\n        "imputation_surrogates": {\n          "flow_duration_valid": 10.290483144360632,\n          "tot_fwd_pkts_valid": 1.0986122886681096,\n          "tot_bwd_pkts_valid": 0.6931471805599453,\n          "totlen_fwd_pkts_valid": 3.828641396489095,\n          "totlen_bwd_pkts_valid": 4.6913478822291435,\n          "pkt_len_mean_valid": 4.013375499688434,\n          "pkt_len_std_valid": 3.589566349852494,\n          "flow_iat_mean_valid": 9.566615235893575,\n          "flow_iat_std_valid": 4.7100445409878064,\n          "fwd_iat_mean_valid": 8.769040810858803,\n          "bwd_iat_mean_valid": 0.0,\n          "flow_pkts_s_valid": 4.726508071127536,\n          "flow_byts_s_valid": 6.752625541472987,\n          "down_up_ratio_valid": 0.0,\n          "syn_flag_cnt_valid": 0.0,\n          "ack_flag_cnt_valid": 0.0,\n          "rst_flag_cnt_valid": 0.0,\n          "fin_flag_cnt_valid": 0.0\n        },\n        "reference_means": [\n          10.62231300764484,\n          1.4363484754094262,\n          1.0798649837457388,\n          3.743066179733096,\n          3.946154886258588,\n          3.1716542466921442,\n          3.210042151135082,\n          9.465636467739502,\n          6.1415758897976875,\n          7.789394109351316,\n          5.044616165162009,\n          5.1301504432809875,\n          6.431092524929756,\n          0.3392943193650265,\n          0.03444876676732746,\n          0.2388277905479885,\n          0.1332528752834452,\n          0.0026621383032741515\n        ],\n        "reference_scales": [\n          4.922656482085623,\n          0.7472669479236256,\n          0.9304382766169819,\n          2.6960342991497512,\n          3.1685009030872178,\n          2.1105019819450357,\n          2.3422247994945606,\n          4.160112954121857,\n          6.278238645416323,\n          6.013749976297104,\n          6.151272138517253,\n          3.840823380586106,\n          4.985637883459185,\n          0.3555765408371492,\n          0.1506367131351718,\n          0.3294001170930551,\n          0.27314421194577493,\n          0.042873913468058716\n        ]\n      },\n      "training": {\n        "fit_rows": 318175,\n        "internal_rows": 14426,\n        "reference_rows": 13191622,\n        "share_of_reference": 0.024119475224502338\n      },\n      "selected_k": 12,\n      "trials": [\n        {\n          "k": 3,\n          "seed": 23013,\n          "initialization": "k-means||",\n          "budget": 16,\n          "iterations": 12,\n          "wssse": 2623770.7352790604,\n          "cluster_sizes": [\n            114305,\n            93389,\n            110481\n          ],\n          "training_rows": 318175,\n          "empty_clusters": 0,\n          "smallest_share": 0.2935145753123281,\n          "fit_seconds": 16.727,\n          "jvm_peak_rss_bytes": 5518626816,\n          "silhouette": 0.5905867376279668,\n          "silhouette_rows": 14426\n        },\n        {\n          "k": 5,\n          "seed": 23013,\n          "initialization": "k-means||",\n          "budget": 16,\n          "iterations": 16,\n          "wssse": 2043482.959962206,\n          "cluster_sizes": [\n            48817,\n            53488,\n            92308,\n            45527,\n            78035\n          ],\n          "training_rows": 318175,\n          "empty_clusters": 0,\n          "smallest_share": 0.14308792331264242,\n          "fit_seconds": 13.498,\n          "jvm_peak_rss_bytes": 5523873792,\n          "silhouette": 0.3828341860007618,\n          "silhouette_rows": 14426\n        },\n        {\n          "k": 8,\n          "seed": 23013,\n          "initialization": "k-means||",\n          "budget": 16,\n          "iterations": 16,\n          "wssse": 1495383.7629115605,\n          "cluster_sizes": [\n            27677,\n            61084,\n            74004,\n            13520,\n            20308,\n            38236,\n            12669,\n            70677\n          ],\n          "training_rows": 318175,\n          "empty_clusters": 0,\n          "smallest_share": 0.03981771037950813,\n          "fit_seconds": 13.774,\n          "jvm_peak_rss_bytes": 5523873792,\n          "silhouette": 0.5347387017919628,\n          "silhouette_rows": 14426\n        },\n        {\n          "k": 12,\n          "seed": 23013,\n          "initialization": "k-means||",\n          "budget": 16,\n          "iterations": 16,\n          "wssse": 826131.3847364823,\n          "cluster_sizes": [\n            72053,\n            32416,\n            39207,\n            34233,\n            24969,\n            1222,\n            20238,\n            12679,\n            24741,\n            3078,\n            36378,\n            16961\n          ],\n          "training_rows": 318175,\n          "empty_clusters": 0,\n          "smallest_share": 0.003840653728294178,\n          "fit_seconds": 13.601,\n          "jvm_peak_rss_bytes": 5523873792,\n          "silhouette": 0.6288732335989271,\n          "silhouette_rows": 14426\n        }\n      ],\n      "robustness": {\n        "paired_rows": 14426,\n        "seed_ari": 0.8503551767126171,\n        "scale_ari": 0.21828525609211355,\n        "alternate_seed": {\n          "k": 12,\n          "seed": 23014,\n          "initialization": "k-means||",\n          "budget": 16,\n          "iterations": 6,\n          "wssse": 824197.6855591865,\n          "cluster_sizes": [\n            72027,\n            54502,\n            16303,\n            16915,\n            24510,\n            34164,\n            1222,\n            20983,\n            28429,\n            15991,\n            13490,\n            19639\n          ],\n          "training_rows": 318175,\n          "empty_clusters": 0,\n          "smallest_share": 0.003840653728294178,\n          "fit_seconds": 8.282,\n          "jvm_peak_rss_bytes": 5523873792\n        },\n        "linear": {\n          "k": 12,\n          "seed": 23013,\n          "initialization": "k-means||",\n          "budget": 16,\n          "iterations": 13,\n          "wssse": 1426972.6993342037,\n          "cluster_sizes": [\n            177802,\n            73,\n            9,\n            2,\n            3597,\n            1222,\n            9,\n            59,\n            28256,\n            1903,\n            93492,\n            11751\n          ],\n          "training_rows": 318175,\n          "empty_clusters": 0,\n          "smallest_share": 6.285848982478196e-06,\n          "fit_seconds": 10.084,\n          "jvm_peak_rss_bytes": 5523873792\n        },\n        "linear_silhouette": 0.5263812389664033,\n        "dominance": [\n          {\n            "feature": "fin_flag_cnt",\n            "share_between_centroids": 0.0649186466394683\n          },\n          {\n            "feature": "syn_flag_cnt",\n            "share_between_centroids": 0.06467752364561152\n          },\n          {\n            "feature": "rst_flag_cnt",\n            "share_between_centroids": 0.06211008232263305\n          },\n          {\n            "feature": "totlen_bwd_pkts",\n            "share_between_centroids": 0.06023091408924288\n          },\n          {\n            "feature": "pkt_len_mean",\n            "share_between_centroids": 0.06006028513836801\n          },\n          {\n            "feature": "fwd_iat_mean",\n            "share_between_centroids": 0.0592081436439001\n          },\n          {\n            "feature": "pkt_len_std",\n            "share_between_centroids": 0.05853597773311246\n          },\n          {\n            "feature": "totlen_fwd_pkts",\n            "share_between_centroids": 0.057959339994185476\n          },\n          {\n            "feature": "bwd_iat_mean",\n            "share_between_centroids": 0.0566050946292509\n          },\n          {\n            "feature": "flow_duration",\n            "share_between_centroids": 0.05623942273305119\n          },\n          {\n            "feature": "tot_bwd_pkts",\n            "share_between_centroids": 0.053953694740570934\n          },\n          {\n            "feature": "flow_pkts_s",\n            "share_between_centroids": 0.05347987389437597\n          },\n          {\n            "feature": "flow_iat_mean",\n            "share_between_centroids": 0.05341174083631976\n          },\n          {\n            "feature": "flow_iat_std",\n            "share_between_centroids": 0.05215410249811013\n          },\n          {\n            "feature": "tot_fwd_pkts",\n            "share_between_centroids": 0.05032721067251503\n          },\n          {\n            "feature": "flow_byts_s",\n            "share_between_centroids": 0.050274400604878325\n          },\n          {\n            "feature": "ack_flag_cnt",\n            "share_between_centroids": 0.046186617034507854\n          },\n          {\n            "feature": "down_up_ratio",\n            "share_between_centroids": 0.039666929149898084\n          }\n        ]\n      },\n      "calibration": {\n        "rows": 14426,\n        "cutoffs": {\n          "0.95": 8.508047634376355,\n          "0.99": 16.67940564007534,\n          "0.999": 35.317581821449224\n        },\n        "accuracy": 8192,\n        "reference_flags": {\n          "0.95": 721,\n          "0.99": 143,\n          "0.999": 16\n        }\n      },\n      "reference_occupancy": [\n        2946330,\n        1352784,\n        1953604,\n        1358963,\n        1004912,\n        62483,\n        806475,\n        453967,\n        988482,\n        122476,\n        1460945,\n        680201\n      ],\n      "retained_rows": 15322987,\n      "model": {\n        "path": "/kaggle/working/mmds_phase5/model",\n        "reload_rows": 32,\n        "sha256": {\n          "metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n          "metadata/part-00000-2c0d5e62-616a-4fc8-86af-688d8012e022-c000.txt": "c81ff6454592ebfb717ba9d802bcfc73d514e52328a55f7dcc106acc27560ad8",\n          "stages/0_SQLTransformer_c1b46ce0d74d/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n          "stages/0_SQLTransformer_c1b46ce0d74d/metadata/part-00000-a8d654ae-ad50-4736-8af3-d221a4522fe2-c000.txt": "c2f875fe3ca0ba7a7f2525d5b085a87f21af7e6c5ab95792418c2f1885fb8277",\n          "stages/1_Imputer_ded6c4fea5cf/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n          "stages/1_Imputer_ded6c4fea5cf/data/part-00000-fb2041e8-1fed-41d8-8b3b-65fc52e86496-c000.snappy.parquet": "9acec6d9df3d2655f3b8388f76a96fa6ade0f42a231f65a2b156629ce12f381c",\n          "stages/1_Imputer_ded6c4fea5cf/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n          "stages/1_Imputer_ded6c4fea5cf/metadata/part-00000-8e883650-80b9-4d1a-a475-fc501557da88-c000.txt": "6282b8ca24229c2232c480a74a042661120568614ef32325223e4d1c5ac5dd14",\n          "stages/2_VectorAssembler_09b31eb4789b/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n          "stages/2_VectorAssembler_09b31eb4789b/metadata/part-00000-7bf6b8c7-cfe0-4be3-8aad-57fe393e6647-c000.txt": "7761b8b94869ce3c7868118827c1538608bfa271171e0f5cc7b7c0b7f59dda53",\n          "stages/3_StandardScaler_140d8fdfd488/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n          "stages/3_StandardScaler_140d8fdfd488/data/part-00000-8c91b2ae-2f4e-4d80-b87f-af347bc89a75-c000.snappy.parquet": "5d40366e6580ccde43123e9f86beb790d91159bef135c806f94425a64857015a",\n          "stages/3_StandardScaler_140d8fdfd488/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n          "stages/3_StandardScaler_140d8fdfd488/metadata/part-00000-b0ea8d09-fb66-4124-bcb2-765abceea562-c000.txt": "bf39baf525e8a513c90ac1deab4bbb394a4382de91a4a3cb4e4e891eea40cbf8",\n          "stages/4_KMeans_827890df35c6/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n          "stages/4_KMeans_827890df35c6/data/part-00000-9b1c962b-9b22-4a80-bdeb-b7164a439669-c000.snappy.parquet": "292de4cee2346e4b1020503e93e2c96a96282d815f46bdeb9b91671ec991ef5d",\n          "stages/4_KMeans_827890df35c6/data/part-00001-9b1c962b-9b22-4a80-bdeb-b7164a439669-c000.snappy.parquet": "2e119fb76d03b9d67d89562135ad5d63bf27a2495cc10f212024eba6609ff785",\n          "stages/4_KMeans_827890df35c6/data/part-00002-9b1c962b-9b22-4a80-bdeb-b7164a439669-c000.snappy.parquet": "05eebc0eb5be2c9849e72daf13e63a30719b8b56abfd3850e1a9cf5411c2a89a",\n          "stages/4_KMeans_827890df35c6/data/part-00003-9b1c962b-9b22-4a80-bdeb-b7164a439669-c000.snappy.parquet": "39ba8b20846d0e2fb2afb79073094501af559e0bd52defa4ce55c92c93222532",\n          "stages/4_KMeans_827890df35c6/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n          "stages/4_KMeans_827890df35c6/metadata/part-00000-c17af254-4358-4e37-b28e-62545b7ff7f7-c000.txt": "8e82b0ef89fcfa118bcacd2b0b2862cc88104fdbf882126fd209e411efa73fc8"\n        },\n        "bytes": 27179\n      },\n      "elapsed_seconds": 652.492,\n      "figures": [\n        "phase5_b_cluster_sizes.png",\n        "phase5_b_cluster_attack_share.png",\n        "phase5_b_family_heatmap.png",\n        "phase5_b_daily_occupancy.png",\n        "phase5_b_temporal_shift.png",\n        "phase5_b_centroid_profiles.png",\n        "phase5_b_distance_histogram.png",\n        "phase5_b_daily_novelty.png",\n        "phase5_b_binary_novelty.png",\n        "phase5_b_supervised_comparison.png"\n      ],\n      "version": 2,\n      "log_events": 735,\n      "downloaded_reload": {\n        "probes": 3,\n        "pipeline_stages": 5,\n        "verified": true\n      }\n    },\n    "C": {\n      "experiment": "C",\n      "config": {\n        "version": "phase5-v1",\n        "seed": 23013,\n        "candidate_k": [\n          3,\n          5,\n          8,\n          12\n        ],\n        "max_iter": 16,\n        "init_steps": 2,\n        "tolerance": 0.0001,\n        "training_permyriad": 250,\n        "internal_permyriad": 12,\n        "reference_percentiles": [\n          0.95,\n          0.99,\n          0.999\n        ],\n        "primary_percentile": 0.99,\n        "quantile_accuracy": 8192,\n        "max_internal_rows": 40000,\n        "rare_support": 100,\n        "window_hours": 3,\n        "lag_reference": "full retained phase4 train-day population",\n        "spark": {\n          "master": "local[4]",\n          "driver_memory": "8g",\n          "shuffle_partitions": 48,\n          "input_partition_bytes": 67108864\n        }\n      },\n      "status": "COMPLETE",\n      "resources": {\n        "spark_version": "4.3.0",\n        "master": "local[4]",\n        "driver_memory": "8g",\n        "shuffle_partitions": 48,\n        "input_partition_bytes": 67108864,\n        "default_parallelism": 4,\n        "cpu_affinity": 4,\n        "scratch_output_free_bytes_start": 20939968512,\n        "memory.max": "32212254720",\n        "cpu.max": "400000 100000",\n        "driver_peak_rss_bytes": 4567228416\n      },\n      "features": [\n        "flow_duration",\n        "tot_fwd_pkts",\n        "tot_bwd_pkts",\n        "totlen_fwd_pkts",\n        "totlen_bwd_pkts",\n        "pkt_len_mean",\n        "pkt_len_std",\n        "flow_iat_mean",\n        "flow_iat_std",\n        "fwd_iat_mean",\n        "bwd_iat_mean",\n        "flow_pkts_s",\n        "flow_byts_s",\n        "down_up_ratio",\n        "syn_flag_cnt",\n        "ack_flag_cnt",\n        "rst_flag_cnt",\n        "fin_flag_cnt"\n      ],\n      "input": {\n        "path": "/kaggle/input/notebooks/ramcharan2345/mmds-phase-2-spark-data-engineering/mmds_phase2/phase2-v1/feature_ready",\n        "days": [\n          "2018-02-14",\n          "2018-02-15",\n          "2018-02-16",\n          "2018-02-20",\n          "2018-02-21",\n          "2018-02-22",\n          "2018-02-23",\n          "2018-02-28",\n          "2018-03-01",\n          "2018-03-02"\n        ],\n        "totals": {\n          "0": 13484708,\n          "1": 2748235\n        },\n        "rows": 16232943,\n        "temporally_flagged": 14,\n        "policy_version": "phase2-v1",\n        "receipt_sha256": "1fa00cd2b9c853ee33eaacee8ca7632dbb41f739573326280d6ffbfb2ccff25a",\n        "producer": {\n          "url": "https://www.kaggle.com/code/ramcharan2345/mmds-phase-2-spark-data-engineering",\n          "kernel_version": 1,\n          "status": "COMPLETE"\n        }\n      },\n      "purge": {\n        "groups_before": 142016,\n        "groups_after": 0,\n        "rows": 859101\n      },\n      "matched_phase4": {\n        "protocol": "C",\n        "kaggle_version": "phase4-v1",\n        "checked": "matched split rules and family/binary totals",\n        "purged_rows": 859101\n      },\n      "preprocessing": {\n        "transform": "signed_log1p",\n        "fit_columns": [\n          "flow_duration",\n          "tot_fwd_pkts",\n          "tot_bwd_pkts",\n          "totlen_fwd_pkts",\n          "totlen_bwd_pkts",\n          "pkt_len_mean",\n          "pkt_len_std",\n          "flow_iat_mean",\n          "flow_iat_std",\n          "fwd_iat_mean",\n          "bwd_iat_mean",\n          "flow_pkts_s",\n          "flow_byts_s",\n          "down_up_ratio",\n          "syn_flag_cnt",\n          "ack_flag_cnt",\n          "rst_flag_cnt",\n          "fin_flag_cnt"\n        ],\n        "active": [\n          "flow_duration",\n          "tot_fwd_pkts",\n          "tot_bwd_pkts",\n          "totlen_fwd_pkts",\n          "totlen_bwd_pkts",\n          "pkt_len_mean",\n          "pkt_len_std",\n          "flow_iat_mean",\n          "flow_iat_std",\n          "fwd_iat_mean",\n          "bwd_iat_mean",\n          "flow_pkts_s",\n          "flow_byts_s",\n          "down_up_ratio",\n          "syn_flag_cnt",\n          "ack_flag_cnt",\n          "rst_flag_cnt",\n          "fin_flag_cnt"\n        ],\n        "dropped": [],\n        "reference_stats": {\n          "flow_duration": {\n            "invalid_or_null": 1,\n            "available": true,\n            "reference_std": 4.869136559640692,\n            "near_constant": false\n          },\n          "tot_fwd_pkts": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 0.7456033837944145,\n            "near_constant": false\n          },\n          "tot_bwd_pkts": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 0.9269881226653868,\n            "near_constant": false\n          },\n          "totlen_fwd_pkts": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 2.6984430402165542,\n            "near_constant": false\n          },\n          "totlen_bwd_pkts": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 3.155665732457084,\n            "near_constant": false\n          },\n          "pkt_len_mean": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 2.1063324960911207,\n            "near_constant": false\n          },\n          "pkt_len_std": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 2.3411929848797985,\n            "near_constant": false\n          },\n          "flow_iat_mean": {\n            "invalid_or_null": 1,\n            "available": true,\n            "reference_std": 4.093219288443269,\n            "near_constant": false\n          },\n          "flow_iat_std": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 6.251545398563713,\n            "near_constant": false\n          },\n          "fwd_iat_mean": {\n            "invalid_or_null": 1,\n            "available": true,\n            "reference_std": 5.957646239775045,\n            "near_constant": false\n          },\n          "bwd_iat_mean": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 6.135597572259061,\n            "near_constant": false\n          },\n          "flow_pkts_s": {\n            "invalid_or_null": 6,\n            "available": true,\n            "reference_std": 3.792265648088158,\n            "near_constant": false\n          },\n          "flow_byts_s": {\n            "invalid_or_null": 5,\n            "available": true,\n            "reference_std": 4.950192311247816,\n            "near_constant": false\n          },\n          "down_up_ratio": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 0.35636566021738614,\n            "near_constant": false\n          },\n          "syn_flag_cnt": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 0.14768589130246518,\n            "near_constant": false\n          },\n          "ack_flag_cnt": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 0.3276131152096508,\n            "near_constant": false\n          },\n          "rst_flag_cnt": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 0.2803962629750321,\n            "near_constant": false\n          },\n          "fin_flag_cnt": {\n            "invalid_or_null": 0,\n            "available": true,\n            "reference_std": 0.0437856949005969,\n            "near_constant": false\n          }\n        },\n        "imputation_surrogates": {\n          "flow_duration_valid": 10.176030606145105,\n          "tot_fwd_pkts_valid": 1.0986122886681096,\n          "tot_bwd_pkts_valid": 0.6931471805599453,\n          "totlen_fwd_pkts_valid": 3.8501476017100584,\n          "totlen_bwd_pkts_valid": 4.727387818712341,\n          "pkt_len_mean_valid": 4.031286426195792,\n          "pkt_len_std_valid": 3.5895663498534005,\n          "flow_iat_mean_valid": 9.478074806618267,\n          "flow_iat_std_valid": 4.992953920547845,\n          "fwd_iat_mean_valid": 8.564458383883352,\n          "bwd_iat_mean_valid": 0.0,\n          "flow_pkts_s_valid": 4.782380974650369,\n          "flow_byts_s_valid": 6.818838181259726,\n          "down_up_ratio_valid": 0.0,\n          "syn_flag_cnt_valid": 0.0,\n          "ack_flag_cnt_valid": 0.0,\n          "rst_flag_cnt_valid": 0.0,\n          "fin_flag_cnt_valid": 0.0\n        },\n        "reference_means": [\n          10.592229755632966,\n          1.4377055249204316,\n          1.0922496697488846,\n          3.7688694528572517,\n          3.9684904565450854,\n          3.181751208753493,\n          3.229336814616343,\n          9.42202464458342,\n          6.200338594152421,\n          7.74548661256098,\n          5.107133262762929,\n          5.150160890621316,\n          6.445406910744578,\n          0.34414538869010725,\n          0.033041776321702514,\n          0.2335106354098546,\n          0.142879132693334,\n          0.0027770336890388905\n        ],\n        "reference_scales": [\n          4.869136559640692,\n          0.7456033837944145,\n          0.9269881226653868,\n          2.6984430402165542,\n          3.155665732457084,\n          2.1063324960911207,\n          2.3411929848797985,\n          4.093219288443269,\n          6.251545398563713,\n          5.957646239775045,\n          6.135597572259061,\n          3.792265648088158,\n          4.950192311247816,\n          0.35636566021738614,\n          0.14768589130246518,\n          0.3276131152096508,\n          0.2803962629750321,\n          0.0437856949005969\n        ]\n      },\n      "training": {\n        "fit_rows": 323731,\n        "internal_rows": 15229,\n        "reference_rows": 13522722,\n        "share_of_reference": 0.02393978076307418\n      },\n      "selected_k": 12,\n      "trials": [\n        {\n          "k": 3,\n          "seed": 23013,\n          "initialization": "k-means||",\n          "budget": 16,\n          "iterations": 9,\n          "wssse": 2679253.389944727,\n          "cluster_sizes": [\n            114634,\n            93969,\n            115128\n          ],\n          "training_rows": 323731,\n          "empty_clusters": 0,\n          "smallest_share": 0.2902687725302798,\n          "fit_seconds": 17.529,\n          "jvm_peak_rss_bytes": 3822231552,\n          "silhouette": 0.576516009487571,\n          "silhouette_rows": 15229\n        },\n        {\n          "k": 5,\n          "seed": 23013,\n          "initialization": "k-means||",\n          "budget": 16,\n          "iterations": 16,\n          "wssse": 2126996.270917423,\n          "cluster_sizes": [\n            113300,\n            23166,\n            114475,\n            33764,\n            39026\n          ],\n          "training_rows": 323731,\n          "empty_clusters": 0,\n          "smallest_share": 0.0715594119809348,\n          "fit_seconds": 14.797,\n          "jvm_peak_rss_bytes": 3822231552,\n          "silhouette": 0.5301041497841288,\n          "silhouette_rows": 15229\n        },\n        {\n          "k": 8,\n          "seed": 23013,\n          "initialization": "k-means||",\n          "budget": 16,\n          "iterations": 16,\n          "wssse": 1494304.9269670998,\n          "cluster_sizes": [\n            33142,\n            44336,\n            72006,\n            92349,\n            3193,\n            49749,\n            16776,\n            12180\n          ],\n          "training_rows": 323731,\n          "empty_clusters": 0,\n          "smallest_share": 0.009863127102439988,\n          "fit_seconds": 15.078,\n          "jvm_peak_rss_bytes": 3822231552,\n          "silhouette": 0.5244041845116935,\n          "silhouette_rows": 15229\n        },\n        {\n          "k": 12,\n          "seed": 23013,\n          "initialization": "k-means||",\n          "budget": 16,\n          "iterations": 11,\n          "wssse": 856400.4034315124,\n          "cluster_sizes": [\n            14243,\n            73439,\n            17883,\n            31132,\n            42530,\n            21892,\n            1297,\n            45116,\n            29322,\n            12352,\n            14601,\n            19924\n          ],\n          "training_rows": 323731,\n          "empty_clusters": 0,\n          "smallest_share": 0.004006412731557991,\n          "fit_seconds": 12.038,\n          "jvm_peak_rss_bytes": 3822231552,\n          "silhouette": 0.6029164097465304,\n          "silhouette_rows": 15229\n        }\n      ],\n      "robustness": {\n        "paired_rows": 15229,\n        "seed_ari": 0.7973700368914762,\n        "scale_ari": 0.4201543819333072,\n        "alternate_seed": {\n          "k": 12,\n          "seed": 23014,\n          "initialization": "k-means||",\n          "budget": 16,\n          "iterations": 9,\n          "wssse": 842656.9529380443,\n          "cluster_sizes": [\n            54239,\n            73342,\n            42519,\n            21892,\n            20582,\n            1297,\n            16207,\n            14455,\n            35981,\n            12293,\n            13091,\n            17833\n          ],\n          "training_rows": 323731,\n          "empty_clusters": 0,\n          "smallest_share": 0.004006412731557991,\n          "fit_seconds": 10.729,\n          "jvm_peak_rss_bytes": 3822231552\n        },\n        "linear": {\n          "k": 12,\n          "seed": 23013,\n          "initialization": "k-means||",\n          "budget": 16,\n          "iterations": 16,\n          "wssse": 1390337.5175443299,\n          "cluster_sizes": [\n            17831,\n            10793,\n            1,\n            129627,\n            93981,\n            70,\n            1297,\n            59,\n            1828,\n            5727,\n            2936,\n            59581\n          ],\n          "training_rows": 323731,\n          "empty_clusters": 0,\n          "smallest_share": 3.0889843728280578e-06,\n          "fit_seconds": 11.83,\n          "jvm_peak_rss_bytes": 3822231552\n        },\n        "linear_silhouette": 0.5941173468373271,\n        "dominance": [\n          {\n            "feature": "fin_flag_cnt",\n            "share_between_centroids": 0.06512712921503346\n          },\n          {\n            "feature": "pkt_len_mean",\n            "share_between_centroids": 0.06047417836859744\n          },\n          {\n            "feature": "totlen_bwd_pkts",\n            "share_between_centroids": 0.06031783885207001\n          },\n          {\n            "feature": "fwd_iat_mean",\n            "share_between_centroids": 0.06002741298235094\n          },\n          {\n            "feature": "pkt_len_std",\n            "share_between_centroids": 0.05903461994649836\n          },\n          {\n            "feature": "totlen_fwd_pkts",\n            "share_between_centroids": 0.05887461070819318\n          },\n          {\n            "feature": "flow_iat_std",\n            "share_between_centroids": 0.0586375097921818\n          },\n          {\n            "feature": "ack_flag_cnt",\n            "share_between_centroids": 0.05840868791776876\n          },\n          {\n            "feature": "bwd_iat_mean",\n            "share_between_centroids": 0.0566957798062644\n          },\n          {\n            "feature": "tot_bwd_pkts",\n            "share_between_centroids": 0.05657030407659864\n          },\n          {\n            "feature": "flow_duration",\n            "share_between_centroids": 0.055108822297945156\n          },\n          {\n            "feature": "syn_flag_cnt",\n            "share_between_centroids": 0.053792429388319865\n          },\n          {\n            "feature": "flow_byts_s",\n            "share_between_centroids": 0.05315918925861557\n          },\n          {\n            "feature": "tot_fwd_pkts",\n            "share_between_centroids": 0.05271965026062504\n          },\n          {\n            "feature": "flow_iat_mean",\n            "share_between_centroids": 0.05141474904055002\n          },\n          {\n            "feature": "flow_pkts_s",\n            "share_between_centroids": 0.051252188351870925\n          },\n          {\n            "feature": "down_up_ratio",\n            "share_between_centroids": 0.04843469529817226\n          },\n          {\n            "feature": "rst_flag_cnt",\n            "share_between_centroids": 0.039950204438344156\n          }\n        ]\n      },\n      "calibration": {\n        "rows": 15229,\n        "cutoffs": {\n          "0.95": 8.43342825481104,\n          "0.99": 17.880003619372005,\n          "0.999": 42.38438316542496\n        },\n        "accuracy": 8192,\n        "reference_flags": {\n          "0.95": 761,\n          "0.99": 153,\n          "0.999": 17\n        }\n      },\n      "reference_occupancy": [\n        574002,\n        2964362,\n        718148,\n        1245715,\n        2195139,\n        952319,\n        62942,\n        1796161,\n        1172002,\n        468073,\n        585595,\n        788264\n      ],\n      "retained_rows": 15373842,\n      "model": {\n        "path": "/kaggle/working/mmds_phase5/model",\n        "reload_rows": 32,\n        "sha256": {\n          "metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n          "metadata/part-00000-4b0ee0c7-d84b-45e3-9817-206310f7c865-c000.txt": "9da180e2ce2ef7d0b4804a19a59f622c064abc2bf8890b332304ca3d406a2082",\n          "stages/0_SQLTransformer_de35da705442/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n          "stages/0_SQLTransformer_de35da705442/metadata/part-00000-65a40e82-9196-455e-8074-c37a6c2bd711-c000.txt": "6d1f231457c948d2f6446014349f659e2e7a9d718039ae040ce58598b33bd503",\n          "stages/1_Imputer_5f45b06a8ef8/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n          "stages/1_Imputer_5f45b06a8ef8/data/part-00000-9f75178b-2fee-47c9-9f70-3f7efeea53cf-c000.snappy.parquet": "7eedbba3a84d32352b0d3e289fc1e63f361ab9cc197456e3623af4c7410b4929",\n          "stages/1_Imputer_5f45b06a8ef8/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n          "stages/1_Imputer_5f45b06a8ef8/metadata/part-00000-1036018f-36e2-483e-8cc0-faf518e5e67c-c000.txt": "757957c77896c481bfbb67d4f44d5b59d75415d16553799068c7c13915847962",\n          "stages/2_VectorAssembler_676a48355ddb/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n          "stages/2_VectorAssembler_676a48355ddb/metadata/part-00000-e14a109c-8af8-4654-8154-6e9161cd951a-c000.txt": "7b960351421f31b84e2e0e38f691e7ce874de3903721b6a00b857f398a995229",\n          "stages/3_StandardScaler_243daebf2f5b/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n          "stages/3_StandardScaler_243daebf2f5b/data/part-00000-9fe65596-b9fc-4519-888a-2319358c49ee-c000.snappy.parquet": "bb04a58adf0ba3f3273c0b854ea101062fbfc6b515dac68d84ae8b8cac146fdf",\n          "stages/3_StandardScaler_243daebf2f5b/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n          "stages/3_StandardScaler_243daebf2f5b/metadata/part-00000-9698ab33-c95f-4830-81ac-9682bb640777-c000.txt": "6fb8b912fa0aaf33c40e738c5c860fefc80db86402b48b31028c440aa9eff021",\n          "stages/4_KMeans_66e40960aec2/data/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n          "stages/4_KMeans_66e40960aec2/data/part-00000-1b083e1e-938c-4d35-aeda-d1b570609c1a-c000.snappy.parquet": "7b963520f010e29da53b05e34cc93382cabf1174d8ae301724264e97789c3150",\n          "stages/4_KMeans_66e40960aec2/data/part-00001-1b083e1e-938c-4d35-aeda-d1b570609c1a-c000.snappy.parquet": "e2ad708c0125969e4c6dc6f18bf31a639ffc80429115f78fcd365c928e209504",\n          "stages/4_KMeans_66e40960aec2/data/part-00002-1b083e1e-938c-4d35-aeda-d1b570609c1a-c000.snappy.parquet": "a2306ebace3f4a8c6f9cde43dea2e44dc407900a01ca7d4b35c87ea0d425b48e",\n          "stages/4_KMeans_66e40960aec2/data/part-00003-1b083e1e-938c-4d35-aeda-d1b570609c1a-c000.snappy.parquet": "d05afce580ec2aafd1da15bac100217174f16571502763502c691713827cbcb8",\n          "stages/4_KMeans_66e40960aec2/metadata/_SUCCESS": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",\n          "stages/4_KMeans_66e40960aec2/metadata/part-00000-db49e59c-444c-443d-8273-1424303b6238-c000.txt": "df1f0c73cd78654cdecf72fb888de40dfd16bd7a2913c4e67a85ddb35e4b91ea"\n        },\n        "bytes": 27181\n      },\n      "elapsed_seconds": 657.111,\n      "figures": [\n        "phase5_c_cluster_sizes.png",\n        "phase5_c_cluster_attack_share.png",\n        "phase5_c_family_heatmap.png",\n        "phase5_c_daily_occupancy.png",\n        "phase5_c_temporal_shift.png",\n        "phase5_c_centroid_profiles.png",\n        "phase5_c_distance_histogram.png",\n        "phase5_c_daily_novelty.png",\n        "phase5_c_binary_novelty.png",\n        "phase5_c_supervised_comparison.png"\n      ],\n      "version": 3,\n      "log_events": 783,\n      "downloaded_reload": {\n        "probes": 3,\n        "pipeline_stages": 5,\n        "verified": true\n      }\n    }\n  }\n}\n', 'src/streaming/alert_engine.py': '"""Versioned, support-aware alert decisions over frozen reference aggregates."""\n\nfrom src.streaming.shift_metrics import distribution_shifts, model_metrics, novelty\n\n\ndef signals(item, reference, context, config):\n    """Emit monitoring measures for all windows and alerts only for held-out evidence."""\n    unsupported = item["rows"] < config["minimum_window"]\n    candidates = [(kind, name, value, baseline, False, unsupported)\n                  for kind, name, value, baseline in distribution_shifts(item, reference, config)]\n    candidates.append(("NOVELTY_RATE", "kmeans_p99", novelty(item)["flag_rate"], novelty(reference)["flag_rate"],\n                       False, unsupported))\n    scoring = config["score_bins"]\n    current = model_metrics(item["scores"]["random_forest"], scoring)\n    historical = model_metrics(reference["scores"]["random_forest"], scoring)\n    rare = min(item["attacks"], item["benign"], reference["attacks"], reference["benign"]) < config["minimum_per_class"]\n    candidates.append(("PERFORMANCE_WARNING", "random_forest_recall", current["recall"], historical["recall"],\n                       True, unsupported or rare))\n    shifts, alerts = [], []\n    for kind, detector, value, baseline, labels, too_small in candidates:\n        threshold = config["alert_thresholds"][kind]\n        support = "insufficient" if too_small or value is None or baseline is None else "supported"\n        delta = None if value is None or baseline is None else (baseline - value if labels else value - baseline)\n        row = context | {"category": kind, "detector": detector, "value": value, "reference_value": baseline,\n                         "delta": delta, "threshold": threshold, "rows": item["rows"], "support": support,\n                         "labels_used": labels, "version": config["version"]}\n        shifts.append(row)\n        if context["split_rank"] > 0 and support == "supported" and delta > threshold:\n            alerts.append(row | {"severity": "high" if delta >= 2 * threshold else "warning",\n                                 "evidence": "retrospective_labels" if labels else "distribution"})\n    return shifts, alerts\n', 'src/streaming/checkpoint_recovery.py': '"""Atomic aggregate accounting for retried Structured Streaming batch IDs."""\n\nimport json\nimport os\n\n\nclass ReplayInterruption(RuntimeError):\n    pass\n\n\ndef commit_file(folder, batch_id):\n    return folder / f"{batch_id:06}.json"\n\n\ndef load_commits(folder):\n    return [json.loads(path.read_text()) for path in sorted(folder.glob("*.json"))]\n\n\ndef commit_aggregate(folder, batch_id, payload):\n    folder.mkdir(parents=True, exist_ok=True)\n    path = commit_file(folder, batch_id)\n    if path.exists():\n        stored = json.loads(path.read_text())\n        if any(stored[key] != payload[key] for key in ("batch_id", "source")):\n            raise ValueError("Batch ID was presented with a different source signature")\n        return False\n    temporary = path.with_suffix(".partial")\n    with temporary.open("w", encoding="utf-8") as stream:\n        json.dump(payload, stream, indent=1, allow_nan=False)\n        stream.flush()\n        os.fsync(stream.fileno())\n    os.replace(temporary, path)\n    descriptor = os.open(folder, os.O_RDONLY)\n    try:\n        os.fsync(descriptor)\n    finally:\n        os.close(descriptor)\n    return True\n\n\ndef committed_ids(checkpoint):\n    return sorted(int(p.name) for p in (checkpoint / "commits").iterdir() if p.name.isdecimal())\n', 'src/streaming/cluster_scoring.py': '"""Score with the original Phase 5 B preprocessing and frozen K-Means stages."""\n\nimport math\n\nfrom pyspark.ml import PipelineModel\nfrom pyspark.sql import functions as F\n\nfrom src.ml.novelty_scoring import squared_distance\nfrom src.streaming.model_scoring import mounted_summary, probe, verified_model\n\n\ndef clustering(spark, root, names, registered):\n    directory, receipt = mounted_summary(root, "phase5_summary.json", "B", "experiment")\n    if (receipt["features"] != names or receipt["status"] != "validated" or\n            receipt["model"]["sha256"] != registered["model"]["sha256"] or\n            receipt["calibration"] != registered["calibration"] or\n            receipt["reference_occupancy"] != registered["reference_occupancy"]):\n        raise ValueError("Mounted clustering receipt differs from Phase 5 B")\n    pipeline = verified_model(spark, directory / "model", registered["model"]["sha256"], names, "_valid", "_filled")\n    preparation, model = PipelineModel(stages=pipeline.stages[:-1]), pipeline.stages[-1]\n    cutoff = registered["calibration"]["cutoffs"]["0.99"]\n    results = cluster_scores(probe(spark, names), preparation, model, cutoff).select("cluster", "distance").collect()\n    k = registered["selected_k"]\n    if (len(model.clusterCenters()) != k or len(results) != 3 or\n            any(not 0 <= r.cluster < k or not math.isfinite(r.distance) or r.distance < 0 for r in results)):\n        raise ValueError("Invalid clustering synthetic probe")\n    return (preparation, model, cutoff), {"probe_rows": len(results), "k": k, "p99": cutoff,\n                                         "sha256": registered["model"]["sha256"]}\n\n\ndef cluster_scores(frame, preparation, model, cutoff):\n    return (squared_distance(model, preparation.transform(frame))\n            .select(*frame.columns, "cluster", F.col("score").alias("distance"))\n            .withColumn("novel", F.col("distance") > cutoff))\n', 'src/streaming/model_scoring.py': '"""Verify and score the original Phase 4 chronological supervised pipelines."""\n\nfrom hashlib import sha256\nimport json\nimport math\nfrom pathlib import Path\n\nfrom pyspark.ml import PipelineModel\nfrom pyspark.ml.functions import vector_to_array\nfrom pyspark.sql import functions as F\n\n\nSUPERVISED = ("random_forest", "logistic_regression", "duration_rule")\n\n\ndef mounted_summary(root, name, identity, key):\n    found = [path for path in root.rglob(name) if json.loads(path.read_text()).get(key) == identity]\n    if len(found) != 1:\n        raise ValueError(f"Expected one mounted chronological {name}, found {len(found)}")\n    return found[0].parent, json.loads(found[0].read_text())\n\n\ndef verified_model(spark, path, hashes, names, imputer_suffix="", output_suffix="_imputed"):\n    files = {p.relative_to(path).as_posix(): sha256(p.read_bytes()).hexdigest()\n             for p in path.rglob("*") if p.is_file() and not p.name.endswith(".crc")}\n    if not files or files != hashes:\n        raise ValueError(f"Model checksum mismatch for {path.name}")\n    metadata = [path / name for name in files\n                if Path(name).parent.name == "metadata" and Path(name).name.startswith("part-")]\n    if not metadata or any(json.loads(item.read_text())["sparkVersion"] != spark.version for item in metadata):\n        raise ValueError(f"Spark version mismatch for {path.name}")\n    model = PipelineModel.load(str(path))\n    imputer, assembler = next(s for s in model.stages if type(s).__name__ == "ImputerModel"), next(\n        s for s in model.stages if type(s).__name__ == "VectorAssembler")\n    if (imputer.getInputCols() != [n + imputer_suffix for n in names] or\n            assembler.getInputCols() != imputer.getOutputCols() or\n            assembler.getInputCols() != [n + output_suffix for n in names]):\n        raise ValueError(f"Model predictor order mismatch for {path.name}")\n    return model\n\n\ndef probe(spark, names):\n    return spark.createDataFrame([\n        tuple(None for _ in names), tuple(0.0 for _ in names), tuple(float(i + 1) for i in range(len(names))),\n    ], ", ".join(name + " DOUBLE" for name in names))\n\n\ndef supervised(spark, root, names, registered):\n    directory, receipt = mounted_summary(root, "phase4_summary.json", "B", "protocol")\n    if (receipt["registered_predictors"] != names or receipt["baselines"] != registered["baselines"] or\n            receipt["status"] != "validated" or receipt["input"]["receipt_sha256"] != registered["input"]["receipt_sha256"]):\n        raise ValueError("Supervised artifact receipt is inconsistent")\n    models, evidence = {}, {}\n    for algorithm in SUPERVISED[:2]:\n        entry = next(m for m in registered["models"] if m["model"] == algorithm and m["training_cohort"] == "group_sample")\n        mounted = next(m for m in receipt["models"] if m["model"] == algorithm and m["training_cohort"] == "group_sample")\n        if mounted["sha256"] != entry["sha256"]:\n            raise ValueError(f"Mounted {algorithm} is not the pinned Phase 4 pipeline")\n        name = Path(entry["path"]).name\n        model = verified_model(spark, directory / "models" / name, entry["sha256"], names)\n        predictions = model.transform(probe(spark, names)).select(vector_to_array("probability")[1]).collect()\n        if len(predictions) != 3 or any(not math.isfinite(r[0]) or not 0 <= r[0] <= 1 for r in predictions):\n            raise ValueError(f"Invalid {algorithm} synthetic probe")\n        models[algorithm] = model\n        evidence[algorithm] = {"directory": name, "files": len(entry["sha256"]), "probe_rows": len(predictions),\n                               "sha256": entry["sha256"]}\n    cutoff = registered["baselines"]["duration_rule"]["benign_training_median_us"]\n    return models, cutoff, evidence\n\n\ndef supervised_scores(frame, models, duration_cutoff):\n    for name, model in models.items():\n        frame = model.transform(frame).select(*frame.columns, vector_to_array("probability")[1].alias(name))\n    return frame.withColumn("duration_rule", F.when(F.col("flow_duration") <= duration_cutoff, 1.0).otherwise(0.0))\n', 'src/streaming/replay_charts.py': '"""Render historical replay evidence from its bounded CSV tables."""\n\nimport csv\nimport json\n\nimport matplotlib\n\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nimport numpy as np\n\n\ndef draw(folder):\n    def read(name):\n        with (folder / f"phase6_{name}.csv").open(newline="", encoding="utf-8") as file:\n            return list(csv.DictReader(file))\n\n    batches, windows, shifts, alerts = (read(name) for name in ("replay_batches", "window_metrics", "shift_metrics", "alerts"))\n    days = [row for row in windows if row["scope"] == "daily"]\n    dates = [r["source_day"][5:] for r in days]\n    output = folder / "figures"\n    output.mkdir(exist_ok=True)\n    names = []\n    plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "font.size": 9})\n\n    def chart(name, title, xlabel, ylabel, x, series, kind="line"):\n        figure, ax = plt.subplots(figsize=(8.7, 3.8), layout="constrained")\n        for label, values in series:\n            if kind == "bar":\n                ax.bar(x, values, color="#127e89")\n            else:\n                ax.plot(x, values, label=label, marker="." if len(x) <= 70 else None)\n        if len(series) > 1:\n            ax.legend(fontsize=8, ncols=2)\n        ax.set(title=title, xlabel=xlabel, ylabel=ylabel)\n        ax.grid(axis="y", alpha=.21)\n        target = f"phase6_{name}.png"\n        figure.savefig(output / target, dpi=120, metadata={"Software": "MMDS Phase 6"})\n        plt.close(figure)\n        names.append(target)\n\n    batch_ids = [int(r["batch_id"]) for r in batches]\n    source_days = {day: i for i, day in enumerate(sorted({r["source_day"] for r in batches}))}\n    chart("timeline", "Chronological release of historical captures", "Structured Streaming batch", "Capture-day order",\n          batch_ids, [("Capture order", [source_days[r["source_day"]] + 1 for r in batches])])\n    chart("batch_rows", "Microbatch size", "Structured Streaming batch", "Flows", batch_ids,\n          [("rows", [int(r["rows"]) for r in batches])], "bar")\n    chart("batch_latency", "Batch processing time", "Structured Streaming batch", "Seconds", batch_ids,\n          [("Callback", [float(r["processing_seconds"]) for r in batches]),\n           ("Scoring and cache materialization", [float(r["scoring_seconds"]) for r in batches])])\n\n    models = [json.loads(r["classifier_metrics"]) for r in days]\n    chart("predicted_attack", "Frozen classifier positive rates by capture day", "Capture day", "Positive flows (%)",\n          dates, [(name.replace("_", " ").title(), [100 * x[name]["predicted_rate"] for x in models]) for name in models[0]])\n    chart("novelty", "Flows above the saved K-Means P99 cutoff", "Capture day", "Flagged flows (%)",\n          dates, [("P99", [100 * json.loads(r["novelty_metrics"])["flag_rate"] for r in days])], "bar")\n    clusters = np.array([json.loads(r["cluster_counts"]) for r in days], dtype=float)\n    clusters /= clusters.sum(axis=1)[:, None]\n    figure, ax = plt.subplots(figsize=(8.7, 3.8), layout="constrained")\n    ax.stackplot(dates, (100 * clusters).T, labels=[str(i) for i in range(clusters.shape[1])], colors=plt.get_cmap("tab20").colors[:clusters.shape[1]])\n    ax.set(title="Frozen K-Means occupancy by capture day", xlabel="Capture day", ylabel="Cluster share (%)")\n    ax.legend(ncols=6, fontsize=7, title="Cluster", loc="upper left")\n    figure.savefig(output / "phase6_occupancy.png", dpi=120, metadata={"Software": "MMDS Phase 6"})\n    plt.close(figure)\n    names.append("phase6_occupancy.png")\n\n    daily = [r for r in shifts if r["scope"] == "daily"]\n    for category, name, title in (("CLUSTER_SHIFT", "cluster_shift", "Cluster occupancy divergence"),\n                                   ("INPUT_SHIFT", "input_shift", "Raw feature divergence"),\n                                   ("SCORE_SHIFT", "score_shift", "Frozen classifier score divergence")):\n        series = [r for r in daily if r["category"] == category]\n        detectors = sorted({r["detector"] for r in series})\n        chart(name, title + " versus historical reference", "Capture day", "Jensen–Shannon bits", dates,\n              [(name, [float(r["value"]) for r in series if r["detector"] == name]) for name in detectors])\n    flagged = {day: sum(row["source_day"] == day for row in alerts) for day in source_days}\n    chart("alerts", "Window and daily alerts by capture day", "Capture day", "Alert records", dates,\n          [("alerts", [flagged[r["source_day"]] for r in days])], "bar")\n    chart("performance", "Retrospective attack recall at the frozen thresholds", "Capture day", "Recall (%)", dates,\n          [(name.replace("_", " ").title(), [100 * x[name]["recall"] if x[name]["recall"] is not None else np.nan for x in models])\n           for name in models[0]])\n    return names\n', 'src/streaming/replay_reporting.py': '"""Generate bounded window evidence, throughput tables and charts from callback commits."""\n\nfrom collections import defaultdict\nimport csv\nimport json\n\nfrom src.ingest.dataset_audit import atomic_json\nfrom src.sql.traffic_analytics import require, write_rows\nfrom src.streaming.alert_engine import signals\nfrom src.streaming.model_scoring import SUPERVISED\nfrom src.streaming.replay_source import CAPTURE_ONLY\nfrom src.streaming.shift_metrics import combine, model_metrics, novelty, tail_upper\n\n\ndef context(day, rank, hour, batch_ids, duration):\n    valid = hour is not None and hour != CAPTURE_ONLY\n    return {"scope": "daily" if hour is None else "intraday" if valid else "capture_only",\n            "source_day": day, "split_rank": rank, "batch_ids": batch_ids, "window_hour": hour,\n            "wall_start": f"{hour:02}:00" if valid else None, "wall_end": f"{hour + duration:02}:00" if valid else None}\n\n\ndef window_evidence(item, reference, ctx, config):\n    scores = {name: model_metrics(item["scores"][name], config["score_bins"]) for name in SUPERVISED}\n    hist = item["scores"]["random_forest"]\n    bins = config["score_bins"] // 32\n    rounded_hist = [0] * 32\n    for score, support in hist.items():\n        rounded_hist[int(score) // bins] += sum(support)\n    row = ctx | {"rows": item["rows"], "eligible": item["eligible"], "attacks": item["attacks"], "benign": item["benign"],\n                 "mean_distance": item["distance_sum"] / item["rows"], "distance_p95_upper": tail_upper(item, config["distance_bounds"]),\n                 "cluster_counts": item["histograms"]["cluster"], "rf_score_counts_32": rounded_hist,\n                 "classifier_metrics": scores, "novelty_metrics": novelty(item)}\n    shifts, alerts = signals(item, reference, ctx, config)\n    return row, shifts, alerts\n\n\ndef scope_summary(item, config):\n    return {"rows": item["rows"], "eligible": item["eligible"], "attacks": item["attacks"], "benign": item["benign"],\n            "models": {name: model_metrics(item["scores"][name], config["score_bins"]) for name in SUPERVISED},\n            "novelty": novelty(item), "cluster_counts": item["histograms"]["cluster"]}\n\n\ndef performance(label, files, documents, result):\n    rows = sum(chunk["rows"] for doc in documents for chunk in doc["source"])\n    return {"configuration": label, "files_per_trigger": files, "batches": len(documents),\n            "rows": rows, "staging_bytes_replayed": sum(chunk["bytes"] for doc in documents for chunk in doc["source"]),\n            "elapsed_seconds": result["elapsed_seconds"], "rows_per_second": rows / result["elapsed_seconds"],\n            "scoring_seconds": sum(doc["scoring_seconds"] for doc in documents),\n            "callback_seconds": sum(doc["processing_seconds"] for doc in documents),\n            "checkpoint_bytes": result["checkpoint_bytes"]}\n\n\ndef publish_results(output, full, run, comparisons, summary, reference, config):\n    windows, shifts, alerts, batches = [], [], [], []\n    days, ranks = defaultdict(list), defaultdict(list)\n    for doc in full:\n        batch_id = doc["batch_id"]\n        for source, chunk in zip(doc["source"], doc["chunks"]):\n            day, split, hour = chunk["source_day"], chunk["split_rank"], chunk["window_hour"]\n            days[day, split].append((batch_id, chunk))\n            ranks[split].append(chunk)\n            batches.append(source | {"batch_id": batch_id, "temporally_eligible": chunk["eligible"],\n                                     "entered_utc": doc["entered_utc"], "finished_utc": doc["finished_utc"],\n                                     "scoring_seconds": doc["scoring_seconds"], "processing_seconds": doc["processing_seconds"],\n                                     "rows_per_second": chunk["rows"] / doc["processing_seconds"]})\n            row, comparison, alarm = window_evidence(chunk, reference, context(day, split, hour, [batch_id], config["window_hours"]), config)\n            windows.append(row)\n            shifts.extend(comparison)\n            alerts.extend(alarm)\n    for (day, split), entries in sorted(days.items()):\n        row, comparison, alarm = window_evidence(combine([chunk for _, chunk in entries]), reference,\n                                               context(day, split, None, [i for i, _ in entries], config["window_hours"]), config)\n        windows.append(row)\n        shifts.extend(comparison)\n        alerts.extend(alarm)\n\n    summary["scopes"] = {name: scope_summary(combine(ranks[i]), config) for i, name in enumerate(("train", "validation", "test"))}\n    require(sum(v["rows"] for v in summary["scopes"].values()) == summary["staging"]["rows"], "Report lost replay rows")\n    require(sum(v["eligible"] for v in summary["scopes"].values()) ==\n            summary["staging"]["rows"] - summary["staging"]["ineligible_retained"],\n            "Report lost capture-day-only records")\n    require(all(r["rows"] == r["eligible"] if r["scope"] == "intraday" else\n                r["eligible"] == 0 if r["scope"] == "capture_only" else True for r in windows), "Ineligible intraday contribution")\n    summary["recovery"] = run["recovery"]\n    summary["configurations"] = [performance("historical", config["files_per_trigger"], full, run)]\n    for label, files, docs, evidence in comparisons:\n        require([x for doc in docs for x in doc["source"]] ==\n                [x for doc in full for x in doc["source"] if x["replay_seq"] in {\n                    i["replay_seq"] for d in docs for i in d["source"]}], "Comparison population differs")\n        expected_sequences = {item["replay_seq"] for document in docs for item in document["chunks"]}\n        expected = combine([x for d in full for x in d["chunks"] if x["replay_seq"] in expected_sequences])\n        observed = combine([x for d in docs for x in d["chunks"]])\n        require(all(value == observed[key] for key, value in expected.items() if key != "distance_sum") and\n                abs(expected["distance_sum"] - observed["distance_sum"]) <= 1e-5 * expected["rows"],\n                "Configuration changed scoring counts")\n        summary["configurations"].append(performance(label, files, docs, evidence))\n    summary["alerts"] = {"count": len(alerts), "high": sum(r["severity"] == "high" for r in alerts)}\n    for name, rows in (("replay_batches", batches), ("window_metrics", windows), ("shift_metrics", shifts),\n                       ("recovery_tests", run["recovery"]), ("performance", summary["configurations"])):\n        write_rows(output / f"phase6_{name}.csv", rows)\n    if alerts:\n        write_rows(output / "phase6_alerts.csv", alerts)\n    else:\n        with (output / "phase6_alerts.csv").open("w", encoding="utf-8") as handle:\n            csv.writer(handle).writerow([*shifts[0], "severity", "evidence"])\n    atomic_json(output / "phase6_stream_progress.json", {"historical": run,\n                **{label: evidence for label, _, _, evidence in comparisons}})\n    # Retain only compact aggregate counts and digests in the published evidence.\n    atomic_json(output / "phase6_reference.json", reference)\n    from src.streaming.replay_charts import draw\n\n    summary["figures"] = draw(output)\n', 'src/streaming/replay_source.py': '"""Build bounded chronological Parquet files and attest their release sequence."""\n\nfrom pathlib import Path\nimport os\nimport time\n\nfrom pyspark import StorageLevel\nfrom pyspark.sql import functions as F\n\nfrom src.ingest.parquet_pipeline import parquet_footers\nfrom src.ml.split_protocols import crossing_groups, group_summary, mark_retention\nfrom src.ml.train_clustering import cohorts, split_counts, verify_phase4_match\nfrom src.sql.traffic_analytics import require\n\n\nAUDIT_FIELDS = (\n    "source_day", "split_rank", "original_timestamp", "parsed_event_time", "temporal_eligibility",\n    "timestamp_parse_status", "source_file", "record_fingerprint", "predictor_key",\n    "binary_label", "attack_family", "original_label",\n)\nCAPTURE_ONLY = 24\n\n\ndef window_key(hours):\n    return F.when(F.col("temporal_eligibility"),\n                  (F.floor(F.hour("parsed_event_time") / hours) * hours).cast("int")).otherwise(CAPTURE_ONLY)\n\n\ndef replay_calendar(rows, config):\n    keys = sorted((row["source_day"], row["window_hour"], row["split_rank"], row["count"]) for row in rows)\n    require(0 < len(keys) <= config["max_chunks"] and len(set((day, hour) for day, hour, _, _ in keys)) == len(keys),\n            "Unexpected number or identity of historical replay chunks")\n    return [{"replay_seq": i, "source_day": day, "window_hour": hour, "split_rank": split, "rows": count}\n            for i, (day, hour, split, count) in enumerate(keys)]\n\n\ndef chunk_file(seq):\n    return f"chunk-{seq:03}.parquet"\n\n\ndef publish(directory, src, seq):\n    target = directory / chunk_file(seq)\n    os.replace(src, target)\n    # FileStreamSource sorts by modification time. Unique times avoid relying on directory order.\n    stamp = 946684800 + 2 * seq\n    os.utime(target, (stamp, stamp))\n    return target.stat().st_size\n\n\ndef stage(spark, receipt, upstream, design, phase4, phase5, names, config, scratch):\n    started = time.monotonic()\n    clean_path = receipt.parent.parent / upstream["policy_version"] / "clean"\n    require(all(upstream["layers"]["clean"][key] == value for key, value in parquet_footers(clean_path).items()),\n            "Unverified clean input")\n    clean = spark.read.parquet(str(clean_path))\n    require(dict(clean.dtypes).get("parsed_event_time") == "timestamp_ntz", "Unexpected timestamp type")\n    extra = ("original_timestamp", "source_file", "timestamp_parse_status")\n    base = cohorts(clean, names, design, "B", phase5["input"]["days"], extra).persist(StorageLevel.DISK_ONLY)\n    cross = crossing_groups(base, "predictor_key").persist(StorageLevel.DISK_ONLY)\n    retained = mark_retention(base, cross, group_summary(cross)["groups"]).where("retained").drop("retained").persist(StorageLevel.DISK_ONLY)\n    try:\n        raw_rows = base.count()\n        pre_anomalies = base.where(~F.col("temporal_eligibility")).count()\n        require(raw_rows == phase5["input"]["rows"] and pre_anomalies == 14, "Original temporal support differs")\n        observed = split_counts(retained)\n        verify_phase4_match(observed, "B", {"protocols": {"B": phase4}})\n        total = sum(row["count"] for row in observed)\n        removed = raw_rows - total\n        require(total == phase5["retained_rows"] and removed == phase5["purge"]["rows"] == 909956,\n                "Chronological retained population differs")\n        require(group_summary(crossing_groups(retained, "predictor_key"))["groups"] == 0, "Retained predictor overlap")\n        cross.unpersist(blocking=True)\n        base.unpersist(blocking=True)\n        staged = retained.withColumn("window_hour", window_key(config["window_hours"]))\n        require(staged.where("window_hour IS NULL OR temporal_eligibility IS NULL OR record_fingerprint IS NULL "\n                             "OR (temporal_eligibility AND CAST(parsed_event_time AS DATE) <> CAST(source_day AS DATE))")\n                .limit(1).count() == 0, "Invalid event window or provenance")\n        calendar = replay_calendar(staged.groupBy("source_day", "window_hour", "split_rank").count().collect(), config)\n        require(all(item["rows"] <= config["max_chunk_rows"] for item in calendar), "Replay chunk exceeds the row budget")\n        lookup = spark.createDataFrame([(item["source_day"], item["window_hour"], item["replay_seq"]) for item in calendar],\n                                       "source_day STRING, window_hour INT, replay_seq INT")\n        packed = (staged.join(F.broadcast(lookup), ["source_day", "window_hour"])\n                  .select(*names, "replay_seq", "window_hour", F.struct(*AUDIT_FIELDS).alias("audit"))\n                  .withColumn("shard", F.col("replay_seq")))\n        schema = packed.drop("shard").schema\n        temporary, directory = scratch / "ordered", scratch / "available"\n        directory.mkdir()\n        (packed.repartition(config["spark"]["shuffle_partitions"], "shard")\n         .sortWithinPartitions("shard", "audit.parsed_event_time", "audit.record_fingerprint", "audit.predictor_key")\n         .write.partitionBy("shard").parquet(str(temporary)))\n        for item in calendar:\n            sequence = item["replay_seq"]\n            paths = list((temporary / f"shard={sequence}").glob("*.parquet"))\n            require(len(paths) == 1, "Replay staging must have exactly one file per sequence")\n            item["bytes"] = publish(directory, paths[0], sequence)\n        total_bytes = sum(item["bytes"] for item in calendar)\n        require(total_bytes <= config["max_staging_bytes"], "Replay staging exceeded the configured size limit")\n        require(sum(item["rows"] for item in calendar) == total, "Replay staging lost records")\n        return directory, schema, calendar, {"rows": total, "purged_rows": removed, "source_anomalies": pre_anomalies,\n                                            "ineligible_retained": sum(x["rows"] for x in calendar if x["window_hour"] == CAPTURE_ONLY),\n                                            "bytes": total_bytes, "seconds": round(time.monotonic() - started, 3),\n                                            "chunks": len(calendar), "cohorts": observed}\n    finally:\n        for data in (base, cross, retained):\n            data.unpersist()\n', 'src/streaming/run_replay.py': '"""Orchestrate verified input, model reload, replay and bounded reporting."""\n\nimport json\n\nfrom src.ingest.dataset_audit import atomic_json, spark_session\nfrom src.ml.feature_registry import registry\nfrom src.ml.train_supervised import verified_input\nfrom src.sql.traffic_analytics import locate_producer, require\nfrom src.streaming.cluster_scoring import clustering\nfrom src.streaming.model_scoring import supervised\n\n\ndef run(project, input_root, output, scratch, mode):\n    def read(name):\n        return json.loads((project / name).read_text())\n\n    config = read("configs/streaming_experiments.yaml")\n    phase2 = read("manifests/parquet_layers.json")\n    phase4 = read("manifests/phase4_models.json")["protocols"]["B"]\n    phase5 = read("manifests/phase5_clustering_models.json")["experiments"]["B"]\n    design = read("configs/ml_experiments.yaml")\n    names = registry(design, phase2["feature_columns"])\n    scratch.mkdir(parents=True, exist_ok=True)\n    output.mkdir(parents=True, exist_ok=True)\n    spark = spark_session(config, scratch / "spark")\n    spark.conf.set("spark.sql.sources.partitionColumnTypeInference.enabled", "false")\n    spark.sparkContext.setLogLevel("WARN")\n    try:\n        require(spark.version == "4.3.0", "Unexpected Spark runtime")\n        receipt = locate_producer(input_root, phase2["independent_mount"]["summary_sha256"])\n        _, input_evidence = verified_input(spark, receipt, phase2, read("configs/cleaning.yaml"))\n        models, duration, supervised_evidence = supervised(spark, input_root, names, phase4)\n        cluster, cluster_evidence = clustering(spark, input_root, names, phase5)\n        summary = {"mode": mode, "status": "inputs_validated", "spark": spark.version, "config": config,\n                   "features": names, "input": input_evidence, "supervised": supervised_evidence,\n                   "clustering": cluster_evidence, "duration_cutoff": duration}\n        atomic_json(output / "phase6_summary.json", summary)\n        print("PHASE6 pinned inputs and three frozen pipelines validated", flush=True)\n        if mode == "full":\n            from src.streaming.structured_replay import full_replay\n\n            full_replay(spark, receipt, phase2, design, phase4, phase5, names, models, duration,\n                        cluster, config, output, scratch, summary)\n    finally:\n        spark.stop()\n', 'src/streaming/shift_metrics.py': '"""Compute compact replay histograms, retrospective metrics and shift distances."""\n\nfrom collections import Counter\nimport math\nimport time\n\nfrom pyspark import StorageLevel\nfrom pyspark.sql import functions as F\n\nfrom src.ml.cluster_evolution import divergence\nfrom src.ml.evaluate_supervised import binned_auc, confusion_metrics, rate\nfrom src.ml.novelty_scoring import novelty_row\nfrom src.sql.traffic_analytics import require\nfrom src.streaming.cluster_scoring import cluster_scores\nfrom src.streaming.model_scoring import SUPERVISED, supervised_scores\n\n\ndef fixed_bin(name, bounds):\n    value = F.col(name)\n    return F.when(value.isNull(), len(bounds) + 1).otherwise(\n        sum((F.when(value >= edge, 1).otherwise(0) for edge in bounds), F.lit(0)))\n\n\ndef score_bin(name, bins):\n    score = F.col(name)\n    return F.when(score.between(0, 1), F.least(F.floor(score * bins), F.lit(bins - 1))).otherwise(-1)\n\n\ndef aggregate(batch, models, duration, cluster, config):\n    started = time.monotonic()\n    aligned = supervised_scores(batch, models, duration)\n    scored = (cluster_scores(aligned, *cluster).select("replay_seq", "window_hour", "audit", "cluster", "distance", "novel",\n                                                       *SUPERVISED, *config["features"])\n              .persist(StorageLevel.MEMORY_AND_DISK))\n    try:\n        rows = scored.count()\n        scoring_seconds = time.monotonic() - started\n        observed = {}\n        label = F.col("audit.binary_label")\n        count = lambda condition: F.count(F.when(condition, 1))\n        histograms = {name: (fixed_bin(name, edges), len(edges) + 2) for name, edges in config["features"].items()}\n        histograms["cluster"] = F.col("cluster"), len(cluster[1].clusterCenters())\n        histograms["distance"] = fixed_bin("distance", config["distance_bounds"]), len(config["distance_bounds"]) + 2\n        fields = [\n            F.count("*").alias("rows"), count(F.col("audit.temporal_eligibility")).alias("eligible"),\n            count(label == 1).alias("attacks"), count(label == 0).alias("benign"),\n            count(F.col("novel") & (label == 1)).alias("novel_attack"),\n            count(F.col("novel") & (label == 0)).alias("novel_benign"),\n            F.sum("distance").alias("distance_sum"),\n            F.countDistinct(F.struct("audit.source_day", "window_hour", "audit.split_rank")).alias("groups"),\n            F.first("audit.source_day").alias("source_day"), F.first("window_hour").alias("window_hour"),\n            F.first("audit.split_rank").alias("split_rank"),\n            count(F.col("distance").isNull() | F.isnan("distance") | (F.col("distance") < 0) |\n                  (F.col("distance") == float("inf"))).alias("bad_distance"),\n        ]\n        fields.extend(count(column == i).alias(f"{name}_{i}") for name, (column, size) in histograms.items() for i in range(size))\n        for row in scored.groupBy("replay_seq").agg(*fields).collect():\n            item = row.asDict()\n            require(item.pop("groups") == 1 and item.pop("bad_distance") == 0, "Replay chunk identity or distance is invalid")\n            item["histograms"] = {name: [item.pop(f"{name}_{i}") for i in range(size)]\n                                  for name, (_, size) in histograms.items()}\n            require(item["rows"] == item["attacks"] + item["benign"] and\n                    all(sum(values) == item["rows"] for values in item["histograms"].values()), "Chunk totals do not reconcile")\n            item["scores"] = {name: {} for name in SUPERVISED}\n            observed[row.replay_seq] = item\n        labels = scored.select("replay_seq", label.alias("label"), F.posexplode(F.array(*[\n            score_bin(name, config["score_bins"]) for name in SUPERVISED\n        ])).alias("model", "bin"))\n        for row in labels.groupBy("replay_seq", "model", "bin", "label").count().collect():\n            require(row.bin >= 0 and row.label in (0, 1), "Invalid classifier score or evaluation label")\n            buckets = observed[row.replay_seq]["scores"][SUPERVISED[row.model]]\n            buckets.setdefault(str(row.bin), [0, 0])[row.label] = row["count"]\n        for item in observed.values():\n            for buckets in item["scores"].values():\n                require(sum(x[0] for x in buckets.values()) == item["benign"] and\n                        sum(x[1] for x in buckets.values()) == item["attacks"], "Classifier score support differs")\n        require(rows == sum(item["rows"] for item in observed.values()), "Microbatch accounting differs")\n        return [observed[seq] for seq in sorted(observed)], scoring_seconds\n    finally:\n        scored.unpersist(blocking=True)\n\n\ndef combine(items):\n    n = {key: sum(item[key] for item in items) for key in\n         ("rows", "eligible", "attacks", "benign", "novel_attack", "novel_benign", "distance_sum")}\n    n["histograms"] = {name: list(map(sum, zip(*(x["histograms"][name] for x in items)))) for name in items[0]["histograms"]}\n    scores = {}\n    for name in SUPERVISED:\n        negative, positive = Counter(), Counter()\n        for item in items:\n            for bucket, (left, right) in item["scores"][name].items():\n                negative[bucket] += left\n                positive[bucket] += right\n        scores[name] = {bucket: [negative[bucket], positive[bucket]] for bucket in sorted(negative.keys() | positive.keys(), key=int)}\n    n["scores"] = scores\n    return n\n\n\ndef model_metrics(scores, bins):\n    hits = [0, 0]\n    misses = [0, 0]\n    for bucket, counts in scores.items():\n        population = hits if int(bucket) >= bins // 2 else misses\n        population[0] += counts[0]\n        population[1] += counts[1]\n    result = confusion_metrics(hits[1], misses[0], hits[0], misses[1])\n    return result | binned_auc({int(key): value for key, value in scores.items()}) | {"predicted_positive": sum(hits), "predicted_rate": rate(sum(hits), result["rows"])}\n\n\ndef unsupervised_vectors(item, bins):\n    result = dict(item["histograms"])\n    for name, scores in item["scores"].items():\n        result[name] = [sum(scores.get(str(i), (0, 0))) for i in range(bins)]\n    return result\n\n\ndef distribution_shifts(item, reference, config):\n    current = unsupervised_vectors(item, config["score_bins"])\n    baseline = unsupervised_vectors(reference, config["score_bins"])\n    for name in (*config["features"], *SUPERVISED, "cluster"):\n        metric = "CLUSTER_SHIFT" if name == "cluster" else "SCORE_SHIFT" if name in SUPERVISED else "INPUT_SHIFT"\n        yield metric, name, divergence(current[name], baseline[name]), 0.0\n\n\ndef tail_upper(item, edges, percentile=0.95):\n    target = math.ceil(percentile * item["rows"])\n    accrued = 0\n    for i, count in enumerate(item["histograms"]["distance"]):\n        accrued += count\n        if target and accrued >= target:\n            return edges[i] if i < len(edges) else None\n    return None\n\n\ndef novelty(item):\n    return novelty_row(item["attacks"], item["benign"], item["novel_attack"], item["novel_benign"])\n', 'src/streaming/structured_replay.py': '"""Drive historical traffic through bounded file-backed Spark Structured Streaming."""\n\nfrom datetime import datetime, timezone\nfrom hashlib import sha256\nimport json\nimport os\nfrom pathlib import Path\nimport time\n\nfrom pyspark.sql import functions as F\nfrom pyspark.sql.streaming import StreamingQueryException\n\nfrom src.ingest.dataset_audit import atomic_json\nfrom src.ingest.parquet_pipeline import driver_peak_bytes, resource_limits\nfrom src.sql.traffic_analytics import require\nfrom src.streaming.checkpoint_recovery import (\n    ReplayInterruption, commit_aggregate, commit_file, committed_ids, load_commits,\n)\nfrom src.streaming.replay_source import CAPTURE_ONLY, chunk_file, stage\nfrom src.streaming.shift_metrics import aggregate, combine\n\n\ndef utc_now():\n    return datetime.now(timezone.utc).isoformat(timespec="milliseconds")\n\n\ndef query_stream(spark, source, schema, calendar, checkpoint, sink, files, process, inject=False):\n    started = time.monotonic()\n    receipt = {"attempts": [], "progress": [], "recovery": []}\n    failure = {"injected": False}\n    recovery_batch = 1\n    batches = (len(calendar) + files - 1) // files\n    require(batches > recovery_batch or not inject, "Need at least two batches to test recovery")\n    observed = spark.readStream.schema(schema).option("maxFilesPerTrigger", files).parquet(str(source))\n    require(observed.isStreaming, "A Structured Streaming DataFrame is required")\n\n    def callback(frame, batch_id):\n        entered = time.monotonic()\n        when = utc_now()\n        expected = calendar[batch_id * files:(batch_id + 1) * files]\n        require(bool(expected), "Spark delivered an unexpected extra batch")\n        file_counts = frame.select(F.col("replay_seq").alias("seq"), F.input_file_name().alias("file")).groupBy("seq", "file").count().collect()\n        actual = sorted((row.seq, Path(row.file).name, row["count"]) for row in file_counts)\n        require(actual == [(item["replay_seq"], chunk_file(item["replay_seq"]), item["rows"]) for item in expected],\n                f"Spark file release order differs at batch {batch_id}")\n        payload = {"batch_id": batch_id, "source": expected}\n        exists = commit_file(sink, batch_id).exists()\n        if exists:\n            require(not commit_aggregate(sink, batch_id, payload), "Retried batch overwrote its sink commit")\n            receipt["recovery"].append({"test": "replayed_committed_sink_batch", "batch_id": batch_id, "passed": True})\n            return\n        items, scoring_seconds = process(frame)\n        require([(i["replay_seq"], i["source_day"], i["window_hour"], i["split_rank"], i["rows"]) for i in items] ==\n                [(i["replay_seq"], i["source_day"], i["window_hour"], i["split_rank"], i["rows"]) for i in expected],\n                "Scored chunk does not match its manifest entry")\n        payload.update(chunks=items, entered_utc=when, finished_utc=utc_now(),\n                       scoring_seconds=round(scoring_seconds, 6), processing_seconds=round(time.monotonic() - entered, 6))\n        require(commit_aggregate(sink, batch_id, payload), "Uncommitted batch already had output")\n        print(f"PHASE6 stream batch {batch_id} committed sink sequences "\n              f"{expected[0][\'replay_seq\']}–{expected[-1][\'replay_seq\']} rows {sum(i[\'rows\'] for i in expected)}", flush=True)\n        if inject and batch_id == recovery_batch and not failure["injected"]:\n            failure["injected"] = True\n            raise ReplayInterruption("PHASE6 planned stop after atomic sink publish")\n\n    def attempt(expected_error):\n        query = (observed.writeStream.option("checkpointLocation", str(checkpoint)).outputMode("append")\n                 .trigger(availableNow=True).foreachBatch(callback).start())\n        try:\n            query.awaitTermination()\n            require(not expected_error, "Expected planned interruption did not occur")\n        except StreamingQueryException as exc:\n            if not expected_error or not failure["injected"] or "PHASE6 planned stop" not in str(exc):\n                raise\n        finally:\n            if query.isActive:\n                query.stop()\n        receipt["attempts"].append({"query_id": str(query.id), "run_id": str(query.runId),\n                                    "outcome": "planned_interruption" if expected_error else "terminated"})\n        for progress in query.recentProgress:\n            receipt["progress"].append(json.loads(progress.json))\n\n    attempt(inject)\n    if inject:\n        path = commit_file(sink, recovery_batch)\n        checksum = sha256(path.read_bytes()).hexdigest()\n        require(committed_ids(checkpoint) == list(range(recovery_batch)), "Checkpoint advanced beyond injected failure")\n        partial = path.with_suffix(".partial")\n        partial.write_text(\'{"interrupted":\')\n        receipt["recovery"].append({"test": "partial_aggregate_ignored", "batch_id": recovery_batch, "passed":\n                                    len(load_commits(sink)) == recovery_batch + 1})\n        attempt(False)\n        require(checksum == sha256(path.read_bytes()).hexdigest(), "Idempotent retry changed an aggregate")\n        require(receipt["attempts"][0]["query_id"] == receipt["attempts"][1]["query_id"] and\n                receipt["attempts"][0]["run_id"] != receipt["attempts"][1]["run_id"], "Restart did not use the same checkpoint")\n        receipt["recovery"].append({"test": "restart_same_query_new_run", "batch_id": recovery_batch, "passed": True})\n        receipt["recovery"].append({"test": "durable_sink_hash_unchanged", "batch_id": recovery_batch, "passed": True})\n        partial.unlink()\n    documents = load_commits(sink)\n    expected_ids = list(range(batches))\n    require([x["batch_id"] for x in documents] == committed_ids(checkpoint) == expected_ids,\n            "Checkpoint and aggregate sink accounting differ")\n    require([i for doc in documents for i in doc["source"]] == calendar, "Replay duplicated or lost source chunks")\n    require({p["batchId"] for p in receipt["progress"] if p["numInputRows"] > 0} == set(expected_ids),\n            "Actual Structured Streaming progress does not cover all data batches")\n    require(all(row["passed"] for row in receipt["recovery"]), "An interruption check failed")\n    receipt["recovery"].append({"test": "checkpoint_and_sink_reconcile", "batch_id": batches - 1, "passed": True})\n    receipt["checkpoint_bytes"] = sum(path.stat().st_size for path in checkpoint.rglob("*") if path.is_file())\n    receipt["elapsed_seconds"] = round(time.monotonic() - started, 3)\n    return documents, receipt\n\n\ndef full_replay(spark, receipt, upstream, design, phase4, phase5, names, models, duration, cluster, config, output, scratch, summary):\n    started = time.monotonic()\n    source, schema, calendar, staging = stage(spark, receipt, upstream, design, phase4, phase5, names, config, scratch)\n    summary.update(status="staged", staging=staging, source_schema=schema.jsonValue(),\n                   resources=resource_limits(spark, config, output))\n    atomic_json(output / "phase6_summary.json", summary)\n    print(f"PHASE6 staged {staging[\'chunks\']} chunks and {staging[\'rows\']} rows, {staging[\'bytes\']} bytes", flush=True)\n\n    def process(frame):\n        require(set(frame.columns) == set(names) | {"replay_seq", "window_hour", "audit"}, "Unexpected streaming source schema")\n        return aggregate(frame, models, duration, cluster, config)\n\n    full, run = query_stream(spark, source, schema, calendar, scratch / "checkpoint", scratch / "sink",\n                             config["files_per_trigger"], process, inject=True)\n    reference = combine([item for doc in full for item in doc["chunks"] if item["split_rank"] == 0])\n    require(reference["histograms"]["cluster"] == phase5["reference_occupancy"], "Phase 5 reference cluster occupancy differs")\n    print(f"PHASE6 streaming and recovery validated over {len(full)} distinct batches", flush=True)\n    subset = [item for item in calendar if item["window_hour"] != CAPTURE_ONLY][:config["comparison_chunks"]]\n    comparison_source = scratch / "comparison-input"\n    comparison_source.mkdir()\n    for item in subset:\n        path = chunk_file(item["replay_seq"])\n        os.link(source / path, comparison_source / path)\n    comparisons = []\n    for files in (config["files_per_trigger"], config["comparison_files_per_trigger"]):\n        label = f"comparison-{files}"\n        docs, evidence = query_stream(spark, comparison_source, schema, subset, scratch / (label + "-checkpoint"),\n                                      scratch / (label + "-sink"), files, process)\n        comparisons.append((label, files, docs, evidence))\n    # Metrics are reconstructed offline from completed immutable callback outputs.\n    from src.streaming.replay_reporting import publish_results\n\n    summary.update(status="validated", elapsed_seconds=round(time.monotonic() - started, 3),\n                   driver_peak_rss_bytes=driver_peak_bytes(spark), reference_rows=reference["rows"],\n                   reference_cluster_occupancy=reference["histograms"]["cluster"])\n    publish_results(output, full, run, comparisons, summary, reference, config)\n    atomic_json(output / "phase6_summary.json", summary)\n    print(f"PHASE6 complete {staging[\'rows\']} retained flows in {len(full)} streaming batches", flush=True)\n'}
for relative, source in sources.items():
    target = project / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(source, encoding="utf-8")
sys.path.insert(0, str(project))


In [ ]:
from src.streaming.run_replay import run
run(project, Path("/kaggle/input"), Path("/kaggle/working/mmds_phase6"),
    Path("/kaggle/temp/mmds_phase6/replay"), mode='full')
